# ASA³P workflow for AVITI bacterial genome reads on HPC — v5.5

**Prepared by:** Cahya Prihatna, Plant Sciences & Plant Pathology Department, Montana State University

ASA³P is run from scratch with **short isolate names** (PI003, PI024_2, ...) written into `config.xls`, so every ASA³P step sees short names. v3 separates what ASA³P does from what is done outside it, so the Methods can say exactly which results came from the pipeline.

| Part | What it is | Label |
|---|---|---|
| **A** | Setup and the ASA³P run itself | `[ASA³P]` |
| **B** | Checking what ASA³P produced; deciding what needs replacing | `[Check]` |
| **C** | Replacement runs, **only** for ASA³P steps (or isolates) whose output is missing or unusable | `[Replacement — runs only if needed]` |
| **D** | Analyses ASA³P does not do | `[Outside ASA³P]` |
| **F** | Functional screens of all genomes: CAZymes / PCWDE, secretion systems, prophages and plasmids, virulence and metal-resistance genes, effector candidates. **Run it before E**: the Methods draft and output map in E report its results | `[Outside ASA³P]` |
| **E** | Tool versions, an auto-generated Methods draft, and the output map | — |

**Rule used throughout:** wherever ASA³P produced a result, that result is the one reported. Part C cells check this themselves and do nothing when ASA³P's output is present. Every decision is written to `downstream/provenance/`, and the Methods draft in Part E is built from those records.

**How to run:** top to bottom, on the HPC login node (Jupyter on the cluster, or paste each cell into a terminal without the `%%bash` line). Every cell sources `config.sh`, so cells can be re-run independently. Heavy steps go to SLURM; wait for a job to finish before running the cell that checks it.

**Nothing large is downloaded:** ASA³P (program, databases, image) is the copy already installed under `asap_project/software/`; the 80 GB Zenodo download is not repeated.

**v5.5 (new D4b and D4c):** a **genus-level core-gene tree** for *Pectobacterium* (D4b) and an optional **genus identification of the isolates that failed the species check** (D4c). ASA³P's own genus-level Roary run had only 22 core genes, because that project holds about seven genomes that are not *Pectobacterium* (the isolates that failed the species check): the core is the genes present in all genomes, and one genome from another genus takes almost all of them out. D4b builds the tree from the isolates that passed the species check plus the *Pectobacterium* reference genomes (optionally with the type-strain genomes NCBI holds for the other described species, found in D4b.1), with Roary at 90 % identity, MAFFT and IQ-TREE (ModelFinder within the GTR family, 1,000 UFBoot, 1,000 SH-aLRT, then site concordance factors). D4b.5 shows which genomes keep a core from forming, for the new run and for ASA³P's old one. D4c names the genus of the isolates that failed the species check with GTDB-Tk. The Methods draft (E2) describes both when their outputs exist, and E1 records the GTDB-Tk version. **Run D4b after D1 (the species gate) and D2.0.** Nothing else changed from v5.4.2, and A0.1 does not need to be run again.

**v5.4.2 (D3 only):** the Mashtree step (D3) now also builds a tree of **every** genome, whatever the species check says, with **bootstrap support** on the nodes (`mashtree_ALL_bootstrap.nwk`) and the Mash distance matrix (`mashtree_ALL_distances.tsv`); the Methods draft (E) describes it when it exists. Re-run **D3** (and E2 if you want the Methods sentence); nothing else changed from v5.4.1.

**v5.4.1 (F0 only):** the Part F setup now retries its downloads, fetches the dbCAN database from its AWS S3 release first (the university server timed out after 30 s), falls back to `pip` for MOB-suite, checks that each tool runs after it is installed, rebuilds an environment that is broken, and keeps one log per item. Part F's steps use a database only once F0 has finished it. Nothing else changed from v5.4; the config cell does not need to be run again.

**Changes since v5.3 (Part F and fixes):** Part F is new. It screens **every genome, including the isolates that failed the species check**, for CAZymes and plant-cell-wall-degrading enzyme families (dbCAN), secretion systems (MacSyFinder with TXSScan), prophages and plasmids (geNomad, MOB-suite, PlasmidFinder), virulence factors (VFDB), metal and biocide resistance genes (BacMet2) and type III effector candidates (EffectiveT3). Each tool is its own recorded step per isolate, so one failure never costs the others, and every result row carries the isolate's species-check status and genome-based identification. Fixes: (1) **every cell that runs a script now rewrites it first.** v5.3's D2.0 ran whatever `prepare_roary_groups.py` was on disk; if cell C4 had not been re-run after the upgrade, that was the v5.2 script, which built Roary groups without the gate. (2) The clinker environment now uses the right package, `clinker-py` (`clinker` on bioconda is a different package, so `clinker_env` was never created), and the clinker cell is optional. (3) The pyGenomeViz job sets `MPLBACKEND=Agg`: a job submitted from Jupyter inherits the notebook's inline plotting backend and fails at the plotting stage. (4) D1.3 now also builds ASA³P's taxonomy table and the two-column identification table for the supplement. (5) D5 stops with a clear message if `source_metadata_correlated.tsv` is not in `metadata/`. **Re-run A0.1 once** (it adds the Part F settings to `config.sh`).

**Changes since v5.2 (species gate):** the first full run showed that 13 of the 25 assemblies are not the organism they were labelled as: 11 matched none of the 10 reference species (best fastANI 78–79 %, the distance between *Pectobacterium* and *Dickeya*), and two *P. versatile* assemblies (PI012, PI053) held a complete *P. versatile* genome **plus** a second genome. v5.2 grouped isolates by their metadata label and would have put all of them into the pan-genomes and trees. v5.3 adds a **gate** in D1: every isolate gets a status from the fastANI species check (`PASS`, `RELABELLED`, `MIXED`, `NO_MATCH`), and only included isolates go into the Roary groups (D2), the Mashtree tree (D3), the gene-content and map steps (D5–D6) and the Methods draft. You can overrule any decision in `metadata/isolate_overrides.tsv` (D1.4). New optional steps: all-vs-all fastANI and 16S extraction to see what the failed isolates are (D1.3), a provisional contig split that separates the expected organism from a second genome in a `MIXED` assembly (D1.5), and annotation of the filtered genome (D1.6). Parts A–C are unchanged, except that A0.1 adds three lines to `config.sh` (**re-run A0.1 once**) and the Prokka array script can annotate a filtered genome. When the isolates are re-sequenced, run them through A–D again: the gate lets them in on their own result.

**Changes since v5.1:** the audit (B1) now recognises ASA³P's real output naming. ASA³P names every genome `<Genus>_<species>_<short name>` (for example `Pectobacterium_atrosepticum_PI004`) for its folders and files, and v5.1 looked for the bare short name, so it found no assembly and no GFF for any isolate. A short name is now matched against the end of the genome name, so `PI024` never claims `…_PI024_2`; the `-discarded.fasta` files are ignored; and B1 lists any genome folder that matches no isolate in the sample sheet. Nothing else changed.

**Changes since v5:** an ASA³P step now counts as usable only if its folder holds files **and** ASA³P's own end-of-run statistics do not report it as failed. ASA³P can leave files in a folder for a step that failed (for example the resistance-gene step `abr`), and v5 would have accepted those. Part A is unchanged, except that A5.2 now refuses to submit a genus whose `asap_<Genus>` job is already queued or running. The audit table (B1), the replacement decisions (B2) and the Methods draft (E) use the new rule.

**Changes since v4:** the reads are taken from the sequencing-centre tarball (`TARBALL` in `config.sh`) by a new step, A1.1. The tarball is read in place, never copied, and only the isolates in `metadata_shortnames.tsv` are extracted. Raw file names keep their long names; short names exist only as symlinks (A3) and in `config.xls` (A4).

**Changes since v2:** replacement steps (resistance genes, virulence factors, genus-level pan-genome) now run only for genera where ASA³P's step came out empty; each fallback-annotated isolate carries a recorded reason; the Methods paragraph is generated from the actual run, and the ASA³P version in it is read from your install (A5.4).

# Part A — The ASA³P pipeline
**`[ASA³P]`** — the ASA³P pipeline itself, or preparing its inputs.

## A0. Setup
### A0.1 Write `config.sh` (edit paths here only)
v5.4 adds the Part F settings (`FUNC`, `FUNC_DB`, `ET3_*`, `ABR_*`, `PLF_*`) to the lines v5.3 added; if you ran v5.3 or earlier, run this cell once more. It only rewrites `config.sh` and creates one folder.

In [1]:
%%bash
mkdir -p "$(dirname "/home/s94m696/asap_shortname_projects/config.sh")"
cat > "/home/s94m696/asap_shortname_projects/config.sh" <<'END_OF_SCRIPT'
# shellcheck shell=bash
# ===========================================================================
# ASA3P short-name workflow -- single source of paths and settings.
# Every notebook cell and every SLURM script sources this file, so a path
# only ever needs to change here.
# ===========================================================================

# ---- edit these -----------------------------------------------------------
export ROOT="/home/s94m696/asap_shortname_projects"
# Reads: the sequencing-centre tarball is read in place (never copied or renamed). READDIR is the flat folder
# its isolates are extracted into (long names as delivered). Use project/scratch storage if home quota is tight.
export TARBALL="/home/s94m696/Choudhury2_Project_002.tar.gz"
export READDIR="/home/s94m696/unicycler_project/reads/Choudhury2_Project_002"
export ASAPDIR="/home/s94m696/asap_project/software/asap"
export ASAP_IMG="/home/s94m696/asap_project/software/asap_latest.sif"
export ASAP_TEMPLATE="$ASAPDIR/config.xls"

export SLURM_ACCOUNT="priority-janakjoshi"
export SLURM_PARTITION="priority"

export USER_FIRST="Cahya"
export USER_SURNAME="Prihatna"
export USER_EMAIL="cahya.prihatna@montana.edu"
export CENTRE="MSU"                      # goes into Prokka GenBank/GFF records
export ASAP_VERSION="1.2.2"               # fallback only: the notebook reads the real version from asap.jar
                                          # or the ASA3P run log; this Zenodo version is used if neither has it

export CONDA_MODULE="Miniconda3/22.11.1-1"
export IQTREE_MODULE="IQ-TREE/2.2.2.6-gompi-2022b"

# Apptainer stores pulled images in ~/.apptainer/cache by default. If your home
# quota is tight, uncomment and point this at a project/scratch folder:
# export APPTAINER_CACHEDIR="/path/to/project_or_scratch/apptainer_cache"

# Files from the first run that are re-used instead of downloaded again
export OLD_REFDIR="/home/s94m696/asap_final_projects_shortnames/all25_speciesrefs_phylogeny/refs"
export OLD_ROARY_IMG="/home/s94m696/asap_final_projects_shortnames/software/roary_3.13.0.sif"
export OLD_MASHTREE_IMG="/home/s94m696/asap_final_projects_shortnames/all25_outgroup_phylogeny/software/mashtree.sif"

# ---- derived (no need to edit) --------------------------------------------
export ASAP_CONFIG="$ROOT/config.sh"
export META="$ROOT/metadata"
export SCRIPTS="$ROOT/scripts"
export LOGS="$ROOT/logs"
export SOFT="$ROOT/software"
export REFDIR="$ROOT/references"
export GENOMES="$ROOT/genomes"            # standardized raw contigs, one <short>.fna per isolate
export DOWNSTREAM="$ROOT/downstream"
export FILTERED="$ROOT/genomes_filtered"    # contigs of the expected organism only, for MIXED assemblies (D1.5)
export OVERRIDES="$META/isolate_overrides.tsv"   # your decisions on top of the species check (D1.4)
export GATE="$DOWNSTREAM/species_check/analysis_set.tsv"   # which isolates go into Part D (D1.2)
export PROVENANCE="$DOWNSTREAM/provenance"  # what ran inside vs. outside ASA3P, and why

export ROARY_IMG="$SOFT/roary_3.13.0.sif"
export MASHTREE_IMG="$SOFT/mashtree.sif"
export FASTANI_IMG="$SOFT/fastani.sif"
export AMRFINDER_IMG="$SOFT/amrfinderplus.sif"
export ABRICATE_IMG="$SOFT/abricate.sif"
export PROKKA_IMG="$ASAP_IMG"             # fallback Prokka uses the ASA3P image

export GENERA="Pectobacterium Dickeya"

# ---- Part F (comparative-genomic screens: CAZymes, secretion systems, mobile elements, metal/biocide, effectors) ----
export FUNC="$DOWNSTREAM/functional"       # proteins, per-tool results and the summary tables
export FUNC_DB="$SOFT/functional_db"       # databases for the Part F tools (downloaded once, in F0)
export ET3_DIR="$SOFT/effectiveT3"         # EffectiveT3 1.0.1: TTSS_GUI-1.0.1.jar + module/ (downloaded in F0)
export ET3_MODEL="TTSS_STD-2.0.2.jar"      # EffectiveT3 model; TTSS_PLANT-1.0.1.jar is the plant-pathogen model
export ET3_CUTOFF="selective"              # selective | sensitive | cutoff=<number>
export ABR_MINID=80                        # ABRicate (VFDB, BacMet2): minimum % identity ...
export ABR_MINCOV=80                       # ... and % coverage
export PLF_MINID=80                        # ABRicate PlasmidFinder: PlasmidFinder's own recommended
export PLF_MINCOV=60                       # thresholds (80 % identity, 60 % coverage)

project_dir() { echo "$ROOT/ASA3P_$1"; }

# conda's activate scripts reference unset variables, so relax `set -u` around them
use_env() {
    set +u
    module load "$CONDA_MODULE"
    source "$(conda info --base)/etc/profile.d/conda.sh"
    conda activate "$1"
    set -u
}

# Newest file matching a glob (replaces hardcoded job IDs / timestamps)
latest() { ls -t "$@" 2>/dev/null | head -1; }

# submit <jobname> <cpus> <mem> <time> <script> [script args...]
# Extra sbatch flags (e.g. --array=1-19) can be passed via SBATCH_EXTRA.
submit() {
    local name=$1 cpus=$2 mem=$3 time=$4 tag="%j"
    shift 4
    mkdir -p "$LOGS"
    case "${SBATCH_EXTRA:-}" in *--array*) tag="%A_%a" ;; esac
    # shellcheck disable=SC2086
    sbatch --parsable \
        --job-name="$name" \
        --account="$SLURM_ACCOUNT" --partition="$SLURM_PARTITION" \
        --cpus-per-task="$cpus" --mem="$mem" --time="$time" \
        --output="$LOGS/${name}_${tag}.out" --error="$LOGS/${name}_${tag}.err" \
        ${SBATCH_EXTRA:-} "$@"
}

# Show my recent jobs whose name contains a pattern
jobs_like() {
    sacct -u "$USER" -S "$(date -d '-7 days' +%F)" \
        --format=JobID%20,JobName%30,State,ExitCode,Elapsed | grep -E "JobID|$1"
}
END_OF_SCRIPT

source "/home/s94m696/asap_shortname_projects/config.sh"
mkdir -p "$META" "$SCRIPTS" "$LOGS" "$SOFT" "$REFDIR" "$GENOMES" "$FILTERED" "$DOWNSTREAM" "$PROVENANCE"
echo "config written: $ASAP_CONFIG"
echo "project root:   $ROOT"
echo "(the ASA3P version is read from your install in A5.4; no need to set it)"

config written: /home/s94m696/asap_shortname_projects/config.sh
project root:   /home/s94m696/asap_shortname_projects
(the ASA3P version is read from your install in A5.4; no need to set it)


### A0.2 Check the ASA³P install and keep the Barrnap fix in place
ASA³P's bundled Prokka could not find Barrnap in the first run; this link fixes the install without changing any ASA³P setting. The cell only creates it if missing.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

ls -lh "$ASAP_IMG" "$ASAPDIR/asap.jar" "$ASAP_TEMPLATE"

BARRNAP_LINK="$ASAPDIR/share/prokka-1.12/binaries/linux/barrnap"
if [ ! -e "$BARRNAP_LINK" ]; then
    ln -s ../../../barrnap-0.8/bin/barrnap "$BARRNAP_LINK"
    echo "created Barrnap link"
fi
ls -l "$BARRNAP_LINK"

apptainer exec --bind "$ASAPDIR":/asap:ro "$ASAP_IMG" bash -c '
  export PATH=/asap/share/prokka-1.12/binaries/linux:/asap/share/prokka-1.12/bin:/asap/share/barrnap-0.8/bin:$PATH
  echo -n "barrnap: "; barrnap --version 2>&1 | tail -1
  echo -n "prokka:  "; /asap/share/prokka/bin/prokka --version 2>&1 | tail -1'

### A0.3 Conda environments
Created only if missing. `download_env` runs the Python helpers; `genomeviz_env` is for Part D6 (`clinker_env` is optional, for the clinker cell there). The Part F tools get their own environments in F0.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

use_env base
have() { conda env list | awk '{print $1}' | grep -qx "$1"; }
have download_env  || conda create -y -n download_env  -c conda-forge "python>=3.9" aria2 xlrd xlwt xlutils
have genomeviz_env || conda create -y -n genomeviz_env -c conda-forge -c bioconda pygenomeviz mummer
have clinker_env   || conda create -y -n clinker_env   -c conda-forge -c bioconda clinker-py                    || echo "WARNING: clinker_env could not be created; it is optional (D6)"
use_env download_env
python -c "import sys, xlrd, xlwt, xlutils; print('download_env OK, python', sys.version.split()[0])"

### A0.4 Containers for Parts C and D
Not used by ASA³P itself. Links (does not copy) the Roary and Mashtree images from the first run, pulls only fastANI, AMRFinderPlus and ABRicate, and records exact versions in `software/container_versions.txt`. If your home quota is tight, set `APPTAINER_CACHEDIR` in `config.sh` first (commented line there).

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/pull_containers.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Pull (or re-use) every standalone container and record tool versions.
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"
mkdir -p "$SOFT"

get() {   # get <image path> <docker uri> [image from first run to link]
    local img=$1 uri=$2 old=${3:-}
    if [ -s "$img" ] || [ -L "$img" ]; then
        echo "present:  $img"
    elif [ -n "$old" ] && [ -s "$old" ]; then
        ln -s "$old" "$img"; echo "linked:   $img -> $old (first-run image, no copy)"
    else
        apptainer pull "$img" "$uri"; echo "pulled:   $img"
    fi
}

get "$ROARY_IMG"     docker://quay.io/biocontainers/roary:3.13.0--pl526h516909a_0 "$OLD_ROARY_IMG"
get "$MASHTREE_IMG"  docker://staphb/mashtree:latest "$OLD_MASHTREE_IMG"
get "$FASTANI_IMG"   docker://staphb/fastani:latest
get "$AMRFINDER_IMG" docker://staphb/ncbi-amrfinderplus:latest
get "$ABRICATE_IMG"  docker://staphb/abricate:latest

# ":latest" tags move over time, so write down exactly what was used (for the Methods)
{
    echo "roary:     $(apptainer exec "$ROARY_IMG" roary --version 2>&1 | tail -1)"
    echo "mashtree:  $(apptainer exec "$MASHTREE_IMG" mashtree --version 2>&1 | tail -1)"
    echo "fastANI:   $(apptainer exec "$FASTANI_IMG" fastANI --version 2>&1 | tail -1)"
    echo "amrfinder: $(apptainer exec "$AMRFINDER_IMG" amrfinder --version 2>&1 | tail -1)"
    echo "abricate:  $(apptainer exec "$ABRICATE_IMG" abricate --version 2>&1 | tail -1)"
} | tee "$SOFT/container_versions.txt"
END_OF_SCRIPT
chmod +x "$SCRIPTS/pull_containers.sh.new" && mv -f "$SCRIPTS/pull_containers.sh.new" "$SCRIPTS/pull_containers.sh"
echo "wrote $SCRIPTS/pull_containers.sh"

JOB=$(submit pull_containers 2 8G 02:00:00 "$SCRIPTS/pull_containers.sh"); echo "submitted $JOB"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like pull_containers
cat "$SOFT/container_versions.txt" 2>/dev/null || echo "not finished yet"

## A1. Isolate metadata and reads (single source of truth)
Upload **`metadata_shortnames.tsv`** (`full_sample, short_name, genus, species`) and **`source_metadata_correlated.tsv`** to `$META` (`/home/s94m696/asap_shortname_projects/metadata/`). All sample lists are derived from `metadata_shortnames.tsv`.

**How the long and short names fit together.** The raw files keep the long names the sequencing centre gave them, for example `PI003_resub_1-PI003_resub_1-CGGACCCGGACC_R1.fastq.gz`. In the table, `full_sample` is that name without `_R1.fastq.gz` / `_R2.fastq.gz`, and `short_name` (`PI003`) is what ASA³P sees. Nothing is renamed: A3 makes short-name symlinks to the long-name files, and A4 writes the short names into `config.xls`. If a long name in the table does not match the tarball exactly, A1.1 stops and lists the closest names in the tarball; fix `full_sample` in the table, not the files.

### A1.1 Reads: use the tarball in place, extract only these isolates
`TARBALL` in `config.sh` points to `/home/s94m696/Choudhury2_Project_002.tar.gz`. It is not copied, moved or edited. ASA³P needs plain `.fastq.gz` files, so the isolates in the table are extracted once into `READDIR` (a flat folder; folder prefixes inside the tarball are dropped). Files already in `READDIR` are skipped, so if your reads from the earlier run are still there, the step ends in seconds. Extracted FASTQ files are already compressed, so they take about as much space as the same files take inside the tarball; point `READDIR` at project or scratch storage in `config.sh` if your home quota is tight. The step runs on SLURM because a gzip stream has to be read from the start (minutes for tens of GB).

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

echo "tarball: $TARBALL"; ls -lh "$TARBALL"
echo
echo "first names in the tarball (reads only the start, so it is fast):"
tar -tzf "$TARBALL" 2>/dev/null | head -n 12
echo
echo "first names in the table (full_sample):"
tail -n +2 "$META/metadata_shortnames.tsv" | cut -f1 | head -n 5

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/extract_reads.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Pull only the isolates listed in metadata_shortnames.tsv out of the sequencing-centre tarball.
#
#  * The tarball is read in place. It is never copied, modified or renamed.
#  * Raw file names are NOT changed. The long names stay exactly as delivered; short names exist
#    only as symlinks (A3) and in config.xls (A4), and the map between them is metadata_shortnames.tsv.
#  * Folder prefixes inside the tarball are dropped, so READDIR is one flat folder.
#  * Files already present in READDIR are skipped, so the step is safe to re-run.
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

[ -n "${TARBALL:-}" ] || { echo "ERROR: TARBALL is not set in config.sh" >&2; exit 1; }
[ -s "$TARBALL" ]     || { echo "ERROR: tarball not found or empty: $TARBALL" >&2; exit 1; }
[ -s "$META/metadata_shortnames.tsv" ] || { echo "ERROR: upload metadata_shortnames.tsv to $META first" >&2; exit 1; }
mkdir -p "$READDIR" "$META" "$PROVENANCE"

WANTED="$META/reads_wanted.txt"     # every file the isolates need (R1 and R2), long names
TODO="$META/reads_todo.txt"         # the subset not yet in READDIR
tr -d '\r' < "$META/metadata_shortnames.tsv" |
    awk -F'\t' 'NR > 1 && $1 != "" {print $1 "_R1.fastq.gz"; print $1 "_R2.fastq.gz"}' > "$WANTED"
: > "$TODO"
while IFS= read -r f; do
    [ -s "$READDIR/$f" ] || echo "$f" >> "$TODO"
done < "$WANTED"

n_wanted=$(wc -l < "$WANTED"); n_todo=$(wc -l < "$TODO")
echo "tarball:  $TARBALL ($(du -h "$TARBALL" | cut -f1))"
echo "READDIR:  $READDIR"
echo "needed:   $n_wanted files for $((n_wanted / 2)) isolates; already present: $((n_wanted - n_todo)); to extract: $n_todo"
df -h "$READDIR" | tail -1
if [ "$n_todo" -eq 0 ]; then
    echo "Nothing to extract."
    {
        echo "reads already in: $READDIR ($((n_wanted / 2)) isolates, R1 and R2, long names unchanged)"
        echo "not re-extracted in this run; the names match metadata_shortnames.tsv, the contents were not compared with the tarball"
        echo "tarball for reference: $TARBALL ($(stat -c %s "$TARBALL") bytes, modified $(stat -c %y "$TARBALL"))"
        echo "checked on: $(date -Is)"
    } > "$PROVENANCE/reads_source.txt"
    echo "Record: $PROVENANCE/reads_source.txt"
    exit 0
fi

# gzip streams cannot be skipped through, so the whole tarball is read once (minutes for tens of GB)
DECOMP=(gzip -dc)
if command -v pigz >/dev/null 2>&1; then DECOMP=(pigz -dc -p "${SLURM_CPUS_PER_TASK:-1}"); fi

LOG="$LOGS/extract_reads_members.txt"; ERR="$LOGS/extract_reads_tar_stderr.txt"
mkdir -p "$LOGS"
# --no-anchored: a name matches after any folder prefix; --transform drops the prefix.
# tar exits 2 when a requested name is absent; that case is handled below, so do not stop here.
set +e
"${DECOMP[@]}" "$TARBALL" | tar -x -f - -C "$READDIR" --wildcards --no-anchored \
    --transform='s#.*/##' --verbose -T "$TODO" > "$LOG" 2> "$ERR"
rc=("${PIPESTATUS[@]}")
set -e
echo "decompress exit ${rc[0]}, tar exit ${rc[1]}; members extracted: $(wc -l < "$LOG")"
if [ "${rc[0]}" -ne 0 ]; then
    echo "ERROR: the tarball could not be read to the end (is the copy complete?)" >&2
    tail -5 "$ERR" >&2; exit 1
fi
if grep -qi "no space left\|disk quota\|write error" "$ERR"; then
    echo "ERROR: out of space while extracting. Free space or point READDIR at project/scratch storage." >&2
    exit 1
fi

# 1. every requested file must now exist
missing=0
while IFS= read -r f; do
    if [ ! -s "$READDIR/$f" ]; then echo "MISSING: $f"; missing=$((missing + 1)); fi
done < "$TODO"
# 2. a name that appears twice in the tarball would silently overwrite itself
if [ "$(wc -l < "$LOG")" -gt "$n_todo" ]; then
    echo "WARNING: $(wc -l < "$LOG") members matched $n_todo names; some names occur more than once in the tarball." >&2
    echo "         Check $LOG before using these reads." >&2
fi
if [ "$missing" -gt 0 ]; then
    echo
    echo "$missing file(s) are not in the tarball under the names in metadata_shortnames.tsv."
    echo "Listing the tarball (one more pass) to look for the closest names..."
    tar -tzf "$TARBALL" > "$META/tarball_listing.txt"
    echo "full listing: $META/tarball_listing.txt"
    while IFS= read -r f; do
        [ -s "$READDIR/$f" ] && continue
        code="${f%%[_-]*}"                       # PI003, PI024, ...
        echo "  wanted: $f"
        grep -F "$code" "$META/tarball_listing.txt" | sed 's#^#      in tarball: #' | head -6
    done < "$TODO"
    echo "Nothing was renamed. Fix full_sample in metadata_shortnames.tsv to the exact tarball name and re-run this step."
    exit 1
fi

# 3. integrity of what was just extracted
bad=$(while IFS= read -r f; do printf '%s\0' "$READDIR/$f"; done < "$TODO" |
      xargs -0 -P "${SLURM_CPUS_PER_TASK:-1}" -I{} sh -c 'gzip -t "$1" 2>/dev/null || echo "$1"' _ {})
if [ -n "$bad" ]; then echo "CORRUPT (gzip -t failed):"; echo "$bad"; exit 1; fi

# 4. provenance for the Methods
{
    echo "tarball:  $TARBALL"
    echo "size:     $(stat -c %s "$TARBALL") bytes"
    echo "modified: $(stat -c %y "$TARBALL")"
    echo "extracted into: $READDIR"
    echo "extracted on:   $(date -Is)"
    echo "files:    $n_wanted ($((n_wanted / 2)) isolates, R1 and R2), long names unchanged"
} > "$PROVENANCE/reads_source.txt"
echo "OK: $n_todo file(s) extracted and gzip-tested. Record: $PROVENANCE/reads_source.txt"
END_OF_SCRIPT
chmod +x "$SCRIPTS/extract_reads.sh.new" && mv -f "$SCRIPTS/extract_reads.sh.new" "$SCRIPTS/extract_reads.sh"
echo "wrote $SCRIPTS/extract_reads.sh"

JOB=$(submit extract_reads 8 16G 12:00:00 "$SCRIPTS/extract_reads.sh"); echo "submitted $JOB"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like extract_reads
LOG=$(latest "$LOGS"/extract_reads_*.out); echo "log: $LOG"; tail -n 25 "$LOG"
echo; echo "FASTQ files in READDIR: $(ls "$READDIR"/*.fastq.gz 2>/dev/null | wc -l)"
cat "$PROVENANCE/reads_source.txt" 2>/dev/null || echo "no record yet: the job has not finished, or it failed (read the log above)"

### A1.2 Check the table against the reads
The validator stops on an unsafe short name (letter first; letters, digits, `_`; ≤ 12 characters), a duplicate, or a missing FASTQ pair, and writes `samples_<Genus>.tsv`.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/validate_metadata.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Validate metadata_shortnames.tsv and write per-genus sample sheets.

metadata_shortnames.tsv is the single source of truth for isolate names.
Columns: full_sample, short_name, genus, species

Checks:
  * required columns present, no blank cells
  * full_sample and short_name both unique
  * short_name safe for every tool downstream: starts with a letter, only
    letters/digits/underscore, <= 12 characters (Prokka contig IDs must stay
    <= 37 chars even after "_contig_000001" is appended)
  * genus is one of the expected genera
  * paired FASTQs exist in READDIR (skipped with --no-reads-check)

Writes META/samples_<Genus>.tsv with columns: short_name, full_sample, species
Exit code 1 if any check fails.
"""
import argparse
import csv
import os
import re
import sys
from pathlib import Path

REQUIRED = ["full_sample", "short_name", "genus", "species"]
SAFE = re.compile(r"^[A-Za-z][A-Za-z0-9_]{1,11}$")
GENERA = {"Pectobacterium", "Dickeya"}

ap = argparse.ArgumentParser()
ap.add_argument("--metadata", default=os.path.join(os.environ.get("META", "."), "metadata_shortnames.tsv"))
ap.add_argument("--outdir", default=os.environ.get("META", "."))
ap.add_argument("--readdir", default=os.environ.get("READDIR", ""))
ap.add_argument("--no-reads-check", action="store_true")
args = ap.parse_args()

errors = []
with open(args.metadata, newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")
    missing_cols = [c for c in REQUIRED if c not in (reader.fieldnames or [])]
    if missing_cols:
        sys.exit(f"ERROR: missing columns {missing_cols} in {args.metadata}")
    rows = [{k: (v or "").strip() for k, v in r.items()} for r in reader if any((v or "").strip() for v in r.values())]

for i, r in enumerate(rows, start=2):
    for c in REQUIRED:
        if not r[c]:
            errors.append(f"line {i}: empty '{c}'")
    if r["short_name"] and not SAFE.match(r["short_name"]):
        errors.append(f"line {i}: short_name '{r['short_name']}' is not tool-safe "
                      "(letter first, letters/digits/_ only, max 12 chars)")
    if r["genus"] and r["genus"] not in GENERA:
        errors.append(f"line {i}: unexpected genus '{r['genus']}'")

for col in ("full_sample", "short_name"):
    seen = {}
    for r in rows:
        seen.setdefault(r[col], 0)
        seen[r[col]] += 1
    dups = [k for k, n in seen.items() if n > 1]
    if dups:
        errors.append(f"duplicate {col}: {dups}")

if not args.no_reads_check:
    readdir = Path(args.readdir)
    for r in rows:
        for mate in ("R1", "R2"):
            fq = readdir / f"{r['full_sample']}_{mate}.fastq.gz"
            if not fq.exists():
                errors.append(f"missing reads: {fq}")

if errors:
    print("VALIDATION FAILED:")
    for e in errors:
        print("  -", e)
    sys.exit(1)

outdir = Path(args.outdir)
outdir.mkdir(parents=True, exist_ok=True)
for genus in sorted({r["genus"] for r in rows}):
    sub = [r for r in rows if r["genus"] == genus]
    out = outdir / f"samples_{genus}.tsv"
    with out.open("w") as f:
        f.write("short_name\tfull_sample\tspecies\n")
        for r in sub:
            f.write(f"{r['short_name']}\t{r['full_sample']}\t{r['species']}\n")
    species_counts = {}
    for r in sub:
        species_counts[r["species"]] = species_counts.get(r["species"], 0) + 1
    print(f"{genus}: {len(sub)} isolates -> {out}")
    for sp, n in sorted(species_counts.items(), key=lambda x: -x[1]):
        print(f"    {genus} {sp}: {n}")

print(f"OK: {len(rows)} isolates passed all checks.")
END_OF_SCRIPT
chmod +x "$SCRIPTS/validate_metadata.py.new" && mv -f "$SCRIPTS/validate_metadata.py.new" "$SCRIPTS/validate_metadata.py"
echo "wrote $SCRIPTS/validate_metadata.py"

ls -l "$META"
use_env download_env
python "$SCRIPTS/validate_metadata.py"

## A2. Reference genomes
One reference per species for ASA³P's multi-reference scaffolding (MeDuSa) and mapping, plus the *Erwinia amylovora* outgroup, which is used only for the Mashtree tree in Part D and is never given to ASA³P. The 11 genomes you downloaded in the first run are copied from there (same labels and headers); only missing ones are downloaded, on the login node.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$META/reference_genomes.tsv" <<'END_OF_TABLE'
label	accession	genus	species	role
REF_Patrosepticum_SCRI1043	GCF_000011605.1	Pectobacterium	atrosepticum	reference
REF_Ppolaris_NIBIO1392	GCF_002288545.1	Pectobacterium	polaris	reference
REF_Pversatile_CFBP6051	GCF_004296685.1	Pectobacterium	versatile	reference
REF_Pcarotovorum	GCF_016864995.1	Pectobacterium	carotovorum	reference
REF_Pbrasiliense	GCF_009873295.1	Pectobacterium	brasiliense	reference
REF_Podoriferum	GCF_002904195.1	Pectobacterium	odoriferum	reference
REF_Dfangzhongdai	GCF_014854755.1	Dickeya	fangzhongdai	reference
REF_Dchrysanthemi_Ech1591	GCF_000023565.1	Dickeya	chrysanthemi	reference
REF_Ddadantii_3937	GCF_000147055.1	Dickeya	dadantii	reference
REF_Ddianthicola_ME23	GCF_003403135.1	Dickeya	dianthicola	reference
OUT_Erwinia_amylovora_ATCC49946	GCF_000027025.1	Erwinia	amylovora	outgroup
END_OF_TABLE
awk -F'\t' '{print NR": NF="NF"  "$1"  "$5}' "$META/reference_genomes.tsv"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/download_references.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Download reference genomes listed in META/reference_genomes.tsv.

Columns: label, accession, genus, species, role   (role = reference | outgroup)
Writes REFDIR/<label>.fasta with short headers (<label>_contig_000001 ...).
Already-present references are skipped, and references downloaded in the first
run (OLD_REFDIR, same labels and header format) are copied instead of downloaded.
Run on the login node (compute nodes usually have no internet access).
"""
import csv
import os
import shutil
import sys
import urllib.request
import zipfile
from pathlib import Path

META = Path(os.environ["META"])
REFDIR = Path(os.environ["REFDIR"])
DL = REFDIR / "downloads"
REFDIR.mkdir(parents=True, exist_ok=True)
DL.mkdir(parents=True, exist_ok=True)


def read_fasta(path):
    header, seq = None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq)
                header, seq = line[1:], []
            else:
                seq.append(line)
    if header is not None:
        yield header, "".join(seq)


def write_wrapped(out, seq, width=80):
    for i in range(0, len(seq), width):
        out.write(seq[i:i + width] + "\n")


with open(META / "reference_genomes.tsv", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

for r in rows:
    label, acc = r["label"], r["accession"]
    final = REFDIR / f"{label}.fasta"
    if final.exists() and final.stat().st_size > 0:
        print(f"exists, skipping: {final.name}")
        continue
    old = Path(os.environ.get("OLD_REFDIR", "/nonexistent")) / f"{label}.fna"
    if old.exists() and old.stat().st_size > 0:      # same labels and header format as the first run
        shutil.copyfile(old, final)
        print(f"re-used from first run: {final.name}")
        continue

    url = (f"https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/{acc}"
           f"/download?include_annotation_type=GENOME_FASTA")
    zpath = DL / f"{label}_{acc}.zip"
    xdir = DL / f"{label}_{acc}"
    if xdir.exists():
        shutil.rmtree(xdir)
    print(f"downloading {label} ({acc}) ...")
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(xdir)

    fnas = sorted(xdir.rglob("*.fna"))
    if not fnas:
        sys.exit(f"ERROR: no .fna inside download for {label} {acc}")

    n, total = 0, 0
    with final.open("w") as out:
        for fna in fnas:
            for _, seq in read_fasta(fna):
                n += 1
                total += len(seq)
                out.write(f">{label}_contig_{n:06d}\n")
                write_wrapped(out, seq)
    print(f"  wrote {final.name}: {n} sequences, {total:,} bp")

print("References ready in", REFDIR)
END_OF_SCRIPT
chmod +x "$SCRIPTS/download_references.py.new" && mv -f "$SCRIPTS/download_references.py.new" "$SCRIPTS/download_references.py"
echo "wrote $SCRIPTS/download_references.py"

use_env download_env
python "$SCRIPTS/download_references.py"
ls -lh "$REFDIR"/*.fasta

## A3. Project folders, short-name read links, references
Each FASTQ is linked into the project's `data/` folder under its short name (`PI024_2_R1.fastq.gz → PI024-2_resub_1-…_R1.fastq.gz`), which is what keeps the long names out of ASA³P. The genus's species references are copied next to the reads.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

for genus in $GENERA; do
    P="$(project_dir "$genus")"
    mkdir -p "$P"/{data,logs,scratch}
    n=0
    while IFS=$'\t' read -r short full species; do
        ln -sfn "$READDIR/${full}_R1.fastq.gz" "$P/data/${short}_R1.fastq.gz"
        ln -sfn "$READDIR/${full}_R2.fastq.gz" "$P/data/${short}_R2.fastq.gz"
        n=$((n + 1))
    done < <(tail -n +2 "$META/samples_${genus}.tsv")
    awk -F'\t' -v g="$genus" 'NR > 1 && $3 == g && $5 == "reference" {print $1}' "$META/reference_genomes.tsv" |
    while read -r label; do cp -f "$REFDIR/${label}.fasta" "$P/data/"; done
    broken=$(find "$P/data" -xtype l | wc -l)
    echo "$genus: $n isolates linked, $(ls "$P"/data/REF_*.fasta | wc -l) references, broken links: $broken"
done

## A4. `config.xls` for each genus
Short names in the Strains sheet, short-name read files, and all of the genus's species references in the Project sheet (MeDuSa is a multi-reference scaffolder, so this is an ASA³P option, not a change to the pipeline). Cell positions are the ones that passed `asap.jar -c` in the first run.

> If listing every species reference makes the run too slow (see `software/manual.pdf` for how ASA³P uses multiple references), set `role` to `skip` for references of single-isolate species in `reference_genomes.tsv` and re-run A3–A4.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/make_asap_config.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Write <ROOT>/ASA3P_<Genus>/config.xls from the ASA3P template.

Usage: make_asap_config.py <Genus>

Strain names are the SHORT names (column B of the Strains sheet), and the read
files are the short-name symlinks created earlier (<short>_R1/_R2.fastq.gz),
so every ASA3P step -- Prokka, Roary, ABR, reports -- sees short names.

All species references for the genus (role == reference in
reference_genomes.tsv) are listed, not just one, so isolates are scaffolded and
mapped against a reference from their own species where one exists.

Cell positions follow the template layout that already passed `asap.jar -c`
in the first run (Project sheet: rows 1-3, 7-9 and references from row 13;
Strains sheet: species, strain, input type, file 1, file 2, file 3).
"""
import csv
import os
import sys
from pathlib import Path

import xlrd
from xlutils.copy import copy as xl_copy

if len(sys.argv) != 2:
    sys.exit("Usage: make_asap_config.py <Genus>")
genus = sys.argv[1]

root = Path(os.environ["ROOT"])
meta = Path(os.environ["META"])
template = os.environ["ASAP_TEMPLATE"]
project_dir = root / f"ASA3P_{genus}"
out = project_dir / "config.xls"

with open(meta / f"samples_{genus}.tsv", newline="") as f:
    samples = list(csv.DictReader(f, delimiter="\t"))
with open(meta / "reference_genomes.tsv", newline="") as f:
    refs = [r for r in csv.DictReader(f, delimiter="\t")
            if r["genus"] == genus and r["role"] == "reference"]

if not samples:
    sys.exit(f"ERROR: no samples for {genus}")
if not refs:
    sys.exit(f"ERROR: no references for {genus}")

ref_files = [f"{r['label']}.fasta" for r in refs]
missing = [x for x in ref_files if not (project_dir / "data" / x).exists()]
if missing:
    sys.exit(f"ERROR: reference files not in {project_dir/'data'}: {missing}")
for s in samples:
    for mate in ("R1", "R2"):
        p = project_dir / "data" / f"{s['short_name']}_{mate}.fastq.gz"
        if not p.exists():
            sys.exit(f"ERROR: read link missing: {p}")

MAX_REF_ROWS = 27          # template rows 13..39 are cleared, as in the first run
if len(ref_files) > MAX_REF_ROWS:
    sys.exit("ERROR: too many references for the template block")

rb = xlrd.open_workbook(template, formatting_info=True)
wb = xl_copy(rb)
project = wb.get_sheet("Project")
strains = wb.get_sheet("Strains")

project.write(1, 1, f"ASA3P_{genus}_AVITI_shortnames")
project.write(2, 1, f"ASA3P analysis of {len(samples)} {genus} AVITI paired-end isolates (short sample names).")
project.write(3, 1, genus)
project.write(7, 1, os.environ.get("USER_FIRST", ""))
project.write(8, 1, os.environ.get("USER_SURNAME", ""))
project.write(9, 1, os.environ.get("USER_EMAIL", ""))

for row in range(13, 13 + MAX_REF_ROWS):
    project.write(row, 1, "")
for i, ref in enumerate(ref_files):
    project.write(13 + i, 1, ref)

for i, s in enumerate(samples, start=1):
    short = s["short_name"]
    strains.write(i, 0, s["species"])
    strains.write(i, 1, short)
    strains.write(i, 2, "paired-end")
    strains.write(i, 3, f"{short}_R1.fastq.gz")
    strains.write(i, 4, f"{short}_R2.fastq.gz")
    strains.write(i, 5, "")
for row in range(len(samples) + 1, 200):
    for col in range(6):
        strains.write(row, col, "")

wb.save(str(out))
print(f"Created {out}")
print(f"  {len(samples)} strains, {len(ref_files)} references:")
for x in ref_files:
    print(f"    {x}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/make_asap_config.py.new" && mv -f "$SCRIPTS/make_asap_config.py.new" "$SCRIPTS/make_asap_config.py"
echo "wrote $SCRIPTS/make_asap_config.py"

use_env download_env
for genus in $GENERA; do
    python "$SCRIPTS/make_asap_config.py" "$genus"
done

## A5. Run ASA³P
### A5.1 Config check (fast, interactive)
Expected for both genera: `All checks successfully passed.`

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_asap.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Run (or config-check) ASA3P for one genus project.
#   run_asap.sh <Genus> check   -> java -jar asap.jar -p /data -c   (fast, run interactively)
#   run_asap.sh <Genus> run     -> full local run with all allocated CPUs (submit via sbatch)
#
# Reads are symlinks into READDIR, so READDIR is bound at its own path inside
# the container; otherwise the links would dangle.
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

genus="${1:?genus required}"
mode="${2:-run}"
PROJECTDIR="$(project_dir "$genus")"
SCRATCHDIR="$PROJECTDIR/scratch"
mkdir -p "$SCRATCHDIR"

case "$mode" in
    check) ASAP_ARGS="-p /data -c" ;;
    run)   ASAP_ARGS="-p /data -l -s ${SLURM_CPUS_PER_TASK:-8}" ;;
    *)     echo "mode must be check or run"; exit 1 ;;
esac

echo "ASA3P $mode: $genus  ($(date))"
apptainer exec \
    --bind "$ASAPDIR":/asap:ro \
    --bind "$PROJECTDIR":/data \
    --bind "$SCRATCHDIR":/var/scratch \
    --bind "$READDIR":"$READDIR":ro \
    "$ASAP_IMG" \
    bash -c "export LC_ALL=C LANG=C
             export PATH=/asap/share/prokka-1.12/binaries/linux:/asap/share/prokka-1.12/bin:/asap/share/barrnap-0.8/bin:\$PATH
             java -jar /asap/asap.jar $ASAP_ARGS"
echo "ASA3P $mode finished: $genus  ($(date))"
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_asap.sh.new" && mv -f "$SCRIPTS/run_asap.sh.new" "$SCRIPTS/run_asap.sh"
echo "wrote $SCRIPTS/run_asap.sh"

for genus in $GENERA; do
    echo "===== $genus ====="
    bash "$SCRIPTS/run_asap.sh" "$genus" check
done

### A5.2 Submit the full runs
Safe to re-run: a genus whose `asap_<Genus>` job is still queued or running is skipped, so a second pair of jobs cannot start by accident.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/run_asap.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Run (or config-check) ASA3P for one genus project.
#   run_asap.sh <Genus> check   -> java -jar asap.jar -p /data -c   (fast, run interactively)
#   run_asap.sh <Genus> run     -> full local run with all allocated CPUs (submit via sbatch)
#
# Reads are symlinks into READDIR, so READDIR is bound at its own path inside
# the container; otherwise the links would dangle.
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

genus="${1:?genus required}"
mode="${2:-run}"
PROJECTDIR="$(project_dir "$genus")"
SCRATCHDIR="$PROJECTDIR/scratch"
mkdir -p "$SCRATCHDIR"

case "$mode" in
    check) ASAP_ARGS="-p /data -c" ;;
    run)   ASAP_ARGS="-p /data -l -s ${SLURM_CPUS_PER_TASK:-8}" ;;
    *)     echo "mode must be check or run"; exit 1 ;;
esac

echo "ASA3P $mode: $genus  ($(date))"
apptainer exec \
    --bind "$ASAPDIR":/asap:ro \
    --bind "$PROJECTDIR":/data \
    --bind "$SCRATCHDIR":/var/scratch \
    --bind "$READDIR":"$READDIR":ro \
    "$ASAP_IMG" \
    bash -c "export LC_ALL=C LANG=C
             export PATH=/asap/share/prokka-1.12/binaries/linux:/asap/share/prokka-1.12/bin:/asap/share/barrnap-0.8/bin:\$PATH
             java -jar /asap/asap.jar $ASAP_ARGS"
echo "ASA3P $mode finished: $genus  ($(date))"
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_asap.sh.new" && mv -f "$SCRIPTS/run_asap.sh.new" "$SCRIPTS/run_asap.sh"
echo "wrote $SCRIPTS/run_asap.sh"

for genus in $GENERA; do
    active=$(squeue -u "$USER" -h -n "asap_${genus}" -o "%i %T" 2>/dev/null | head -1)
    if [ -n "$active" ]; then
        echo "$genus: NOT submitted, a job is already active (job state: $active). Use A5.3 to monitor it."
        continue
    fi
    JOB=$(submit "asap_${genus}" 16 128G 5-00:00:00 "$SCRIPTS/run_asap.sh" "$genus" run)
    echo "$genus: job $JOB"
done

### A5.3 Monitor
Shows the newest log for each genus. The first lines of the log are also where to look for the ASA³P version if you have not filled in `ASAP_VERSION`.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like asap_
for genus in $GENERA; do
    echo; echo "===== $genus: $(latest "$LOGS"/asap_"${genus}"_*.out) ====="
    tail -n 30 "$(latest "$LOGS"/asap_"${genus}"_*.out)" 2>/dev/null
    tail -n 30 "$(latest "$LOGS"/asap_"${genus}"_*.err)" 2>/dev/null
done

### A5.4 Which ASA³P version ran
Read from the installed `asap.jar`, or from ASA³P's run log; if neither states it, the Zenodo download version (1.2.2, from `config.sh`) is used and the Methods notes mark it as unconfirmed. Re-run after the jobs finish.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/detect_asap_version.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Find the ASA3P version that is actually installed and used, instead of guessing.

Tried in order (first hit wins):
  1. the version stamped inside asap.jar (META-INF/MANIFEST.MF)
  2. a version line printed by ASA3P in its run logs (LOGS/asap_*.out|err,
     ROOT/ASA3P_<Genus>/*.log)
  3. ASAP_VERSION from config.sh (the Zenodo download version, 1.2.2) --
     marked "unconfirmed" so the Methods notes ask you to check it

Writes PROVENANCE/asap_version.txt as "<version>\t<source>" and prints it.
"""
import os
import re
import sys
import zipfile
from pathlib import Path

VERSION = re.compile(r"\b(\d+\.\d+\.\d+)\b")


def from_jar(jar):
    try:
        with zipfile.ZipFile(jar) as z:
            text = z.read("META-INF/MANIFEST.MF").decode("utf-8", "replace")
    except (OSError, KeyError, zipfile.BadZipFile):
        return None
    for line in text.splitlines():
        if re.match(r"(Implementation|Specification|Bundle)-Version\s*:", line, re.I):
            m = VERSION.search(line)
            if m:
                return m.group(1)
    return None


def from_logs(paths):
    pat = re.compile(r"(asa.?3?p|asap)[^\n]{0,40}?\bv(?:ersion)?[\s:=]*v?(\d+\.\d+\.\d+)", re.I)
    for p in paths:
        try:
            with open(p, errors="replace") as f:
                for i, line in enumerate(f):
                    m = pat.search(line)
                    if m:
                        return m.group(2), p
                    if i > 500:          # the version is printed near the top
                        break
        except OSError:
            continue
    return None, None


def detect():
    root, logs = Path(os.environ["ROOT"]), Path(os.environ["LOGS"])
    jar = Path(os.environ["ASAPDIR"]) / "asap.jar"
    v = from_jar(jar)
    if v:
        return v, f"asap.jar manifest ({jar})"
    logfiles = sorted(logs.glob("asap_*.out")) + sorted(logs.glob("asap_*.err")) + sorted(root.glob("ASA3P_*/*.log"))
    v, src = from_logs(logfiles)
    if v:
        return v, f"ASA3P run log ({src})"
    fallback = os.environ.get("ASAP_VERSION", "").strip()
    if fallback:
        return fallback, "unconfirmed: Zenodo download version from config.sh"
    return "", "unknown"


if __name__ == "__main__":
    version, source = detect()
    prov = Path(os.environ["PROVENANCE"])
    prov.mkdir(parents=True, exist_ok=True)
    (prov / "asap_version.txt").write_text(f"{version}\t{source}\n")
    print(f"ASA3P version: {version or '[unknown]'}   (source: {source})")
    if source.startswith("unconfirmed"):
        print("Not found in asap.jar or the run logs yet. Re-run this after ASA3P has run; "
              "if it still says unconfirmed, 1.2.2 (your Zenodo download) is the value to report.")
    sys.exit(0)
END_OF_SCRIPT
chmod +x "$SCRIPTS/detect_asap_version.py.new" && mv -f "$SCRIPTS/detect_asap_version.py.new" "$SCRIPTS/detect_asap_version.py"
echo "wrote $SCRIPTS/detect_asap_version.py"

use_env download_env
python "$SCRIPTS/detect_asap_version.py"

## A6. ASA³P results (these are the results to report)
Everything ASA³P produced, per step, with the tool ASA³P uses for it (Schwengers et al. 2020). Open `reports/index.html` for each genus (copy it to your laptop) and confirm every isolate appears in each table before relying on a step.

| ASA³P folder | Step | Tool inside ASA³P |
|---|---|---|
| `reads_qc/` | Read QC and trimming | FastQC, Trimmomatic |
| `assemblies/` | *De novo* assembly | SPAdes |
| `scaffolds/` | Reference-guided contig ordering | MeDuSa (multi-reference) |
| `annotations/` | Annotation | Prokka |
| `taxonomy/` | Taxonomic classification | Kraken, 16S BLAST+ vs. SILVA, nucmer ANI |
| `mlst/` | MLST | BLAST+ vs. PubMLST |
| `abr/` | Resistance genes | RGI + CARD |
| `vf/` | Virulence factors | BLAST+ vs. VFDB |
| `mappings/` | Read mapping | Bowtie2 |
| `snps/` | SNP detection | SAMtools, SnpEff |
| `corepan/` | Core/pan-genome (whole project = genus) | Roary |
| `phylogeny/` | Phylogeny | FastTreeMP |

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

for genus in $GENERA; do
    P="$(project_dir "$genus")"
    echo "===== $genus ====="
    ls -lh "$P/reports/index.html"
    for m in reads_qc assemblies scaffolds annotations taxonomy mlst abr vf mappings snps corepan phylogeny; do
        printf '  %-12s %6s files\n' "$m" "$(find "$P/$m" -type f 2>/dev/null | wc -l)"
    done
done

# Part B — Check what ASA³P produced
**`[Check]`** — bookkeeping only; no analysis.

## B1. Audit outputs and standardize contigs
For each isolate: finds the raw assembly in `assemblies/` (never the reference-ordered `*-pseudo.fasta`), the GFF and GenBank files, and copies the contigs to `genomes/<short>.fna` with headers `<short>_contig_000001` (`<short>.header_map.tsv` keeps the original names). These standardized contigs are the input for Parts C and D. Flags:

* ASA³P steps whose folder is empty, or that ASA³P's own run log reports as failed (a failed step can still leave files in its folder)
* isolates with no GFF
* GFFs with 1–3 sequences for a many-contig assembly → ASA³P annotated the **pseudo-genome**
* annotation covering < 98 % of the assembly → contigs dropped

Files are matched on exact name boundaries, so `PI024` can never pick up `PI024_2`'s files.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/locate_asap_outputs.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Audit ASA3P outputs, locate per-isolate files, and standardize contigs.

For each genus project (ROOT/ASA3P_<Genus>):
  1. Module audit: which ASA3P output folders exist, how many files each
     holds, and what ASA3P's own run log says about that step (the final
     "genome analyses statistics" block of the newest asap_<Genus>_*.out)
     ->  DOWNSTREAM/asap_module_audit.tsv
     A folder can hold files although ASA3P reported the step as failed, so a
     module is "usable" only if it has files AND the log shows no failure.
  2. Per isolate: find the raw assembly (assemblies/, never the reference-
     ordered *-pseudo* file), the GFF and the GenBank file from annotations/
  3. Copy the assembly to GENOMES/<short>.fna with headers <short>_contig_NNNNNN
     (plus <short>.header_map.tsv so every new header traces back)
  4. QC: assembly contigs / length / N50 vs. the sequences declared in the GFF.
     Flags (a) a GFF with 1-3 sequences for a many-contig assembly, i.e. ASA3P
     annotated the reference-ordered pseudo-genome; (b) annotation covering
     < 98% of the assembly, i.e. contigs were dropped.
  ->  DOWNSTREAM/asap_paths.tsv

File-to-isolate matching is by exact name boundaries, so PI024 can never
claim PI024_2's files (the substring match in the first run allowed that).
"""
import csv
import os
import re
import sys
from pathlib import Path

ROOT = Path(os.environ["ROOT"])
META = Path(os.environ["META"])
GENOMES = Path(os.environ["GENOMES"])
DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
GENERA = os.environ.get("GENERA", "Pectobacterium Dickeya").split()

LOGS = Path(os.environ.get("LOGS", ""))

# module folder -> label used in ASA3P's "<label> steps:  ok | err | skip" lines
LOG_STEP = {"reads_qc": "qc", "assemblies": "assembly", "scaffolds": "scaffolding", "annotations": "annotation",
            "taxonomy": "taxonomy", "mlst": "mlst", "abr": "abr", "vf": "vf", "mappings": "mapping", "snps": "snps"}
LOG_COMPARATIVE = {"phylogeny": "Phylogeny", "corepan": "Core/Pan Genome"}
ANSI_RE = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")
STEP_RE = re.compile(r"^\s*([A-Za-z_]+) steps:\s+(\d+)\s*\|\s*(\d+)\s*\|\s*(\d+)")
COMP_RE = re.compile(r"^\s*(Pipeline|Phylogeny|Core/Pan Genome):\s+(\w+)")


def asap_log_summary(genus):
    """Newest ASA3P run log of the genus -> (per-isolate step counts, comparative step status, log path).
    The first two are None when the log has no final statistics block (run not finished, or no log)."""
    logs = sorted(LOGS.glob(f"asap_{genus}_*.out"), key=lambda p: p.stat().st_mtime) if LOGS.is_dir() else []
    if not logs:
        return None, None, None
    steps, comp, seen = {}, {}, False
    for line in logs[-1].read_text(errors="replace").splitlines():
        line = ANSI_RE.sub("", line)
        if line.strip().startswith("genome analyses statistics"):
            seen, steps, comp = True, {}, {}          # a re-run appends a new block: keep the last one
            continue
        if not seen:
            continue
        m = STEP_RE.match(line)
        if m:
            steps[m.group(1)] = tuple(int(x) for x in m.groups()[1:])
        m = COMP_RE.match(line)
        if m:
            comp[m.group(1)] = m.group(2)
    return (steps, comp, logs[-1]) if seen else (None, None, logs[-1])


def module_log_status(module, steps, comp):
    """-> (text for the audit table, False if ASA3P itself reported the step as failed)"""
    if steps is None:
        return "no final statistics in the log", True
    if module in LOG_STEP:
        t = steps.get(LOG_STEP[module])
        if t is None:
            return "not listed in the log", True
        ok, err, skip = t
        return f"{ok} ok / {err} failed" + (f" / {skip} skipped" if skip else ""), not (err > 0 or ok == 0)
    if module in LOG_COMPARATIVE:
        s = comp.get(LOG_COMPARATIVE[module])
        return (s, s.lower() == "finished") if s else ("not reported", False)
    return "", True


MODULES = ["reads_qc", "assemblies", "scaffolds", "annotations", "taxonomy", "mlst",
           "abr", "vf", "mappings", "snps", "phylogeny", "corepan", "reports"]
FASTA_EXT = {".fasta", ".fa", ".fna", ".fas", ".contigs"}
GFF_EXT = {".gff", ".gff3"}
GBK_EXT = {".gbk", ".gbff", ".gb"}
AVOID = re.compile(r"pseudo|unscaffold|unmapped|discard|filtered_out", re.I)


def root_name(p: Path) -> str:
    return p.name.split(".")[0]


def genome_id(p: Path, base: Path) -> str:
    """ASA3P names each genome <Genus>_<species>_<short_name> (e.g. Pectobacterium_atrosepticum_PI004):
    a sub-folder of the module folder (assemblies/<id>/<id>.fasta) or, in abr/, flat <id>.failed files."""
    parts = p.relative_to(base).parts
    gid = parts[0] if len(parts) > 1 else p.name
    for suffix in (".std.log", ".failed", ".fasta", ".gff", ".gbk", ".faa", ".ffn"):
        if gid.endswith(suffix):
            gid = gid[: -len(suffix)]
    return re.sub(r"-(discarded|pseudo)$", "", gid)


def id_matches(gid: str, s: str) -> bool:
    """True if genome id gid belongs to isolate s: equal, or ends in _<s>.  Because the match is on the END of
    the id, PI024 never claims Pectobacterium_x_PI024_2 (which ends in _PI024_2)."""
    return gid == s or gid.endswith("_" + s)


def matches_id_or_bare(gid: str, s: str) -> bool:
    return id_matches(gid, s) or gid.startswith(s + "-") or gid.startswith(s + "_")


def matches(p: Path, base: Path, s: str) -> bool:
    if id_matches(genome_id(p, base), s):          # real ASA3P naming
        return True
    parts = p.relative_to(base).parts               # fallback: bare short names as folder or file prefix
    if s in parts[:-1]:
        return True
    r = root_name(p)
    return r == s or any(r.startswith(s + sep) for sep in "-_")


def owner_candidates(files, base, s, all_names):
    longer = [t for t in all_names if t != s and t.startswith(s)]
    out = []
    for p in files:
        if matches(p, base, s) and not any(matches(p, base, t) for t in longer):
            out.append(p)
    return out


def pick(cands, s, label, notes):
    if not cands:
        return None
    clean = [p for p in cands if not AVOID.search(p.name)] or cands
    exact = [p for p in clean if root_name(p) == s] or clean
    exact.sort(key=lambda p: (-p.stat().st_size, str(p)))
    if len(exact) > 1:
        notes.append(f"{label}: {len(exact)} candidates, chose {exact[0].name}")
    return exact[0]


def read_fasta(path):
    header, seq = None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq)
                header, seq = line[1:], []
            else:
                seq.append(line)
    if header is not None:
        yield header, "".join(seq)


def n50(lengths):
    total, acc = sum(lengths), 0
    for L in sorted(lengths, reverse=True):
        acc += L
        if acc >= total / 2:
            return L
    return 0


def gff_regions(path):
    total, n = 0, 0
    with open(path) as f:
        for line in f:
            if line.startswith("##sequence-region"):
                parts = line.split()
                total += int(parts[3]) - int(parts[2]) + 1
                n += 1
            elif line.startswith("##FASTA"):
                break
    return total, n


GENOMES.mkdir(parents=True, exist_ok=True)
DOWNSTREAM.mkdir(parents=True, exist_ok=True)

audit_rows, path_rows, problems = [], [], []

for genus in GENERA:
    proj = ROOT / f"ASA3P_{genus}"
    sheet = META / f"samples_{genus}.tsv"
    if not proj.exists() or not sheet.exists():
        print(f"skip {genus}: project or sample sheet missing")
        continue
    with open(sheet, newline="") as f:
        samples = list(csv.DictReader(f, delimiter="\t"))
    names = [s["short_name"] for s in samples]

    log_steps, log_comp, log_path = asap_log_summary(genus)
    if log_path is None:
        print(f"WARNING {genus}: no ASA3P run log (asap_{genus}_*.out) in {LOGS}; judging steps by their files only.")
    elif log_steps is None:
        print(f"WARNING {genus}: {log_path.name} has no final statistics block; has the run finished? Judging steps by their files only.")
    for m in MODULES:
        d = proj / m
        n = sum(1 for p in d.rglob("*") if p.is_file()) if d.exists() else 0
        text, log_ok = module_log_status(m, log_steps, log_comp)
        audit_rows.append({"genus": genus, "module": m, "exists": d.exists(), "n_files": n,
                           "log_status": text, "usable": "yes" if n > 0 and log_ok else "no"})

    asm_files = [p for p in (proj / "assemblies").rglob("*")
                 if p.is_file() and p.suffix.lower() in FASTA_EXT] if (proj / "assemblies").exists() else []
    ann_dir = proj / "annotations"
    ann_files = [p for p in ann_dir.rglob("*") if p.is_file()] if ann_dir.exists() else []

    for s in samples:
        short, species = s["short_name"], s["species"]
        notes = []
        asm = pick(owner_candidates(asm_files, proj / "assemblies", short, names), short, "assembly", notes)
        gff = pick(owner_candidates([p for p in ann_files if p.suffix.lower() in GFF_EXT], ann_dir, short, names),
                   short, "gff", notes)
        gbk = pick(owner_candidates([p for p in ann_files if p.suffix.lower() in GBK_EXT], ann_dir, short, names),
                   short, "gbk", notes)

        row = {"short_name": short, "genus": genus, "species": species,
               "assembly": str(asm or ""), "gff": str(gff or ""), "gbk": str(gbk or ""),
               "genome_fna": "", "n_contigs": "", "assembly_bp": "", "n50": "",
               "n_bases_N": "", "gff_regions": "", "gff_seq_bp": "", "annotated_over_assembly": "", "notes": ""}

        if asm:
            out_fna = GENOMES / f"{short}.fna"
            lengths, n_count = [], 0
            with out_fna.open("w") as out, open(GENOMES / f"{short}.header_map.tsv", "w") as hm:
                hm.write("new_header\toriginal_header\tlength\n")
                for i, (h, seq) in enumerate(read_fasta(asm), start=1):
                    new = f"{short}_contig_{i:06d}"
                    out.write(f">{new}\n")
                    for k in range(0, len(seq), 80):
                        out.write(seq[k:k + 80] + "\n")
                    hm.write(f"{new}\t{h}\t{len(seq)}\n")
                    lengths.append(len(seq))
                    n_count += seq.upper().count("N")
            row.update(genome_fna=str(out_fna), n_contigs=len(lengths), assembly_bp=sum(lengths),
                       n50=n50(lengths), n_bases_N=n_count)
        else:
            problems.append(f"{genus} {short}: no assembly found")

        if gff:
            g, nreg = gff_regions(gff)
            row["gff_seq_bp"], row["gff_regions"] = g, nreg
            if asm and sum(lengths):
                ratio = g / sum(lengths)
                row["annotated_over_assembly"] = f"{ratio:.3f}"
                if ratio < 0.98:
                    notes.append(f"annotation covers only {ratio:.1%} of the assembly (contigs dropped)")
                elif ratio > 1.02:
                    notes.append(f"annotated sequence longer than assembly ({ratio:.3f}): N-linked pseudo-genome?")
                if nreg <= 3 < len(lengths):
                    notes.append(f"GFF has {nreg} sequence(s) but the assembly has {len(lengths)} contigs: "
                                 "ASA3P annotated the reference-ordered pseudo-genome")
        else:
            problems.append(f"{genus} {short}: no GFF found (see fallback Prokka section)")

        if asm and len(lengths) <= 3:
            notes.append("<=3 sequences: looks like a pseudo-genome, not raw contigs")
        row["notes"] = "; ".join(notes)
        if notes:
            problems.append(f"{genus} {short}: " + "; ".join(notes))
        path_rows.append(row)

    # genome folders that no isolate in the sample sheet claims (catches a naming mismatch at once)
    seen_ids = sorted({genome_id(p, proj / "assemblies") for p in asm_files} |
                      {genome_id(p, ann_dir) for p in ann_files})
    unclaimed = [g for g in seen_ids if not any(matches_id_or_bare(g, n) for n in names)]
    if unclaimed:
        problems.append(f"{genus}: {len(unclaimed)} genome folder(s) in assemblies/annotations match no isolate "
                        f"in the sample sheet, e.g. {', '.join(unclaimed[:3])}")

if not path_rows:
    sys.exit("ERROR: no isolates processed -- are the ASA3P runs finished and the sample sheets in META?")

with open(DOWNSTREAM / "asap_module_audit.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["genus", "module", "exists", "n_files", "log_status", "usable"], delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(audit_rows)

fields = list(path_rows[0].keys()) if path_rows else ["short_name"]
with open(DOWNSTREAM / "asap_paths.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields, delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(path_rows)

print("ASA3P module audit (files in the folder | what ASA3P's log reports for the step):")
for r in audit_rows:
    flag = ""
    if not r["n_files"]:
        flag = "   <-- EMPTY / MISSING"
    elif r["usable"] == "no":
        flag = "   <-- FILES PRESENT BUT ASA3P REPORTED THE STEP AS FAILED"
    print(f"  {r['genus']:15s} {r['module']:12s} {r['n_files']:6d}  {r['log_status']}{flag}")
print()
print(f"{'isolate':9s} {'contigs':>7s} {'assembly_bp':>12s} {'N50':>9s} {'gff_seqs':>8s} {'ann/asm':>8s}")
for r in path_rows:
    print(f"{r['short_name']:9s} {str(r['n_contigs']):>7s} {str(r['assembly_bp']):>12s} "
          f"{str(r['n50']):>9s} {str(r['gff_regions'] or 'NO GFF'):>8s} {str(r['annotated_over_assembly']):>8s}")
print()
if problems:
    print("Needs attention:")
    for p in problems:
        print("  -", p)
else:
    print("All isolates have an assembly and a GFF; annotated length matches assembly length.")
print(f"\nWrote {DOWNSTREAM/'asap_paths.tsv'} and {DOWNSTREAM/'asap_module_audit.tsv'}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/locate_asap_outputs.py.new" && mv -f "$SCRIPTS/locate_asap_outputs.py.new" "$SCRIPTS/locate_asap_outputs.py"
echo "wrote $SCRIPTS/locate_asap_outputs.py"

use_env download_env
python "$SCRIPTS/locate_asap_outputs.py"

## B2. Decide which ASA³P steps need a replacement
Per genus: a step is used as-is only if its folder holds files **and** ASA³P's final statistics (the end of `logs/asap_<Genus>_<job>.out`) report no failed isolates for it. A replacement runs if the folder is empty or the log reports the step as failed; ASA³P's partial files for that step are then not used. The decision table is saved to `downstream/provenance/replacements.tsv` and used by Part C and the Methods draft.

| ASA³P step | Replacement outside ASA³P (only if empty or failed) |
|---|---|
| `abr` (RGI + CARD) | AMRFinderPlus + ABRicate/CARD (C2) |
| `vf` (BLAST+ / VFDB) | ABRicate/VFDB (C3) |
| `corepan` (Roary, genus) | Roary container, same genus, `-i 80` (C4) |
| `phylogeny` (FastTreeMP) | none; the Mashtree tree (D3) is used |
| `annotations` | decided per isolate in C1 |

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/decide_replacements.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Decide, per genus, which ASA3P steps need a replacement run outside ASA3P.

Rule: an ASA3P step is used as-is only if its output folder holds files AND ASA3P's own
run log does not report the step as failed ("usable" column of asap_module_audit.tsv).
A replacement runs for a step whose folder is empty or missing, or whose steps the log
reports as failed (a failed step can still leave files behind).

  step       ASA3P tool                      replacement outside ASA3P
  abr        RGI + CARD                      AMRFinderPlus + ABRicate (CARD)
  vf         BLAST+ + VFDB                   ABRicate (VFDB)
  corepan    Roary (whole genus project)     Roary 3.13.0 container, same genus, -i 80
  phylogeny  FastTreeMP                      none -- the Mashtree tree (Part D) is used

Annotation is decided per isolate, not per genus (see make_prokka_todo.py).

Reads   DOWNSTREAM/asap_module_audit.tsv   (from locate_asap_outputs.py)
Writes  PROVENANCE/replacements.tsv        step, genus, asa3p_files, decision, reason
"""
import csv
import os
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
PROV = Path(os.environ["PROVENANCE"])
PROV.mkdir(parents=True, exist_ok=True)

REPLACEMENT = {
    "abr": "replacement: AMRFinderPlus + ABRicate/CARD",
    "vf": "replacement: ABRicate/VFDB",
    "corepan": "replacement: Roary container (genus level)",
    "phylogeny": "no replacement: use Mashtree tree (Part D)",
}

with open(DOWNSTREAM / "asap_module_audit.tsv", newline="") as f:
    audit = list(csv.DictReader(f, delimiter="\t"))

rows = []
for a in audit:
    step = a["module"]
    if step not in REPLACEMENT:
        continue
    n = int(a["n_files"])
    usable = a.get("usable") or ("yes" if n > 0 else "no")      # older audit files have no "usable" column
    log = a.get("log_status", "")
    if usable == "yes":
        decision, reason = "use ASA3P output", log
    elif n == 0:
        decision, reason = REPLACEMENT[step], "folder empty or missing"
    else:
        decision, reason = REPLACEMENT[step], f"ASA3P log reports the step as failed ({log}); files in the folder are not used"
    rows.append({"step": step, "genus": a["genus"], "asa3p_files": n, "decision": decision, "reason": reason})

with open(PROV / "replacements.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["step", "genus", "asa3p_files", "decision", "reason"], delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)

print(f"{'step':10s} {'genus':15s} {'ASA3P files':>11s}  decision  (reason)")
for r in rows:
    print(f"{r['step']:10s} {r['genus']:15s} {r['asa3p_files']:>11d}  {r['decision']}  ({r['reason']})")
n_rep = sum(r["decision"].startswith("replacement") for r in rows)
print(f"\n{n_rep} replacement run(s) needed. Wrote {PROV/'replacements.tsv'}")
print("Open the ASA3P report (reports/index.html) and check that every isolate appears before relying on a step.")
END_OF_SCRIPT
chmod +x "$SCRIPTS/decide_replacements.py.new" && mv -f "$SCRIPTS/decide_replacements.py.new" "$SCRIPTS/decide_replacements.py"
echo "wrote $SCRIPTS/decide_replacements.py"

use_env download_env
python "$SCRIPTS/decide_replacements.py"

# Part C — Replacements (only where ASA³P's output is missing or failed)
**`[Replacement — runs only if needed]`** — redoes an ASA³P step with another tool, only when ASA³P's own output for it is missing or failed.

Each cell below checks the decision from Part B and does nothing when ASA³P's own result exists.

## C1. Annotation, per isolate (Prokka outside ASA³P)
Only isolates whose ASA³P annotation is missing, was made on the pseudo-genome, dropped contigs, or lacks the `##FASTA` block Roary needs. Uses the same Prokka 1.12 and ASA³P protein database from the ASA³P image, on the standardized raw contigs, one isolate per SLURM array task. The reason for each isolate is saved in `downstream/prokka_fallback/todo.tsv`.

`ANNOTATE_ALL=yes` re-annotates all 25 this way instead (one consistent annotation set); the Methods draft then says so.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/make_prokka_todo.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""List isolates that need the fallback Prokka run.

Default: isolates where ASA3P produced no GFF, annotated the pseudo-genome,
dropped contigs (flags written by locate_asap_outputs.py), or wrote a GFF
without the ##FASTA block Roary needs.
--all: every isolate, so all 25 annotations come from the same raw-contig
       Prokka run (most consistent input for Roary).
  -> DOWNSTREAM/prokka_fallback/todo.tsv  (short_name, genus, species, reason)
--filtered: only the isolates you set to use_filtered in isolate_overrides.tsv
       whose contig-filtered genome exists (D1.5); annotated from that genome.
  -> DOWNSTREAM/prokka_fallback/todo_filtered.tsv  (todo.tsv is left alone)
Prints the --array range to pass to sbatch.
"""
import csv
import os
import sys
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
annotate_all = "--all" in sys.argv
filtered_only = "--filtered" in sys.argv
FILTERED = Path(os.environ.get("FILTERED", Path(os.environ["ROOT"]) / "genomes_filtered"))

with open(DOWNSTREAM / "asap_paths.tsv", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

PROBLEMS = ("pseudo-genome", "contigs dropped")


def lacks_fasta(gff):
    with open(gff) as f:
        return not any(line.startswith("##FASTA") for line in f)


def reason(r):
    if not r["gff"]:
        return "ASA3P produced no annotation"
    if "pseudo-genome" in r["notes"]:
        return "ASA3P annotated the reference-ordered pseudo-genome"
    if "contigs dropped" in r["notes"]:
        return "ASA3P annotation omitted contigs"
    if lacks_fasta(r["gff"]):
        return "ASA3P GFF lacks the sequence block Roary needs"
    return "re-annotated for consistency (--all)" if annotate_all else ""


if filtered_only:
    gate_path = DOWNSTREAM / "species_check" / "analysis_set.tsv"
    if not gate_path.exists():
        raise SystemExit("ERROR: no species gate yet -- run D1.2 first.")
    with open(gate_path, newline="") as f:
        wanted = {g["short_name"]: g for g in csv.DictReader(f, delimiter="\t") if g["override"] == "use_filtered"}
    todo = []
    for r in rows:
        g = wanted.get(r["short_name"])
        if g is None:
            continue
        if not (FILTERED / f"{r['short_name']}.fna").exists():
            print(f"{r['short_name']}: use_filtered is set but {FILTERED / (r['short_name'] + '.fna')} does not exist (run D1.5; "
                  "the split may have judged the kept contigs not plausible as one genome)")
            continue
        r = dict(r, genus=g["genus_final"] or r["genus"], species=g["species_final"] or r["species"])
        todo.append((r, "contig-filtered genome (second organism removed)"))
else:
    todo = [(r, reason(r)) for r in rows]
    todo = [(r, why) for r, why in todo if why and r["genome_fna"]]   # need standardized contigs as input

out_dir = DOWNSTREAM / "prokka_fallback"
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / ("todo_filtered.tsv" if filtered_only else "todo.tsv"), "w") as f:
    f.write("short_name\tgenus\tspecies\treason\n")
    for r, why in todo:
        f.write(f"{r['short_name']}\t{r['genus']}\t{r['species']}\t{why}\n")

print(f"{len(todo)} isolate(s) to annotate outside ASA3P{' (all)' if annotate_all else ''}{' (contig-filtered genomes)' if filtered_only else ''}:")
for r, why in todo:
    print(f"  {r['short_name']:9s} {why}")
print(f"ARRAY=1-{len(todo)}" if todo else "ARRAY=")
END_OF_SCRIPT
chmod +x "$SCRIPTS/make_prokka_todo.py.new" && mv -f "$SCRIPTS/make_prokka_todo.py.new" "$SCRIPTS/make_prokka_todo.py"
echo "wrote $SCRIPTS/make_prokka_todo.py"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/make_prokka_todo.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""List isolates that need the fallback Prokka run.

Default: isolates where ASA3P produced no GFF, annotated the pseudo-genome,
dropped contigs (flags written by locate_asap_outputs.py), or wrote a GFF
without the ##FASTA block Roary needs.
--all: every isolate, so all 25 annotations come from the same raw-contig
       Prokka run (most consistent input for Roary).
  -> DOWNSTREAM/prokka_fallback/todo.tsv  (short_name, genus, species, reason)
--filtered: only the isolates you set to use_filtered in isolate_overrides.tsv
       whose contig-filtered genome exists (D1.5); annotated from that genome.
  -> DOWNSTREAM/prokka_fallback/todo_filtered.tsv  (todo.tsv is left alone)
Prints the --array range to pass to sbatch.
"""
import csv
import os
import sys
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
annotate_all = "--all" in sys.argv
filtered_only = "--filtered" in sys.argv
FILTERED = Path(os.environ.get("FILTERED", Path(os.environ["ROOT"]) / "genomes_filtered"))

with open(DOWNSTREAM / "asap_paths.tsv", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

PROBLEMS = ("pseudo-genome", "contigs dropped")


def lacks_fasta(gff):
    with open(gff) as f:
        return not any(line.startswith("##FASTA") for line in f)


def reason(r):
    if not r["gff"]:
        return "ASA3P produced no annotation"
    if "pseudo-genome" in r["notes"]:
        return "ASA3P annotated the reference-ordered pseudo-genome"
    if "contigs dropped" in r["notes"]:
        return "ASA3P annotation omitted contigs"
    if lacks_fasta(r["gff"]):
        return "ASA3P GFF lacks the sequence block Roary needs"
    return "re-annotated for consistency (--all)" if annotate_all else ""


if filtered_only:
    gate_path = DOWNSTREAM / "species_check" / "analysis_set.tsv"
    if not gate_path.exists():
        raise SystemExit("ERROR: no species gate yet -- run D1.2 first.")
    with open(gate_path, newline="") as f:
        wanted = {g["short_name"]: g for g in csv.DictReader(f, delimiter="\t") if g["override"] == "use_filtered"}
    todo = []
    for r in rows:
        g = wanted.get(r["short_name"])
        if g is None:
            continue
        if not (FILTERED / f"{r['short_name']}.fna").exists():
            print(f"{r['short_name']}: use_filtered is set but {FILTERED / (r['short_name'] + '.fna')} does not exist (run D1.5; "
                  "the split may have judged the kept contigs not plausible as one genome)")
            continue
        r = dict(r, genus=g["genus_final"] or r["genus"], species=g["species_final"] or r["species"])
        todo.append((r, "contig-filtered genome (second organism removed)"))
else:
    todo = [(r, reason(r)) for r in rows]
    todo = [(r, why) for r, why in todo if why and r["genome_fna"]]   # need standardized contigs as input

out_dir = DOWNSTREAM / "prokka_fallback"
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / ("todo_filtered.tsv" if filtered_only else "todo.tsv"), "w") as f:
    f.write("short_name\tgenus\tspecies\treason\n")
    for r, why in todo:
        f.write(f"{r['short_name']}\t{r['genus']}\t{r['species']}\t{why}\n")

print(f"{len(todo)} isolate(s) to annotate outside ASA3P{' (all)' if annotate_all else ''}{' (contig-filtered genomes)' if filtered_only else ''}:")
for r, why in todo:
    print(f"  {r['short_name']:9s} {why}")
print(f"ARRAY=1-{len(todo)}" if todo else "ARRAY=")
END_OF_SCRIPT
chmod +x "$SCRIPTS/make_prokka_todo.py.new" && mv -f "$SCRIPTS/make_prokka_todo.py.new" "$SCRIPTS/make_prokka_todo.py"
echo "wrote $SCRIPTS/make_prokka_todo.py"

cat > "$SCRIPTS/prokka_fallback_array.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# REPLACEMENT step (per isolate): Prokka outside ASA3P, one isolate per SLURM array task.
# Only for isolates listed in todo.tsv (ASA3P annotation missing or unusable; reason in column 4).
# A failure in one task no longer stops the remaining isolates.
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

TODO="${TODO_FILE:-$DOWNSTREAM/prokka_fallback/todo.tsv}"
line=$(awk -v n="$SLURM_ARRAY_TASK_ID" 'NR == n + 1' "$TODO")   # +1 skips header
if [ -z "$line" ]; then
    echo "No line for array task $SLURM_ARRAY_TASK_ID in $TODO"; exit 1
fi
IFS=$'\t' read -r short genus species reason <<< "$line"

input="$GENOMES/${short}.fna"
filtered=no
case "$reason" in contig-filtered*) input="${FILTERED:-$ROOT/genomes_filtered}/${short}.fna"; filtered=yes ;; esac
outdir="$DOWNSTREAM/prokka_fallback/${short}"
case "$genus" in
    Pectobacterium) tagcode="PCT" ;;
    Dickeya)        tagcode="DCK" ;;
    *)              tagcode="BAC" ;;
esac
locustag="${tagcode}$(echo "$short" | tr -cd '[:alnum:]')"

echo "Prokka fallback: $short ($genus $species), locus tag $locustag -- reason: $reason"
test -s "$input" || { echo "Missing input $input"; exit 1; }

# keep the earlier annotation of the full (mixed) assembly instead of overwriting it
if [ "$filtered" = yes ] && [ -d "$outdir" ] && [ ! -e "$outdir/INPUT_FILTERED" ] && [ ! -d "${outdir}.full_assembly" ]; then
    mv "$outdir" "${outdir}.full_assembly"
fi
rm -f "$outdir/INPUT_FILTERED"

apptainer exec \
    --bind "$ASAPDIR":/asap:ro \
    --bind "$ROOT":"$ROOT" \
    "$PROKKA_IMG" \
    bash -c 'export LC_ALL=C LANG=C
             export PATH=/asap/share/prokka-1.12/binaries/linux:/asap/share/prokka-1.12/bin:/asap/share/barrnap-0.8/bin:$PATH
             exec /asap/share/prokka/bin/prokka "$@"' _ \
        --kingdom Bacteria \
        --genus "$genus" \
        --species "$species" \
        --strain "$short" \
        --prefix "$short" \
        --locustag "$locustag" \
        --centre "$CENTRE" \
        --cpus "${SLURM_CPUS_PER_TASK:-8}" \
        --outdir "$outdir" \
        --proteins /asap/db/sequences/asap-proteins.faa \
        --force \
        --addgenes \
        --rfam \
        --rawproduct \
        --compliant \
        "$input" < /dev/null

[ "$filtered" = yes ] && echo "annotated from the contig-filtered genome $input" > "$outdir/INPUT_FILTERED"
ls -lh "$outdir/${short}.gff" "$outdir/${short}.gbk"
END_OF_SCRIPT
chmod +x "$SCRIPTS/prokka_fallback_array.sh.new" && mv -f "$SCRIPTS/prokka_fallback_array.sh.new" "$SCRIPTS/prokka_fallback_array.sh"
echo "wrote $SCRIPTS/prokka_fallback_array.sh"

ANNOTATE_ALL=no          # yes = re-annotate every isolate from raw contigs

use_env download_env
if [ "$ANNOTATE_ALL" = yes ]; then out=$(python "$SCRIPTS/make_prokka_todo.py" --all)
else                             out=$(python "$SCRIPTS/make_prokka_todo.py"); fi
echo "$out"
ARRAY=$(echo "$out" | sed -n 's/^ARRAY=//p')
if [ -n "$ARRAY" ]; then
    JOB=$(SBATCH_EXTRA="--array=$ARRAY" submit prokka_fallback 8 32G 08:00:00 "$SCRIPTS/prokka_fallback_array.sh")
    echo "submitted array job $JOB ($ARRAY)"
else
    echo "No replacement needed: every isolate has a usable ASA3P annotation."
fi

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like prokka_fallback
find "$DOWNSTREAM/prokka_fallback" -name "*.gff" | sort
grep -l -i "error" "$LOGS"/prokka_fallback_*.err 2>/dev/null || echo "no errors in fallback logs"

## C2–C3. Resistance genes and virulence factors (only for genera where ASA³P's `abr/` or `vf/` is empty or failed)

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_amr_vf.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# REPLACEMENT step: runs only for genera whose ASA3P abr/ or vf/ step was empty or reported as failed.
#   run_amr_vf.sh abr <Genus> [<Genus> ...]   AMRFinderPlus + ABRicate (CARD)
#   run_amr_vf.sh vf  <Genus> [<Genus> ...]   ABRicate (VFDB)
# ASA3P's own resistance step runs RGI in a container inside the container,
# which often fails under Apptainer; that is the usual reason this is needed.
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

step="${1:?abr or vf}"; shift
[ "$#" -gt 0 ] || { echo "no genera given"; exit 1; }
OUT="$DOWNSTREAM/replacement_${step}"
mkdir -p "$OUT"
CPUS="${SLURM_CPUS_PER_TASK:-8}"

for genus in "$@"; do
    mapfile -t isolates < <(awk -F'\t' -v g="$genus" 'NR > 1 && $2 == g {print $1}' "$DOWNSTREAM/asap_paths.tsv")
    echo "== $step replacement for $genus: ${#isolates[@]} isolates =="
    for s in "${isolates[@]}"; do
        fna="$GENOMES/${s}.fna"
        if [ "$step" = abr ]; then
            apptainer exec --bind "$ROOT":"$ROOT" "$AMRFINDER_IMG" \
                amrfinder -n "$fna" --threads "$CPUS" --name "$s" -o "$OUT/${s}_amrfinder.tsv" \
                || echo "WARNING: amrfinder failed for $s"
            apptainer exec --bind "$ROOT":"$ROOT" "$ABRICATE_IMG" \
                abricate --db card --threads "$CPUS" "$fna" > "$OUT/${s}_card.tsv" \
                || echo "WARNING: abricate card failed for $s"
        else
            apptainer exec --bind "$ROOT":"$ROOT" "$ABRICATE_IMG" \
                abricate --db vfdb --threads "$CPUS" "$fna" > "$OUT/${s}_vfdb.tsv" \
                || echo "WARNING: abricate vfdb failed for $s"
        fi
    done
done

# combined tables
if [ "$step" = abr ]; then
    first=$(ls "$OUT"/*_amrfinder.tsv 2>/dev/null | head -1)
    if [ -n "$first" ]; then
        head -n 1 "$first" > "$OUT/amrfinder_all.tsv"
        for f in "$OUT"/*_amrfinder.tsv; do tail -n +2 "$f"; done >> "$OUT/amrfinder_all.tsv"
    fi
    apptainer exec --bind "$ROOT":"$ROOT" "$ABRICATE_IMG" abricate --summary "$OUT"/*_card.tsv > "$OUT/abricate_card_summary.tsv"
else
    apptainer exec --bind "$ROOT":"$ROOT" "$ABRICATE_IMG" abricate --summary "$OUT"/*_vfdb.tsv > "$OUT/abricate_vfdb_summary.tsv"
fi
echo "$step replacement results in $OUT"
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_amr_vf.sh.new" && mv -f "$SCRIPTS/run_amr_vf.sh.new" "$SCRIPTS/run_amr_vf.sh"
echo "wrote $SCRIPTS/run_amr_vf.sh"

for step in abr vf; do
    genera=$(awk -F'\t' -v s="$step" '$1 == s && $4 ~ /^replacement/ {print $2}' "$PROVENANCE/replacements.tsv" | xargs)
    if [ -n "$genera" ]; then
        JOB=$(submit "${step}_replacement" 8 32G 08:00:00 "$SCRIPTS/run_amr_vf.sh" "$step" $genera)
        echo "$step: ASA3P output empty or failed for [$genera] -> replacement job $JOB"
    else
        echo "$step: ASA3P output present for all genera -> no replacement run"
    fi
done

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like "abr_replacement|vf_replacement"
ls -lh "$DOWNSTREAM"/replacement_abr/*summary*.tsv "$DOWNSTREAM"/replacement_abr/amrfinder_all.tsv \
       "$DOWNSTREAM"/replacement_vf/*summary*.tsv 2>/dev/null || echo "no replacement outputs (none needed, or still running)"

## C4. Genus-level pan-genome (only for genera where ASA³P's `corepan/` is empty or failed)
First collects one annotation per isolate (ASA³P's, or C1's where it exists) into `downstream/annotation_files.tsv` and builds the Roary input folders for both C4 and D2. Then runs a genus-level Roary only for genera flagged in B2. GFFs are copied as `<short>.gff`, so Roary's sample names are the short names.

This step respects the species gate from D1.2 once it exists (isolates that failed are left out and groups use the ANI species). Before D1 has been run it uses every isolate; **D2.0 re-runs it after the gate**, so the order in the notebook is safe.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/prepare_roary_groups.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Collect one GFF/GBK per isolate and build Roary input folders.

GFF/GBK source per isolate: the fallback Prokka output
(DOWNSTREAM/prokka_fallback/<short>/) if it exists -- it is only created for
isolates whose ASA3P annotation was missing or unusable -- otherwise ASA3P's
own annotation (from asap_paths.tsv).
  -> DOWNSTREAM/annotation_files.tsv

Roary groups (column "role" in roary_groups.tsv):
  * additional   every species with >= 2 isolates (identity 95, Roary default).
                 ASA3P only computes one pan-genome per project (= per genus here),
                 so per-species pan-genomes are an analysis outside ASA3P.
  * replacement  a genus-level run (identity 80), ONLY for a genus whose ASA3P
                 corepan/ folder came out empty (PROVENANCE/replacements.tsv).
                 Otherwise ASA3P's own genus-level Roary result is the one to use.
A genus-level pan-genome that contains genomes of another genus has almost no core: ASA3P's own genus-level
Roary run for Pectobacterium had 22 core genes because about seven of its genomes were ~78% ANI to Pectobacterium
(a different genus). This replacement run uses only the isolates that passed the species gate. For a genus-level core-gene tree use D4b (clean isolates + Pectobacterium reference genomes);
the Mashtree tree (D3) shows genus-level relationships of all genomes.

GFFs are COPIED (not symlinked) as <short>.gff, so Roary's sample names are
the short names and nothing points outside the container's bind paths.
  -> DOWNSTREAM/roary/<group>/gff/*.gff and DOWNSTREAM/roary/roary_groups.tsv

The species gate (species_check/analysis_set.tsv, D1.2) decides which isolates
are used and under which species: an isolate with include = no is left out of
every group, and groups are formed from the gate's species (the ANI species),
not from the metadata label. Without a gate (D1 not run yet) all isolates are
used with their metadata species, and a note says so.
"""
import csv
import os
import shutil
from collections import defaultdict
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
PROV = Path(os.environ["PROVENANCE"])
FALLBACK = DOWNSTREAM / "prokka_fallback"
ROARY = DOWNSTREAM / "roary"

genus_replacement = set()
if (PROV / "replacements.tsv").exists():
    with open(PROV / "replacements.tsv", newline="") as f:
        for r in csv.DictReader(f, delimiter="\t"):
            if r["step"] == "corepan" and r["decision"].startswith("replacement"):
                genus_replacement.add(r["genus"])
else:
    print("NOTE: no replacements.tsv yet (run Part B) -- building species groups only.")

with open(DOWNSTREAM / "asap_paths.tsv", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))
if not rows:
    raise SystemExit("ERROR: asap_paths.tsv is empty -- run Sec. 7 after ASA3P has finished.")

GATE = DOWNSTREAM / "species_check" / "analysis_set.tsv"
gate = {}
if GATE.exists():
    with open(GATE, newline="") as f:
        gate = {g["short_name"]: g for g in csv.DictReader(f, delimiter="\t")}
else:
    print("NOTE: no species gate yet (D1.2): every isolate is used with its metadata species. "
          "Re-run this cell after D1.2 before D2.")


def has_fasta_section(path):
    with open(path) as f:
        return any(line.startswith("##FASTA") for line in f)


ann_rows, missing, no_fasta, left_out = [], [], [], []
for r in rows:
    s = r["short_name"]
    in_analysis, genus, species = "yes", r["genus"], r["species"]
    if gate:
        g = gate.get(s)
        if g is None:
            in_analysis = "no"
            left_out.append((s, "not in the species check: re-run D1"))
        elif g["include"] != "yes":
            in_analysis = "no"
            left_out.append((s, f"{g['qc_status']}" + (f" ({g['note']})" if g["note"] else "")))
        else:
            genus, species = g["genus_final"], g["species_final"]
    gff, gbk, source = r["gff"], r["gbk"], "asa3p"
    fb_gff, fb_gbk = FALLBACK / s / f"{s}.gff", FALLBACK / s / f"{s}.gbk"
    if fb_gff.exists():   # fallback only exists where ASA3P's annotation was missing or unusable
        gff, gbk, source = str(fb_gff), str(fb_gbk if fb_gbk.exists() else ""), "prokka_fallback"
    if not gff:
        if in_analysis == "yes":
            missing.append(s)
        source = "none"
    elif in_analysis == "yes" and not has_fasta_section(gff):
        no_fasta.append(s)   # Roary needs the ##FASTA block that Prokka appends
    ann_rows.append({"short_name": s, "genus": genus, "species": species,
                     "gff": gff, "gbk": gbk, "source": source, "in_analysis": in_analysis})

with open(DOWNSTREAM / "annotation_files.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(ann_rows[0].keys()), delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(ann_rows)

groups = defaultdict(list)
for a in ann_rows:
    if a["gff"] and a["in_analysis"] == "yes":
        groups[(f"{a['genus']}_{a['species']}", "additional", "species", 95)].append(a)
        if a["genus"] in genus_replacement:
            groups[(f"{a['genus']}_all", "replacement", "genus", 80)].append(a)

ROARY.mkdir(parents=True, exist_ok=True)
summary = []
if not any(len(m) >= 2 for m in groups.values()):
    raise SystemExit("ERROR: no group has >= 2 annotated isolates -- check annotation_files.tsv.")
for (name, role, level, ident), members in sorted(groups.items()):
    if len(members) < 2:
        continue
    gdir = ROARY / name / "gff"
    if gdir.exists():
        shutil.rmtree(gdir)
    gdir.mkdir(parents=True)
    for a in members:
        shutil.copyfile(a["gff"], gdir / f"{a['short_name']}.gff")
    summary.append({"group": name, "role": role, "level": level, "identity": ident, "n": len(members),
                    "gff_dir": str(gdir), "out_dir": str(ROARY / name / "roary_output"),
                    "members": ",".join(a["short_name"] for a in members)})

with open(ROARY / "roary_groups.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(summary[0].keys()), delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(summary)

print(f"{'group':32s} {'role':12s} {'n':>3s}  members")
for s in summary:
    print(f"{s['group']:32s} {s['role']:12s} {s['n']:>3d}  {s['members']}")
if genus_replacement:
    print("\nGenus-level replacement run(s) because ASA3P corepan/ was empty:", ", ".join(sorted(genus_replacement)))
    if "Pectobacterium" in genus_replacement:
        print("  Only isolates that passed the species gate are in this group. For a genus-level core-gene tree, with the "
              "Pectobacterium reference genomes added, use D4b.")
if gate:
    n_in = sum(a["in_analysis"] == "yes" for a in ann_rows)
    print(f"\nSpecies gate: {n_in} of {len(ann_rows)} isolates used; left out:")
    for s_, why in left_out:
        print(f"  {s_:9s} {why[:110]}")
if missing:
    print("\nNo GFF for:", ", ".join(missing), "-> run the fallback Prokka section first.")
if no_fasta:
    print("\nGFF without a ##FASTA section (Roary will reject these):", ", ".join(no_fasta),
          "\n-> annotate these with the fallback Prokka section instead.")
print(f"\nWrote {ROARY/'roary_groups.tsv'} and {DOWNSTREAM/'annotation_files.tsv'}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/prepare_roary_groups.py.new" && mv -f "$SCRIPTS/prepare_roary_groups.py.new" "$SCRIPTS/prepare_roary_groups.py"
echo "wrote $SCRIPTS/prepare_roary_groups.py"

use_env download_env
python "$SCRIPTS/prepare_roary_groups.py"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_roary_groups.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Roary (standalone container) for the groups in roary_groups.tsv with a given role:
#   run_roary_groups.sh replacement   genus-level runs for genera whose ASA3P corepan/ step was empty or failed
#   run_roary_groups.sh additional    per-species pan-genomes (outside ASA3P)
#
# Fix vs. first run: the output folder is deleted and NOT re-created before
# Roary starts. If -f points at an existing folder, Roary writes to
# <folder>_<timestamp> instead -- that is where roary_output_container_1778273395
# came from. With the folder absent, Roary writes exactly where we ask.
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

want_role="${1:?role required: replacement or additional}"
GROUPS_TSV="$DOWNSTREAM/roary/roary_groups.tsv"
CPUS="${SLURM_CPUS_PER_TASK:-8}"
failed=()

while IFS=$'\t' read -r group role level ident n gff_dir out_dir _; do
    [ "$role" = "$want_role" ] || continue
    echo "=== $group [$role] ($level, n=$n, -i $ident) ==="
    rm -rf "$out_dir"
    mapfile -t gffs < <(find "$gff_dir" -maxdepth 1 -name '*.gff' | sort)
    if apptainer exec --bind "$ROOT":"$ROOT" "$ROARY_IMG" \
           roary -p "$CPUS" -f "$out_dir" -e -n -i "$ident" "${gffs[@]}" < /dev/null; then
        echo "done: $out_dir"
        grep -E "Core genes|Total genes" "$out_dir/summary_statistics.txt" || true
    else
        echo "WARNING: Roary failed for $group"
        failed+=("$group")
    fi
done < <(tail -n +2 "$GROUPS_TSV")

if [ "${#failed[@]}" -gt 0 ]; then
    echo "Failed groups: ${failed[*]}"
    exit 1
fi
echo "All '$want_role' Roary groups finished."
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_roary_groups.sh.new" && mv -f "$SCRIPTS/run_roary_groups.sh.new" "$SCRIPTS/run_roary_groups.sh"
echo "wrote $SCRIPTS/run_roary_groups.sh"

if awk -F'\t' 'NR > 1 && $2 == "replacement" {found = 1} END {exit !found}' "$DOWNSTREAM/roary/roary_groups.tsv"; then
    JOB=$(submit roary_replacement 16 64G 2-00:00:00 "$SCRIPTS/run_roary_groups.sh" replacement)
    echo "submitted genus-level replacement Roary: $JOB"
else
    echo "ASA3P corepan/ present for all genera -> no replacement Roary run"
fi

# Part D — Additional analyses outside ASA³P
**`[Outside ASA³P]`** — an additional analysis ASA³P does not do.

## D1. Species check and analysis gate (fastANI vs. the species references)
ASA³P's own ANI (nucmer, against its RefSeq-based database) is in `taxonomy/`. This is an independent check against one reference per species, and it **decides which isolates enter D2–D6**.

A species is called at ≥ 95 % ANI with ≥ 50 % of the isolate's fragments aligned:

| Status | Meaning | Goes into Part D? |
|---|---|---|
| `PASS` | matches the species in the metadata | yes |
| `RELABELLED` | matches a different reference species than the metadata | yes, under the ANI species |
| `MIXED` | ≥ 95 % ANI to a reference over < 50 % of the assembly: the expected organism **plus** other DNA (a second genome in the culture or library) | no, unless overridden (D1.4–D1.6) |
| `NO_MATCH` | best reference hit < 95 % ANI. About 78 % is the distance between *Pectobacterium* and *Dickeya*: the isolate is none of the reference species (wrong colony, another genus, or a species not on the list) | no, unless overridden |

Species labels in `metadata_shortnames.tsv` are the initial identification and are not changed; the gate holds the ANI-based species.

### D1.1 Run fastANI
Every isolate against every species reference (outgroup excluded).

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_fastani.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# fastANI: every isolate vs. every species reference (outgroup excluded).
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

OUT="$DOWNSTREAM/species_check"
mkdir -p "$OUT"

ls "$GENOMES"/*.fna > "$OUT/query_list.txt"
awk -F'\t' -v d="$REFDIR" 'NR > 1 && $5 == "reference" {print d "/" $1 ".fasta"}' \
    "$META/reference_genomes.tsv" > "$OUT/reference_list.txt"

echo "queries:    $(wc -l < "$OUT/query_list.txt")"
echo "references: $(wc -l < "$OUT/reference_list.txt")"

apptainer exec --bind "$ROOT":"$ROOT" "$FASTANI_IMG" \
    fastANI --ql "$OUT/query_list.txt" --rl "$OUT/reference_list.txt" \
            -t "${SLURM_CPUS_PER_TASK:-8}" -o "$OUT/fastani_raw.tsv"

echo "fastANI done: $OUT/fastani_raw.tsv ($(wc -l < "$OUT/fastani_raw.tsv") pairs above fastANI's reporting floor)"
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_fastani.sh.new" && mv -f "$SCRIPTS/run_fastani.sh.new" "$SCRIPTS/run_fastani.sh"
echo "wrote $SCRIPTS/run_fastani.sh"

if [ -z "${GATE:-}" ] || [ -z "${OVERRIDES:-}" ] || [ -z "${FILTERED:-}" ]; then
    echo "config.sh is the v5.2 one: re-run A0.1 (it adds three lines), then this cell."; exit 1
fi
JOB=$(submit fastani 16 32G 04:00:00 "$SCRIPTS/run_fastani.sh"); echo "submitted $JOB"

### D1.2 Species calls and the gate
Prints one line per isolate and writes `species_check/analysis_set.tsv` (the gate: status, include yes/no, species, and the genome file to use) and `species_calls.tsv`. Re-run this cell after any change to `isolate_overrides.tsv` or after D1.6.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/summarize_fastani.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Species check and analysis gate.

Reads the fastANI table (every isolate vs. the species references), compares each
isolate with the species in metadata_shortnames.tsv, applies your decisions from
isolate_overrides.tsv, and writes two tables:

  species_check/species_calls.tsv   best and second reference hit per isolate
  species_check/analysis_set.tsv    THE GATE: which isolates go into Part D, with
                                    the genome file to use for each

Status per isolate (reference hit with ANI >= 95 % AND >= 50 % of the isolate's
fragments aligned counts as a species-level match):
  PASS        matches the species in the metadata                          -> included
  RELABELLED  matches a different reference species than the metadata      -> included, under the ANI species
  MIXED       >= 95 % ANI to a reference, but < 50 % of the genome aligns:
              the assembly holds the expected organism plus other DNA      -> excluded unless overridden
  NO_MATCH    best reference hit is < 95 % ANI (or there is none); about
              78 % ANI is the distance between Pectobacterium and Dickeya,
              i.e. the isolate is not one of the reference species          -> excluded unless overridden

isolate_overrides.tsv (META; tab-separated; lines starting with # are ignored):
  short_name <TAB> action <TAB> reason
  action = exclude        leave an isolate out even though it passed
           include        keep an isolate that failed (the metadata label is then used; say why)
           use_filtered   use the contig-filtered genome from D1.5 (MIXED isolates), once it is annotated (D1.6)
"""
import csv
import os
from pathlib import Path

META = Path(os.environ["META"])
DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
GENOMES = Path(os.environ["GENOMES"])
FILTERED = Path(os.environ.get("FILTERED", Path(os.environ["ROOT"]) / "genomes_filtered"))
OVERRIDES = Path(os.environ.get("OVERRIDES", META / "isolate_overrides.tsv"))
OUT = DOWNSTREAM / "species_check"
FALLBACK = DOWNSTREAM / "prokka_fallback"
ANI_CUT, FRAG_CUT = 95.0, 0.50
ACTIONS = {"exclude", "include", "use_filtered"}

refs = {}
with open(META / "reference_genomes.tsv", newline="") as f:
    for r in csv.DictReader(f, delimiter="\t"):
        refs[r["label"]] = f"{r['genus']} {r['species']}"

meta = {}
with open(META / "metadata_shortnames.tsv", newline="") as f:
    for r in csv.DictReader(f, delimiter="\t"):
        meta[r["short_name"]] = f"{r['genus']} {r['species']}"

overrides = {}
if OVERRIDES.exists():
    with open(OVERRIDES, newline="") as f:
        lines = [ln for ln in f if ln.strip() and not ln.lstrip().startswith("#")]
    for n, parts in enumerate(csv.reader(lines, delimiter="\t"), start=1):
        if n == 1 and parts[0].strip() == "short_name":
            continue                                   # header row
        short, action = parts[0].strip(), (parts[1].strip() if len(parts) > 1 else "")
        reason = parts[2].strip() if len(parts) > 2 else ""
        if short not in meta:
            raise SystemExit(f"ERROR in {OVERRIDES}: '{short}' is not in metadata_shortnames.tsv")
        if action not in ACTIONS:
            raise SystemExit(f"ERROR in {OVERRIDES}: action for {short} must be one of {sorted(ACTIONS)}, got '{action}'")
        overrides[short] = (action, reason)

hits = {}
with open(OUT / "fastani_raw.tsv") as f:
    for line in f:
        q, rf, ani, mapped, total = line.rstrip("\n").split("\t")[:5]
        q, rf = Path(q).name.rsplit(".", 1)[0], Path(rf).name.rsplit(".", 1)[0]
        ani, frac = float(ani), int(mapped) / max(int(total), 1)
        hits.setdefault(q, []).append((ani, frac, rf, int(total)))

call_rows, gate_rows = [], []
for short in sorted(meta):
    h = sorted(hits.get(short, []), reverse=True)
    ani, frac, ref, total = h[0] if h else (None, None, "", 0)
    second = h[1] if len(h) > 1 else (None, None, "", 0)
    expected = meta[short]
    approx_mb = total * 3000 / 1e6                        # fastANI fragments are 3 kb

    if ani is None or ani < ANI_CUT:
        status, call = "NO_MATCH", "unresolved"
        note = (f"best hit {ani:.1f}% ANI to {ref}" if ani is not None else "no reference hit")
        if ani is not None and ani < 85:
            note += " (genus level or beyond: not one of the reference species)"
    else:
        call = refs.get(ref, "unresolved")
        if frac < FRAG_CUT:
            status = "MIXED"
            note = (f"{ani:.1f}% ANI to {ref} but only {frac:.0%} of the genome aligns "
                    f"(about {approx_mb:.1f} Mb in total): a second genome is likely present")
        elif call == expected:
            status, note = "PASS", ""
        else:
            status = "RELABELLED"
            note = f"matches {call}, not the metadata label {expected}"

    call_rows.append({
        "short_name": short, "metadata_species": expected,
        "best_ref": ref, "best_ani": f"{ani:.2f}" if ani is not None else "",
        "aligned_fraction": f"{frac:.2f}" if frac is not None else "",
        "second_ref": second[2], "second_ani": f"{second[0]:.2f}" if second[0] is not None else "",
        "ani_species_call": call,
        "status": {"PASS": "agree", "RELABELLED": "DISAGREE"}.get(status, "unresolved"),
    })

    include = "yes" if status in {"PASS", "RELABELLED"} else "no"
    final = call if status in {"PASS", "RELABELLED", "MIXED"} and call != "unresolved" else "unresolved"
    genome = GENOMES / f"{short}.fna"
    action, why = overrides.get(short, ("", ""))
    if action == "exclude":
        include = "no"
        note = (note + "; " if note else "") + f"excluded by you: {why}"
    elif action == "include":
        include = "yes"
        if final == "unresolved":
            final = expected                              # forced: the metadata label stands in
        note = (note + "; " if note else "") + f"kept by you although {status}: {why}"
    elif action == "use_filtered":
        filt = FILTERED / f"{short}.fna"
        annotated = (FALLBACK / short / "INPUT_FILTERED").exists()
        if filt.exists() and annotated:
            include, genome = "yes", filt
            note = (note + "; " if note else "") + f"using the contig-filtered genome (D1.5/D1.6): {why}"
        else:
            include = "no"
            todo = "run D1.5" if not filt.exists() else "annotate it in D1.6"
            note = (note + "; " if note else "") + f"use_filtered requested but the filtered genome is not ready ({todo})"
    genus_final, _, species_final = final.partition(" ")
    gate_rows.append({
        "short_name": short, "genus_meta": expected.split(" ")[0], "species_meta": expected.split(" ", 1)[1],
        "best_ref": ref, "best_ani": f"{ani:.2f}" if ani is not None else "",
        "aligned_fraction": f"{frac:.2f}" if frac is not None else "", "approx_mb": f"{approx_mb:.1f}",
        "genus_final": genus_final if final != "unresolved" else "", "species_final": species_final,
        "qc_status": status, "override": action, "include": include, "genome_path": str(genome), "note": note,
    })

for name, rows in (("species_calls.tsv", call_rows), ("analysis_set.tsv", gate_rows)):
    with open(OUT / name, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()), delimiter="\t", lineterminator="\n")
        w.writeheader()
        w.writerows(rows)

print(f"{'isolate':9s} {'ANI':>6s} {'frac':>5s} {'Mb':>5s}  {'status':10s} {'in?':3s}  {'metadata':28s} note")
for g in gate_rows:
    print(f"{g['short_name']:9s} {g['best_ani']:>6s} {g['aligned_fraction']:>5s} {g['approx_mb']:>5s}  "
          f"{g['qc_status']:10s} {g['include']:3s}  {g['genus_meta'][0]}. {g['species_meta']:25s} {g['note']}")
counts = {s: sum(g["qc_status"] == s for g in gate_rows) for s in ("PASS", "RELABELLED", "MIXED", "NO_MATCH")}
n_in = sum(g["include"] == "yes" for g in gate_rows)
print(f"\n{len(gate_rows)} isolates: " + ", ".join(f"{v} {k}" for k, v in counts.items()) +
      f".  {n_in} go into Part D, {len(gate_rows) - n_in} do not.")
if overrides:
    print("Your overrides: " + ", ".join(f"{k}={v[0]}" for k, v in sorted(overrides.items())))
print(f"Wrote {OUT/'analysis_set.tsv'} (the gate) and {OUT/'species_calls.tsv'}")
if counts["MIXED"] or counts["NO_MATCH"]:
    print("MIXED / NO_MATCH isolates are left out until you decide (D1.3-D1.6). Likely causes: a second organism in the "
          "culture or library (MIXED), a wrong colony or a species not on the reference list (NO_MATCH).")
END_OF_SCRIPT
chmod +x "$SCRIPTS/summarize_fastani.py.new" && mv -f "$SCRIPTS/summarize_fastani.py.new" "$SCRIPTS/summarize_fastani.py"
echo "wrote $SCRIPTS/summarize_fastani.py"

jobs_like fastani
use_env download_env
python "$SCRIPTS/summarize_fastani.py"

### D1.3 What are the isolates that did not pass? (optional, a few minutes)
* **All-vs-all fastANI** groups the isolates by ANI among themselves: do the `NO_MATCH` isolates form one organism or several, and do any of them match a `PASS` group?
* **16S rRNA gene** of every isolate that did not pass, from the Prokka annotation (needs C1). Paste the FASTA into NCBI nucleotide BLAST (16S ribosomal RNA sequences, Bacteria and Archaea) and note the top hit per isolate; this names the genus.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_fastani_allvsall.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# fastANI of every isolate against every isolate: do the isolates that match no reference
# group with each other (one organism, or several)? Needs query_list.txt from D1.1.
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

OUT="$DOWNSTREAM/species_check"
test -s "$OUT/query_list.txt" || { echo "run D1.1 first (query_list.txt missing)"; exit 1; }
apptainer exec --bind "$ROOT":"$ROOT" "$FASTANI_IMG" \
    fastANI --ql "$OUT/query_list.txt" --rl "$OUT/query_list.txt" \
            -t "${SLURM_CPUS_PER_TASK:-8}" -o "$OUT/fastani_allvsall.tsv"
echo "fastANI all-vs-all done: $OUT/fastani_allvsall.tsv ($(wc -l < "$OUT/fastani_allvsall.tsv") pairs)"
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_fastani_allvsall.sh.new" && mv -f "$SCRIPTS/run_fastani_allvsall.sh.new" "$SCRIPTS/run_fastani_allvsall.sh"
echo "wrote $SCRIPTS/run_fastani_allvsall.sh"

JOB=$(submit fastani_allvsall 16 32G 01:00:00 "$SCRIPTS/run_fastani_allvsall.sh"); echo "submitted $JOB"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/summarize_allvsall.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Group isolates by ANI among themselves (all-vs-all fastANI).

Two isolates are in the same group if ANI >= 95 % with >= 50 % of the smaller side's
fragments aligned in at least one direction. Each group is shown with its species-gate
status, so you can see, for example, whether the isolates that matched no reference are
one organism or several.
  -> DOWNSTREAM/species_check/allvsall_groups.tsv
"""
import csv
import os
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
OUT = DOWNSTREAM / "species_check"
ANI_CUT, FRAG_CUT = 95.0, 0.50

gate = {}
with open(OUT / "analysis_set.tsv", newline="") as f:
    for g in csv.DictReader(f, delimiter="\t"):
        gate[g["short_name"]] = g

link, parent = {}, {s: s for s in gate}


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


best = {}
with open(OUT / "fastani_allvsall.tsv") as f:
    for line in f:
        q, r, ani, m, t = line.rstrip("\n").split("\t")[:5]
        q, r = Path(q).name.rsplit(".", 1)[0], Path(r).name.rsplit(".", 1)[0]
        if q == r or q not in gate or r not in gate:
            continue
        key = tuple(sorted((q, r)))
        a, fr = float(ani), int(m) / max(int(t), 1)
        old = best.get(key, (0.0, 0.0))
        best[key] = (max(old[0], a), max(old[1], fr))      # best fraction over the two directions
for (q, r), (a, fr) in best.items():
    if a >= ANI_CUT and fr >= FRAG_CUT:
        parent[find(q)] = find(r)

groups = {}
for s in gate:
    groups.setdefault(find(s), []).append(s)
rows = []
print(f"{len(gate)} isolates fall into {len(groups)} group(s) at >= {ANI_CUT:.0f}% ANI:\n")
for n, members in enumerate(sorted(groups.values(), key=lambda m: (-len(m), m[0])), start=1):
    members.sort()
    status = sorted({gate[m]["qc_status"] for m in members})
    species = sorted({f"{gate[m]['genus_final']} {gate[m]['species_final']}".strip() or "-" for m in members})
    print(f"group {n:2d} (n = {len(members):2d})  status: {'/'.join(status):18s} species: {', '.join(species)}")
    print(f"           {', '.join(members)}")
    for m in members:
        rows.append({"group": n, "short_name": m, "qc_status": gate[m]["qc_status"],
                     "species_final": gate[m]["species_final"], "metadata": f"{gate[m]['genus_meta']} {gate[m]['species_meta']}"})
with open(OUT / "allvsall_groups.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()), delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)
print(f"\nWrote {OUT/'allvsall_groups.tsv'}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/summarize_allvsall.py.new" && mv -f "$SCRIPTS/summarize_allvsall.py.new" "$SCRIPTS/summarize_allvsall.py"
echo "wrote $SCRIPTS/summarize_allvsall.py"

jobs_like fastani_allvsall
use_env download_env
python "$SCRIPTS/summarize_allvsall.py"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/extract_16S.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# First 16S rRNA gene (from the Prokka/Barrnap annotation) of every isolate that did not PASS the species
# check, for NCBI BLAST (nucleotide BLAST against "16S ribosomal RNA sequences (Bacteria and Archaea)").
# Uses the fallback Prokka annotation (prokka_fallback/<short>/<short>.ffn).
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

OUT="$DOWNSTREAM/species_check/16S_not_passed.fasta"
GATE="${GATE:-$DOWNSTREAM/species_check/analysis_set.tsv}"
test -s "$GATE" || { echo "run D1.2 first"; exit 1; }
: > "$OUT"
n=0
for s in $(awk -F'\t' 'NR == 1 {for (i = 1; i <= NF; i++) c[$i] = i; next} $c["qc_status"] != "PASS" {print $c["short_name"]}' "$GATE"); do
    ffn="$DOWNSTREAM/prokka_fallback/$s/$s.ffn"
    if [ ! -s "$ffn" ]; then echo "  $s: no $ffn (annotate it in C1 first)"; continue; fi
    got=$(awk -v s="$s" 'BEGIN {RS = ">"} /16S ribosomal RNA/ && !d {k = split($0, l, "\n"); printf ">%s_16S\n", s; for (i = 2; i <= k; i++) if (l[i] != "") print l[i]; d = 1}' "$ffn")
    if [ -n "$got" ]; then printf '%s\n' "$got" >> "$OUT"; n=$((n + 1)); else echo "  $s: no 16S rRNA gene in the annotation"; fi
done
echo "$n 16S sequence(s) written to $OUT"
echo "BLAST them at https://blast.ncbi.nlm.nih.gov/ (blastn, 16S ribosomal RNA sequences) and note the top hit for each."
END_OF_SCRIPT
chmod +x "$SCRIPTS/extract_16S.sh.new" && mv -f "$SCRIPTS/extract_16S.sh.new" "$SCRIPTS/extract_16S.sh"
echo "wrote $SCRIPTS/extract_16S.sh"

bash "$SCRIPTS/extract_16S.sh"

Two tables from ASA³P's own taxonomy (the `taxonomy/*.json` files hold, per genome, the closest genome in ASA³P's large database, its k-mer ANI and the fraction of the genome that aligns, and a 16S call). Neither needs a job:
* `species_check/asap_taxonomy.tsv`: ASA³P's identification next to the gate status. Look at the **conserved DNA** column, not only the ANI: an ANI over 1 % of the genome says nothing about the species.
* `species_check/supplement_isolate_identity.tsv`: the two-column table for the supplement, **initial identification** (metadata) and **genome-based identification** (fastANI species for isolates that passed; "+ second organism" for mixed assemblies; ASA³P's closest genome for the rest), with the basis of each call. Part F carries the genome-based identification into its result tables.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/asap_taxonomy_summary.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""ASA3P's own taxonomy result for every genome, next to the species gate.

ASA3P/taxonomy/<genome>.json holds, per genome:
  kmer.classification   closest genome in ASA3P's large k-mer database (any genus), with ANI and the
                        fraction of the genome that aligns ("conserved DNA")
  rrna                  16S rRNA classification (empty when no rRNA gene was assembled)
  ani.all               nucmer ANI against the project's own references; the conserved DNA is
                        listed with it and is the number to look at: an "ANI" over 0.5 % of the
                        genome says nothing about the species
  -> DOWNSTREAM/species_check/asap_taxonomy.tsv
"""
import csv
import glob
import json
import os
from pathlib import Path

ROOT = Path(os.environ["ROOT"])
DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
OUT = DOWNSTREAM / "species_check"

gate = {}
gpath = OUT / "analysis_set.tsv"
if gpath.exists():
    with open(gpath, newline="") as f:
        gate = {g["short_name"]: g for g in csv.DictReader(f, delimiter="\t")}

pct = lambda v: "" if v in (None, "") else f"{100 * float(v):.1f}"
rows = []
for p in sorted(glob.glob(str(ROOT / "ASA3P_*" / "taxonomy" / "*.json"))):
    try:
        d = json.load(open(p))
    except Exception as e:                       # a broken file should not hide the rest
        print(f"skipped {Path(p).name}: {e}")
        continue
    short = (d.get("genome") or {}).get("strain") or Path(p).stem.split("_")[-1]
    k = (d.get("kmer") or {}).get("classification") or {}
    refs = ((d.get("ani") or {}).get("all")) or []
    best = max(refs, key=lambda r: r.get("ani") or 0) if refs else {}
    g = gate.get(short, {})
    rows.append({
        "short_name": short,
        "gate_status": g.get("qc_status", ""),
        "metadata_label": f"{g.get('genus_meta', '')} {g.get('species_meta', '')}".strip(),
        "asap_closest_genome": k.get("classification", ""),
        "kmer_ani_pct": pct(k.get("ani")),
        "kmer_conserved_dna_pct": pct(k.get("conservedDNA")),
        "kmer_is_species": k.get("isSpecies", ""),
        "rrna_classification": (d.get("rrna") or {}).get("classification") or "",
        "best_project_ref": best.get("reference", ""),
        "ref_ani_pct": pct(best.get("ani")),
        "ref_conserved_dna_pct": pct(best.get("conservedDNA")),
    })

if not rows:
    raise SystemExit("ERROR: no taxonomy JSON found under ASA3P_*/taxonomy/")
rows.sort(key=lambda r: (r["gate_status"] == "PASS", r["short_name"]))
with open(OUT / "asap_taxonomy.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()), delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)

print(f"{'isolate':8s} {'gate':9s} {'metadata label':28s} {'ASA3P closest genome (k-mer)':48s} {'ANI%':>5s} {'cons%':>5s}  16S | best project reference (ANI% / conserved DNA%)")
for r in rows:
    print(f"{r['short_name']:8s} {r['gate_status']:9s} {r['metadata_label'][:28]:28s} {r['asap_closest_genome'][:48]:48s} "
          f"{r['kmer_ani_pct']:>5s} {r['kmer_conserved_dna_pct']:>5s}  {r['rrna_classification'] or '-'} | "
          f"{r['best_project_ref']} ({r['ref_ani_pct']} / {r['ref_conserved_dna_pct']})")
print(f"\nWrote {OUT/'asap_taxonomy.tsv'}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/asap_taxonomy_summary.py.new" && mv -f "$SCRIPTS/asap_taxonomy_summary.py.new" "$SCRIPTS/asap_taxonomy_summary.py"
echo "wrote $SCRIPTS/asap_taxonomy_summary.py"

use_env download_env
python "$SCRIPTS/asap_taxonomy_summary.py"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/make_identity_table.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Two-column identification table for the supplement (and for Part F).

Initial identification (from the metadata) next to the genome-based identification:
  PASS / RELABELLED  fastANI species (>=95% ANI over >=50% of the assembly)
  MIXED              that species + "second organism (mixed assembly)"
  NO_MATCH           ASA3P k-mer classification (closest database genome), else "mixed culture, no single-genome match"
Needs D1.2 (analysis_set.tsv) and, for NO_MATCH isolates, asap_taxonomy.tsv (D1.3).
  -> DOWNSTREAM/species_check/supplement_isolate_identity.tsv
"""
import csv
import os
from pathlib import Path

OUT = Path(os.environ["DOWNSTREAM"]) / "species_check"
rd = lambda p: {r["short_name"]: r for r in csv.DictReader(open(p, newline=""), delimiter="\t")}
gate = rd(OUT / "analysis_set.tsv")
tax = rd(OUT / "asap_taxonomy.tsv") if (OUT / "asap_taxonomy.tsv").exists() else {}
if not tax:
    print("NOTE: asap_taxonomy.tsv not found (run the ASA3P taxonomy summary in D1.3 first): NO_MATCH isolates will read 'mixed culture, no single-genome match'.")

rows = []
for s in sorted(gate):
    g, t = gate[s], tax.get(s, {})
    status = g["qc_status"]
    initial = f"{g['genus_meta']} {g['species_meta']}"
    if status in ("PASS", "RELABELLED"):
        final = f"{g['genus_final']} {g['species_final']}"
        ani, aln = g["best_ani"], f"{100 * float(g['aligned_fraction']):.0f}"
        basis = "fastANI to species reference (>=95% ANI, >=50% aligned)"
    elif status == "MIXED":
        final = f"{g['genus_final']} {g['species_final']} + second organism (mixed assembly)"
        ani, aln = g["best_ani"], f"{100 * float(g['aligned_fraction']):.0f}"
        basis = "fastANI to species reference; <50% of the assembly aligns"
    else:  # NO_MATCH
        final = t.get("asap_closest_genome") or "mixed culture, no single-genome match"
        ani, aln = t.get("kmer_ani_pct", ""), t.get("kmer_conserved_dna_pct", "")
        basis = "ASA3P k-mer classification (closest database genome)"
    rows.append({"isolate": s, "initial_identification": initial, "genome_based_identification": final,
                 "ani_pct": ani, "genome_aligned_pct": aln, "basis": basis,
                 "gate_status": status, "in_analysis": g["include"]})

out = OUT / "supplement_isolate_identity.tsv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]), delimiter="\t", lineterminator="\n"); w.writeheader(); w.writerows(rows)
for r in rows:
    print(f"{r['isolate']:8s} {r['initial_identification'][:28]:28s} -> {r['genome_based_identification'][:62]:62s} {r['gate_status']}")
print(f"\nWrote {out}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/make_identity_table.py.new" && mv -f "$SCRIPTS/make_identity_table.py.new" "$SCRIPTS/make_identity_table.py"
echo "wrote $SCRIPTS/make_identity_table.py"

use_env download_env
python "$SCRIPTS/make_identity_table.py"

### D1.4 Your decisions (optional)
`metadata/isolate_overrides.tsv` is yours: one line per isolate, `short_name`, `action`, `reason`, separated by tabs. Lines starting with `#` are ignored. Actions:

* `exclude`: leave out an isolate that passed (for example, one you are re-sequencing).
* `include`: keep an isolate that did not pass. The metadata label is then used; give the reason, because the Methods draft will ask you to state it.
* `use_filtered`: for a `MIXED` isolate, use the contig-filtered genome from D1.5 once it is annotated (D1.6).

The cell creates the file if missing and shows it. To add a decision, uncomment and edit one of the `printf` lines (they write the tabs for you), run the cell, then re-run D1.2.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

if [ ! -s "$OVERRIDES" ]; then
    printf '%s\n' "# Your decisions on top of the species check (D1.2). Columns: short_name, action, reason (tab-separated)." \
        "# action = exclude | include | use_filtered. Lines starting with # are ignored." > "$OVERRIDES"
    echo "created $OVERRIDES"
fi
# --- add a decision by uncommenting and editing one line: ---
# printf '%s\t%s\t%s\n' PI012 use_filtered "second organism removed by contig split (D1.5); replace after re-sequencing" >> "$OVERRIDES"
# printf '%s\t%s\t%s\n' PI027 exclude "re-sequencing pending" >> "$OVERRIDES"
echo "--- $OVERRIDES ---"; cat "$OVERRIDES"

### D1.5 Separate the expected organism from a second genome in a `MIXED` assembly (provisional)
For each `MIXED` isolate: every contig ≥ 1 kb is compared with the species reference by fastANI, and contigs with ≥ 90 % ANI are kept as the expected organism. The summary shows kept vs. removed size, GC and coverage, and whether the kept set is as long as one genome of that species. Only if you set `use_filtered` for the isolate (D1.4) and it is annotated (D1.6) does the filtered genome enter Part D.

**Limits:** contigs made only of genes the reference lacks (islands, plasmids) are removed together with the second organism, and contigs < 1 kb cannot be classified. Treat the result as a stop-gap until the isolate is re-sequenced from a single colony.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/contig_split.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Separate the expected organism from a second genome in a MIXED assembly (provisional).

Used for isolates the species check marked MIXED (>= 95 % ANI to a reference, but
well under half of the assembly aligns), and for any isolate you set to
use_filtered in isolate_overrides.tsv. Two steps, with a fastANI run in between
(run_contig_split.sh does that part on SLURM):

  prepare   write every contig >= 1 kb to its own file + the list fastANI needs
            -> DOWNSTREAM/contig_split/<short>/{contigs/,contigs.txt,ref.txt,contigs.tsv}
            -> DOWNSTREAM/contig_split/targets.tsv
  assign    read fastANI's contig-vs-reference table and keep the contigs with
            >= 90 % ANI to the species reference
            -> DOWNSTREAM/contig_split/<short>/contig_assignment.tsv
            -> GENOMES_FILTERED/<short>.fna   (kept contigs, original headers)
            -> DOWNSTREAM/contig_split/split_summary.tsv

Limits to keep in mind: fastANI only sees the parts of a contig that resemble the
reference, so contigs made only of genes the reference lacks (islands, plasmids)
are left out together with the second organism, and contigs < 1 kb cannot be
classified. The summary prints GC and coverage of the kept and removed contigs,
and the size of the kept set against the reference, so you can judge whether the
split is clean. This is a stop-gap until the isolate is re-sequenced from a
single colony.
"""
import csv
import os
import re
import shutil
import statistics
import sys
from pathlib import Path

GENOMES = Path(os.environ["GENOMES"])
DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
REFDIR = Path(os.environ["REFDIR"])
FILTERED = Path(os.environ.get("FILTERED", Path(os.environ["ROOT"]) / "genomes_filtered"))
GATE = DOWNSTREAM / "species_check" / "analysis_set.tsv"
SPLIT = DOWNSTREAM / "contig_split"
MIN_LEN = 1000          # also the fastANI fragment length used in run_contig_split.sh
ASSIGN_ANI = 90.0


def read_fasta(path):
    name, seq = None, []
    with open(path) as f:
        for line in f:
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(seq)
                name, seq = line[1:].strip().split()[0], []
            else:
                seq.append(line.strip())
    if name is not None:
        yield name, "".join(seq)


def gc(seq):
    s = seq.upper()
    acgt = sum(s.count(b) for b in "ACGT")
    return 100.0 * (s.count("G") + s.count("C")) / acgt if acgt else 0.0


def original_cov():
    """contig id -> k-mer coverage from the SPAdes header kept in <short>.header_map.tsv"""
    def load(short):
        p = GENOMES / f"{short}.header_map.tsv"
        out = {}
        if p.exists():
            with open(p, newline="") as f:
                for r in csv.DictReader(f, delimiter="\t"):
                    m = re.search(r"cov_([0-9.]+)", r["original_header"])
                    out[r["new_header"]] = float(m.group(1)) if m else None
        return out
    return load


def targets():
    if not GATE.exists():
        sys.exit("ERROR: no species gate yet -- run D1.2 first.")
    with open(GATE, newline="") as f:
        rows = list(csv.DictReader(f, delimiter="\t"))
    return [r for r in rows if r["qc_status"] == "MIXED" or r["override"] == "use_filtered"]


def prepare():
    tg = targets()
    SPLIT.mkdir(parents=True, exist_ok=True)
    cov_of = original_cov()
    lines = []
    for r in tg:
        short, ref_label = r["short_name"], r["best_ref"]
        ref = REFDIR / f"{ref_label}.fasta"
        fna = GENOMES / f"{short}.fna"
        if not ref_label or not ref.exists() or not fna.exists():
            print(f"{short}: skipped (no reference hit or missing file: {ref_label or '-'}, {fna.name})")
            continue
        d = SPLIT / short
        if d.exists():
            shutil.rmtree(d)
        (d / "contigs").mkdir(parents=True)
        cov = cov_of(short)
        n_long = 0
        with open(d / "contigs.tsv", "w") as ct, open(d / "contigs.txt", "w") as lst:
            ct.write("contig\tlength\tgc\tkmer_cov\n")
            for name, seq in read_fasta(fna):
                c = cov.get(name)
                ct.write(f"{name}\t{len(seq)}\t{gc(seq):.2f}\t{'' if c is None else c}\n")
                if len(seq) >= MIN_LEN:
                    p = d / "contigs" / f"{name}.fa"
                    p.write_text(f">{name}\n{seq}\n")
                    lst.write(f"{p}\n")
                    n_long += 1
        (d / "ref.txt").write_text(f"{ref}\n")
        lines.append(f"{short}\t{ref}")
        print(f"{short}: {n_long} contigs >= {MIN_LEN} bp written; reference {ref_label} ({r['species_final'] or r['species_meta']})")
    (SPLIT / "targets.tsv").write_text("\n".join(lines) + ("\n" if lines else ""))
    if not lines:
        print("No MIXED isolates (and none set to use_filtered): nothing to split.")


def median(vals):
    vals = [v for v in vals if v is not None]
    return statistics.median(vals) if vals else None


def assign():
    FILTERED.mkdir(parents=True, exist_ok=True)
    tpath = SPLIT / "targets.tsv"
    if not tpath.exists() or not tpath.read_text().strip():
        print("No targets: run `prepare` first (or there is nothing to split).")
        return
    summary = []
    for line in tpath.read_text().splitlines():
        short, ref = line.split("\t")
        d = SPLIT / short
        fa = d / "fastani_contigs.tsv"
        if not fa.exists():
            print(f"{short}: no fastANI output at {fa} (has the job finished?)")
            continue
        ani = {}
        with open(fa) as f:
            for row in f:
                q, _, a, m, t = row.rstrip("\n").split("\t")[:5]
                ani[Path(q).name.rsplit(".", 1)[0]] = (float(a), int(m), int(t))
        info = {}
        with open(d / "contigs.tsv", newline="") as f:
            for r in csv.DictReader(f, delimiter="\t"):
                info[r["contig"]] = (int(r["length"]), float(r["gc"]), float(r["kmer_cov"]) if r["kmer_cov"] else None)
        seqs = dict(read_fasta(GENOMES / f"{short}.fna"))
        ref_bp = sum(len(s) for _, s in read_fasta(ref))
        keep, other = [], []
        with open(d / "contig_assignment.tsv", "w") as out:
            out.write("contig\tlength\tgc\tkmer_cov\tani_to_reference\tfragments_mapped\tfragments_total\tassignment\n")
            for name, (ln, g, c) in info.items():
                if ln < MIN_LEN:
                    cls, a = "too_short", None
                elif name in ani and ani[name][0] >= ASSIGN_ANI:
                    cls, a = "expected_organism", ani[name]
                else:
                    cls, a = "not_matching", ani.get(name)
                (keep if cls == "expected_organism" else other).append((name, ln, g, c))
                out.write(f"{name}\t{ln}\t{g:.2f}\t{'' if c is None else c}\t"
                          f"{'' if a is None else a[0]}\t{'' if a is None else a[1]}\t{'' if a is None else a[2]}\t{cls}\n")
        kept_bp, other_bp = sum(k[1] for k in keep), sum(o[1] for o in other)
        wgc = lambda xs: (sum(x[1] * x[2] for x in xs) / sum(x[1] for x in xs)) if xs else None
        ratio = kept_bp / ref_bp if ref_bp else 0
        plausible = bool(keep) and 0.85 <= ratio <= 1.30
        if keep:
            with open(FILTERED / f"{short}.fna", "w") as out:
                for name, *_ in keep:
                    s = seqs[name]
                    out.write(f">{name}\n")
                    for k in range(0, len(s), 80):
                        out.write(s[k:k + 80] + "\n")
        fmt = lambda v, p=1: "-" if v is None else f"{v:.{p}f}"
        print(f"\n== {short} vs {Path(ref).stem} ==")
        print(f"  kept     {len(keep):4d} contigs  {kept_bp/1e6:6.2f} Mb  GC {fmt(wgc(keep))}%  median k-mer cov {fmt(median([k[3] for k in keep]))}")
        print(f"  removed  {len(other):4d} contigs  {other_bp/1e6:6.2f} Mb  GC {fmt(wgc(other))}%  median k-mer cov {fmt(median([o[3] for o in other]))}")
        print(f"  kept set = {ratio:.0%} of the reference length ({ref_bp/1e6:.2f} Mb): "
              + ("plausible for one genome" if plausible else "NOT plausible for one genome -- do not use the filtered file"))
        if plausible:
            print(f"  wrote {FILTERED / (short + '.fna')}")
        elif (FILTERED / f"{short}.fna").exists():
            (FILTERED / f"{short}.fna").unlink()
        summary.append({"short_name": short, "reference": Path(ref).stem, "contigs_total": len(info),
                        "contigs_kept": len(keep), "kept_mb": f"{kept_bp/1e6:.2f}", "removed_mb": f"{other_bp/1e6:.2f}",
                        "reference_mb": f"{ref_bp/1e6:.2f}", "kept_over_reference": f"{ratio:.2f}",
                        "gc_kept": fmt(wgc(keep)), "gc_removed": fmt(wgc(other)),
                        "plausible_single_genome": "yes" if plausible else "no"})
    if summary:
        with open(SPLIT / "split_summary.tsv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(summary[0].keys()), delimiter="\t", lineterminator="\n")
            w.writeheader()
            w.writerows(summary)
        print(f"\nWrote {SPLIT/'split_summary.tsv'}")


if __name__ == "__main__":
    cmd = sys.argv[1] if len(sys.argv) > 1 else ""
    {"prepare": prepare, "assign": assign}.get(cmd, lambda: sys.exit("usage: contig_split.py prepare|assign"))()
END_OF_SCRIPT
chmod +x "$SCRIPTS/contig_split.py.new" && mv -f "$SCRIPTS/contig_split.py.new" "$SCRIPTS/contig_split.py"
echo "wrote $SCRIPTS/contig_split.py"

use_env download_env
python "$SCRIPTS/contig_split.py" prepare

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_contig_split.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# fastANI of every contig (>= 1 kb) of each MIXED isolate against its species reference.
# `contig_split.py prepare` writes the lists first; `contig_split.py assign` reads the result.
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

TARGETS="$DOWNSTREAM/contig_split/targets.tsv"
[ -s "$TARGETS" ] || { echo "no targets"; exit 0; }
while IFS=$'\t' read -r short ref; do
    d="$DOWNSTREAM/contig_split/$short"
    echo "== $short vs $(basename "$ref") =="
    apptainer exec --bind "$ROOT":"$ROOT" "$FASTANI_IMG" \
        fastANI --ql "$d/contigs.txt" --rl "$d/ref.txt" --fragLen 1000 \
                -t "${SLURM_CPUS_PER_TASK:-8}" -o "$d/fastani_contigs.tsv" \
        || echo "WARNING: fastANI returned an error for $short"
    [ -f "$d/fastani_contigs.tsv" ] || : > "$d/fastani_contigs.tsv"
    echo "  $(wc -l < "$d/fastani_contigs.tsv") contigs matched the reference"
done < "$TARGETS"
echo "contig split: fastANI done"
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_contig_split.sh.new" && mv -f "$SCRIPTS/run_contig_split.sh.new" "$SCRIPTS/run_contig_split.sh"
echo "wrote $SCRIPTS/run_contig_split.sh"

if [ -s "$DOWNSTREAM/contig_split/targets.tsv" ]; then
    JOB=$(submit contig_split 8 16G 02:00:00 "$SCRIPTS/run_contig_split.sh"); echo "submitted $JOB"
else
    echo "Nothing to split."
fi

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/contig_split.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Separate the expected organism from a second genome in a MIXED assembly (provisional).

Used for isolates the species check marked MIXED (>= 95 % ANI to a reference, but
well under half of the assembly aligns), and for any isolate you set to
use_filtered in isolate_overrides.tsv. Two steps, with a fastANI run in between
(run_contig_split.sh does that part on SLURM):

  prepare   write every contig >= 1 kb to its own file + the list fastANI needs
            -> DOWNSTREAM/contig_split/<short>/{contigs/,contigs.txt,ref.txt,contigs.tsv}
            -> DOWNSTREAM/contig_split/targets.tsv
  assign    read fastANI's contig-vs-reference table and keep the contigs with
            >= 90 % ANI to the species reference
            -> DOWNSTREAM/contig_split/<short>/contig_assignment.tsv
            -> GENOMES_FILTERED/<short>.fna   (kept contigs, original headers)
            -> DOWNSTREAM/contig_split/split_summary.tsv

Limits to keep in mind: fastANI only sees the parts of a contig that resemble the
reference, so contigs made only of genes the reference lacks (islands, plasmids)
are left out together with the second organism, and contigs < 1 kb cannot be
classified. The summary prints GC and coverage of the kept and removed contigs,
and the size of the kept set against the reference, so you can judge whether the
split is clean. This is a stop-gap until the isolate is re-sequenced from a
single colony.
"""
import csv
import os
import re
import shutil
import statistics
import sys
from pathlib import Path

GENOMES = Path(os.environ["GENOMES"])
DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
REFDIR = Path(os.environ["REFDIR"])
FILTERED = Path(os.environ.get("FILTERED", Path(os.environ["ROOT"]) / "genomes_filtered"))
GATE = DOWNSTREAM / "species_check" / "analysis_set.tsv"
SPLIT = DOWNSTREAM / "contig_split"
MIN_LEN = 1000          # also the fastANI fragment length used in run_contig_split.sh
ASSIGN_ANI = 90.0


def read_fasta(path):
    name, seq = None, []
    with open(path) as f:
        for line in f:
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(seq)
                name, seq = line[1:].strip().split()[0], []
            else:
                seq.append(line.strip())
    if name is not None:
        yield name, "".join(seq)


def gc(seq):
    s = seq.upper()
    acgt = sum(s.count(b) for b in "ACGT")
    return 100.0 * (s.count("G") + s.count("C")) / acgt if acgt else 0.0


def original_cov():
    """contig id -> k-mer coverage from the SPAdes header kept in <short>.header_map.tsv"""
    def load(short):
        p = GENOMES / f"{short}.header_map.tsv"
        out = {}
        if p.exists():
            with open(p, newline="") as f:
                for r in csv.DictReader(f, delimiter="\t"):
                    m = re.search(r"cov_([0-9.]+)", r["original_header"])
                    out[r["new_header"]] = float(m.group(1)) if m else None
        return out
    return load


def targets():
    if not GATE.exists():
        sys.exit("ERROR: no species gate yet -- run D1.2 first.")
    with open(GATE, newline="") as f:
        rows = list(csv.DictReader(f, delimiter="\t"))
    return [r for r in rows if r["qc_status"] == "MIXED" or r["override"] == "use_filtered"]


def prepare():
    tg = targets()
    SPLIT.mkdir(parents=True, exist_ok=True)
    cov_of = original_cov()
    lines = []
    for r in tg:
        short, ref_label = r["short_name"], r["best_ref"]
        ref = REFDIR / f"{ref_label}.fasta"
        fna = GENOMES / f"{short}.fna"
        if not ref_label or not ref.exists() or not fna.exists():
            print(f"{short}: skipped (no reference hit or missing file: {ref_label or '-'}, {fna.name})")
            continue
        d = SPLIT / short
        if d.exists():
            shutil.rmtree(d)
        (d / "contigs").mkdir(parents=True)
        cov = cov_of(short)
        n_long = 0
        with open(d / "contigs.tsv", "w") as ct, open(d / "contigs.txt", "w") as lst:
            ct.write("contig\tlength\tgc\tkmer_cov\n")
            for name, seq in read_fasta(fna):
                c = cov.get(name)
                ct.write(f"{name}\t{len(seq)}\t{gc(seq):.2f}\t{'' if c is None else c}\n")
                if len(seq) >= MIN_LEN:
                    p = d / "contigs" / f"{name}.fa"
                    p.write_text(f">{name}\n{seq}\n")
                    lst.write(f"{p}\n")
                    n_long += 1
        (d / "ref.txt").write_text(f"{ref}\n")
        lines.append(f"{short}\t{ref}")
        print(f"{short}: {n_long} contigs >= {MIN_LEN} bp written; reference {ref_label} ({r['species_final'] or r['species_meta']})")
    (SPLIT / "targets.tsv").write_text("\n".join(lines) + ("\n" if lines else ""))
    if not lines:
        print("No MIXED isolates (and none set to use_filtered): nothing to split.")


def median(vals):
    vals = [v for v in vals if v is not None]
    return statistics.median(vals) if vals else None


def assign():
    FILTERED.mkdir(parents=True, exist_ok=True)
    tpath = SPLIT / "targets.tsv"
    if not tpath.exists() or not tpath.read_text().strip():
        print("No targets: run `prepare` first (or there is nothing to split).")
        return
    summary = []
    for line in tpath.read_text().splitlines():
        short, ref = line.split("\t")
        d = SPLIT / short
        fa = d / "fastani_contigs.tsv"
        if not fa.exists():
            print(f"{short}: no fastANI output at {fa} (has the job finished?)")
            continue
        ani = {}
        with open(fa) as f:
            for row in f:
                q, _, a, m, t = row.rstrip("\n").split("\t")[:5]
                ani[Path(q).name.rsplit(".", 1)[0]] = (float(a), int(m), int(t))
        info = {}
        with open(d / "contigs.tsv", newline="") as f:
            for r in csv.DictReader(f, delimiter="\t"):
                info[r["contig"]] = (int(r["length"]), float(r["gc"]), float(r["kmer_cov"]) if r["kmer_cov"] else None)
        seqs = dict(read_fasta(GENOMES / f"{short}.fna"))
        ref_bp = sum(len(s) for _, s in read_fasta(ref))
        keep, other = [], []
        with open(d / "contig_assignment.tsv", "w") as out:
            out.write("contig\tlength\tgc\tkmer_cov\tani_to_reference\tfragments_mapped\tfragments_total\tassignment\n")
            for name, (ln, g, c) in info.items():
                if ln < MIN_LEN:
                    cls, a = "too_short", None
                elif name in ani and ani[name][0] >= ASSIGN_ANI:
                    cls, a = "expected_organism", ani[name]
                else:
                    cls, a = "not_matching", ani.get(name)
                (keep if cls == "expected_organism" else other).append((name, ln, g, c))
                out.write(f"{name}\t{ln}\t{g:.2f}\t{'' if c is None else c}\t"
                          f"{'' if a is None else a[0]}\t{'' if a is None else a[1]}\t{'' if a is None else a[2]}\t{cls}\n")
        kept_bp, other_bp = sum(k[1] for k in keep), sum(o[1] for o in other)
        wgc = lambda xs: (sum(x[1] * x[2] for x in xs) / sum(x[1] for x in xs)) if xs else None
        ratio = kept_bp / ref_bp if ref_bp else 0
        plausible = bool(keep) and 0.85 <= ratio <= 1.30
        if keep:
            with open(FILTERED / f"{short}.fna", "w") as out:
                for name, *_ in keep:
                    s = seqs[name]
                    out.write(f">{name}\n")
                    for k in range(0, len(s), 80):
                        out.write(s[k:k + 80] + "\n")
        fmt = lambda v, p=1: "-" if v is None else f"{v:.{p}f}"
        print(f"\n== {short} vs {Path(ref).stem} ==")
        print(f"  kept     {len(keep):4d} contigs  {kept_bp/1e6:6.2f} Mb  GC {fmt(wgc(keep))}%  median k-mer cov {fmt(median([k[3] for k in keep]))}")
        print(f"  removed  {len(other):4d} contigs  {other_bp/1e6:6.2f} Mb  GC {fmt(wgc(other))}%  median k-mer cov {fmt(median([o[3] for o in other]))}")
        print(f"  kept set = {ratio:.0%} of the reference length ({ref_bp/1e6:.2f} Mb): "
              + ("plausible for one genome" if plausible else "NOT plausible for one genome -- do not use the filtered file"))
        if plausible:
            print(f"  wrote {FILTERED / (short + '.fna')}")
        elif (FILTERED / f"{short}.fna").exists():
            (FILTERED / f"{short}.fna").unlink()
        summary.append({"short_name": short, "reference": Path(ref).stem, "contigs_total": len(info),
                        "contigs_kept": len(keep), "kept_mb": f"{kept_bp/1e6:.2f}", "removed_mb": f"{other_bp/1e6:.2f}",
                        "reference_mb": f"{ref_bp/1e6:.2f}", "kept_over_reference": f"{ratio:.2f}",
                        "gc_kept": fmt(wgc(keep)), "gc_removed": fmt(wgc(other)),
                        "plausible_single_genome": "yes" if plausible else "no"})
    if summary:
        with open(SPLIT / "split_summary.tsv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(summary[0].keys()), delimiter="\t", lineterminator="\n")
            w.writeheader()
            w.writerows(summary)
        print(f"\nWrote {SPLIT/'split_summary.tsv'}")


if __name__ == "__main__":
    cmd = sys.argv[1] if len(sys.argv) > 1 else ""
    {"prepare": prepare, "assign": assign}.get(cmd, lambda: sys.exit("usage: contig_split.py prepare|assign"))()
END_OF_SCRIPT
chmod +x "$SCRIPTS/contig_split.py.new" && mv -f "$SCRIPTS/contig_split.py.new" "$SCRIPTS/contig_split.py"
echo "wrote $SCRIPTS/contig_split.py"

jobs_like contig_split
use_env download_env
python "$SCRIPTS/contig_split.py" assign

### D1.6 Annotate the filtered genomes (only isolates set to `use_filtered`)
Prokka on `genomes_filtered/<short>.fna`, same settings as C1. The earlier annotation of the full assembly is kept as `prokka_fallback/<short>.full_assembly/`. When the job has finished, **re-run D1.2** so the gate lets the isolate in.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/make_prokka_todo.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""List isolates that need the fallback Prokka run.

Default: isolates where ASA3P produced no GFF, annotated the pseudo-genome,
dropped contigs (flags written by locate_asap_outputs.py), or wrote a GFF
without the ##FASTA block Roary needs.
--all: every isolate, so all 25 annotations come from the same raw-contig
       Prokka run (most consistent input for Roary).
  -> DOWNSTREAM/prokka_fallback/todo.tsv  (short_name, genus, species, reason)
--filtered: only the isolates you set to use_filtered in isolate_overrides.tsv
       whose contig-filtered genome exists (D1.5); annotated from that genome.
  -> DOWNSTREAM/prokka_fallback/todo_filtered.tsv  (todo.tsv is left alone)
Prints the --array range to pass to sbatch.
"""
import csv
import os
import sys
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
annotate_all = "--all" in sys.argv
filtered_only = "--filtered" in sys.argv
FILTERED = Path(os.environ.get("FILTERED", Path(os.environ["ROOT"]) / "genomes_filtered"))

with open(DOWNSTREAM / "asap_paths.tsv", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

PROBLEMS = ("pseudo-genome", "contigs dropped")


def lacks_fasta(gff):
    with open(gff) as f:
        return not any(line.startswith("##FASTA") for line in f)


def reason(r):
    if not r["gff"]:
        return "ASA3P produced no annotation"
    if "pseudo-genome" in r["notes"]:
        return "ASA3P annotated the reference-ordered pseudo-genome"
    if "contigs dropped" in r["notes"]:
        return "ASA3P annotation omitted contigs"
    if lacks_fasta(r["gff"]):
        return "ASA3P GFF lacks the sequence block Roary needs"
    return "re-annotated for consistency (--all)" if annotate_all else ""


if filtered_only:
    gate_path = DOWNSTREAM / "species_check" / "analysis_set.tsv"
    if not gate_path.exists():
        raise SystemExit("ERROR: no species gate yet -- run D1.2 first.")
    with open(gate_path, newline="") as f:
        wanted = {g["short_name"]: g for g in csv.DictReader(f, delimiter="\t") if g["override"] == "use_filtered"}
    todo = []
    for r in rows:
        g = wanted.get(r["short_name"])
        if g is None:
            continue
        if not (FILTERED / f"{r['short_name']}.fna").exists():
            print(f"{r['short_name']}: use_filtered is set but {FILTERED / (r['short_name'] + '.fna')} does not exist (run D1.5; "
                  "the split may have judged the kept contigs not plausible as one genome)")
            continue
        r = dict(r, genus=g["genus_final"] or r["genus"], species=g["species_final"] or r["species"])
        todo.append((r, "contig-filtered genome (second organism removed)"))
else:
    todo = [(r, reason(r)) for r in rows]
    todo = [(r, why) for r, why in todo if why and r["genome_fna"]]   # need standardized contigs as input

out_dir = DOWNSTREAM / "prokka_fallback"
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / ("todo_filtered.tsv" if filtered_only else "todo.tsv"), "w") as f:
    f.write("short_name\tgenus\tspecies\treason\n")
    for r, why in todo:
        f.write(f"{r['short_name']}\t{r['genus']}\t{r['species']}\t{why}\n")

print(f"{len(todo)} isolate(s) to annotate outside ASA3P{' (all)' if annotate_all else ''}{' (contig-filtered genomes)' if filtered_only else ''}:")
for r, why in todo:
    print(f"  {r['short_name']:9s} {why}")
print(f"ARRAY=1-{len(todo)}" if todo else "ARRAY=")
END_OF_SCRIPT
chmod +x "$SCRIPTS/make_prokka_todo.py.new" && mv -f "$SCRIPTS/make_prokka_todo.py.new" "$SCRIPTS/make_prokka_todo.py"
echo "wrote $SCRIPTS/make_prokka_todo.py"
cat > "$SCRIPTS/prokka_fallback_array.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# REPLACEMENT step (per isolate): Prokka outside ASA3P, one isolate per SLURM array task.
# Only for isolates listed in todo.tsv (ASA3P annotation missing or unusable; reason in column 4).
# A failure in one task no longer stops the remaining isolates.
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

TODO="${TODO_FILE:-$DOWNSTREAM/prokka_fallback/todo.tsv}"
line=$(awk -v n="$SLURM_ARRAY_TASK_ID" 'NR == n + 1' "$TODO")   # +1 skips header
if [ -z "$line" ]; then
    echo "No line for array task $SLURM_ARRAY_TASK_ID in $TODO"; exit 1
fi
IFS=$'\t' read -r short genus species reason <<< "$line"

input="$GENOMES/${short}.fna"
filtered=no
case "$reason" in contig-filtered*) input="${FILTERED:-$ROOT/genomes_filtered}/${short}.fna"; filtered=yes ;; esac
outdir="$DOWNSTREAM/prokka_fallback/${short}"
case "$genus" in
    Pectobacterium) tagcode="PCT" ;;
    Dickeya)        tagcode="DCK" ;;
    *)              tagcode="BAC" ;;
esac
locustag="${tagcode}$(echo "$short" | tr -cd '[:alnum:]')"

echo "Prokka fallback: $short ($genus $species), locus tag $locustag -- reason: $reason"
test -s "$input" || { echo "Missing input $input"; exit 1; }

# keep the earlier annotation of the full (mixed) assembly instead of overwriting it
if [ "$filtered" = yes ] && [ -d "$outdir" ] && [ ! -e "$outdir/INPUT_FILTERED" ] && [ ! -d "${outdir}.full_assembly" ]; then
    mv "$outdir" "${outdir}.full_assembly"
fi
rm -f "$outdir/INPUT_FILTERED"

apptainer exec \
    --bind "$ASAPDIR":/asap:ro \
    --bind "$ROOT":"$ROOT" \
    "$PROKKA_IMG" \
    bash -c 'export LC_ALL=C LANG=C
             export PATH=/asap/share/prokka-1.12/binaries/linux:/asap/share/prokka-1.12/bin:/asap/share/barrnap-0.8/bin:$PATH
             exec /asap/share/prokka/bin/prokka "$@"' _ \
        --kingdom Bacteria \
        --genus "$genus" \
        --species "$species" \
        --strain "$short" \
        --prefix "$short" \
        --locustag "$locustag" \
        --centre "$CENTRE" \
        --cpus "${SLURM_CPUS_PER_TASK:-8}" \
        --outdir "$outdir" \
        --proteins /asap/db/sequences/asap-proteins.faa \
        --force \
        --addgenes \
        --rfam \
        --rawproduct \
        --compliant \
        "$input" < /dev/null

[ "$filtered" = yes ] && echo "annotated from the contig-filtered genome $input" > "$outdir/INPUT_FILTERED"
ls -lh "$outdir/${short}.gff" "$outdir/${short}.gbk"
END_OF_SCRIPT
chmod +x "$SCRIPTS/prokka_fallback_array.sh.new" && mv -f "$SCRIPTS/prokka_fallback_array.sh.new" "$SCRIPTS/prokka_fallback_array.sh"
echo "wrote $SCRIPTS/prokka_fallback_array.sh"

use_env download_env
out=$(python "$SCRIPTS/make_prokka_todo.py" --filtered); echo "$out"
ARRAY=$(echo "$out" | sed -n 's/^ARRAY=//p')
if [ -n "$ARRAY" ]; then
    JOB=$(SBATCH_EXTRA="--array=$ARRAY --export=ALL,TODO_FILE=$DOWNSTREAM/prokka_fallback/todo_filtered.tsv" \
          submit prokka_filtered 8 32G 08:00:00 "$SCRIPTS/prokka_fallback_array.sh")
    echo "submitted array job $JOB ($ARRAY)"
else
    echo "No filtered genomes to annotate (set use_filtered in D1.4 and run D1.5 first)."
fi

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like prokka_filtered
ls -l "$DOWNSTREAM"/prokka_fallback/*/INPUT_FILTERED 2>/dev/null || echo "no filtered annotations yet"

## D2. Per-species pan-genomes (Roary)
**D2.0 first:** rebuild the Roary input folders so they follow the gate (isolates that failed are left out; groups use the ANI species). The cell rewrites `prepare_roary_groups.py` itself (v5.3 used whichever copy was on disk) and prints the groups and the isolates left out: check that list before submitting. Re-run D2.0 after any change to the gate. If Roary already ran on ungated groups, `scancel` it and move `downstream/roary/*` aside first.

ASA³P computes one pan-genome per project (= per genus here). These are per-species runs for every species with ≥ 2 isolates (`-i 95`), using the annotations collected in C4. Check that there is one `roary_output` folder per group and no `roary_output_<timestamp>` folders. Groups with fewer than 7 isolates cannot have cloud genes, so `Cloud genes 0` there is expected.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/prepare_roary_groups.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Collect one GFF/GBK per isolate and build Roary input folders.

GFF/GBK source per isolate: the fallback Prokka output
(DOWNSTREAM/prokka_fallback/<short>/) if it exists -- it is only created for
isolates whose ASA3P annotation was missing or unusable -- otherwise ASA3P's
own annotation (from asap_paths.tsv).
  -> DOWNSTREAM/annotation_files.tsv

Roary groups (column "role" in roary_groups.tsv):
  * additional   every species with >= 2 isolates (identity 95, Roary default).
                 ASA3P only computes one pan-genome per project (= per genus here),
                 so per-species pan-genomes are an analysis outside ASA3P.
  * replacement  a genus-level run (identity 80), ONLY for a genus whose ASA3P
                 corepan/ folder came out empty (PROVENANCE/replacements.tsv).
                 Otherwise ASA3P's own genus-level Roary result is the one to use.
A genus-level pan-genome that contains genomes of another genus has almost no core: ASA3P's own genus-level
Roary run for Pectobacterium had 22 core genes because about seven of its genomes were ~78% ANI to Pectobacterium
(a different genus). This replacement run uses only the isolates that passed the species gate. For a genus-level core-gene tree use D4b (clean isolates + Pectobacterium reference genomes);
the Mashtree tree (D3) shows genus-level relationships of all genomes.

GFFs are COPIED (not symlinked) as <short>.gff, so Roary's sample names are
the short names and nothing points outside the container's bind paths.
  -> DOWNSTREAM/roary/<group>/gff/*.gff and DOWNSTREAM/roary/roary_groups.tsv

The species gate (species_check/analysis_set.tsv, D1.2) decides which isolates
are used and under which species: an isolate with include = no is left out of
every group, and groups are formed from the gate's species (the ANI species),
not from the metadata label. Without a gate (D1 not run yet) all isolates are
used with their metadata species, and a note says so.
"""
import csv
import os
import shutil
from collections import defaultdict
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
PROV = Path(os.environ["PROVENANCE"])
FALLBACK = DOWNSTREAM / "prokka_fallback"
ROARY = DOWNSTREAM / "roary"

genus_replacement = set()
if (PROV / "replacements.tsv").exists():
    with open(PROV / "replacements.tsv", newline="") as f:
        for r in csv.DictReader(f, delimiter="\t"):
            if r["step"] == "corepan" and r["decision"].startswith("replacement"):
                genus_replacement.add(r["genus"])
else:
    print("NOTE: no replacements.tsv yet (run Part B) -- building species groups only.")

with open(DOWNSTREAM / "asap_paths.tsv", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))
if not rows:
    raise SystemExit("ERROR: asap_paths.tsv is empty -- run Sec. 7 after ASA3P has finished.")

GATE = DOWNSTREAM / "species_check" / "analysis_set.tsv"
gate = {}
if GATE.exists():
    with open(GATE, newline="") as f:
        gate = {g["short_name"]: g for g in csv.DictReader(f, delimiter="\t")}
else:
    print("NOTE: no species gate yet (D1.2): every isolate is used with its metadata species. "
          "Re-run this cell after D1.2 before D2.")


def has_fasta_section(path):
    with open(path) as f:
        return any(line.startswith("##FASTA") for line in f)


ann_rows, missing, no_fasta, left_out = [], [], [], []
for r in rows:
    s = r["short_name"]
    in_analysis, genus, species = "yes", r["genus"], r["species"]
    if gate:
        g = gate.get(s)
        if g is None:
            in_analysis = "no"
            left_out.append((s, "not in the species check: re-run D1"))
        elif g["include"] != "yes":
            in_analysis = "no"
            left_out.append((s, f"{g['qc_status']}" + (f" ({g['note']})" if g["note"] else "")))
        else:
            genus, species = g["genus_final"], g["species_final"]
    gff, gbk, source = r["gff"], r["gbk"], "asa3p"
    fb_gff, fb_gbk = FALLBACK / s / f"{s}.gff", FALLBACK / s / f"{s}.gbk"
    if fb_gff.exists():   # fallback only exists where ASA3P's annotation was missing or unusable
        gff, gbk, source = str(fb_gff), str(fb_gbk if fb_gbk.exists() else ""), "prokka_fallback"
    if not gff:
        if in_analysis == "yes":
            missing.append(s)
        source = "none"
    elif in_analysis == "yes" and not has_fasta_section(gff):
        no_fasta.append(s)   # Roary needs the ##FASTA block that Prokka appends
    ann_rows.append({"short_name": s, "genus": genus, "species": species,
                     "gff": gff, "gbk": gbk, "source": source, "in_analysis": in_analysis})

with open(DOWNSTREAM / "annotation_files.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(ann_rows[0].keys()), delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(ann_rows)

groups = defaultdict(list)
for a in ann_rows:
    if a["gff"] and a["in_analysis"] == "yes":
        groups[(f"{a['genus']}_{a['species']}", "additional", "species", 95)].append(a)
        if a["genus"] in genus_replacement:
            groups[(f"{a['genus']}_all", "replacement", "genus", 80)].append(a)

ROARY.mkdir(parents=True, exist_ok=True)
summary = []
if not any(len(m) >= 2 for m in groups.values()):
    raise SystemExit("ERROR: no group has >= 2 annotated isolates -- check annotation_files.tsv.")
for (name, role, level, ident), members in sorted(groups.items()):
    if len(members) < 2:
        continue
    gdir = ROARY / name / "gff"
    if gdir.exists():
        shutil.rmtree(gdir)
    gdir.mkdir(parents=True)
    for a in members:
        shutil.copyfile(a["gff"], gdir / f"{a['short_name']}.gff")
    summary.append({"group": name, "role": role, "level": level, "identity": ident, "n": len(members),
                    "gff_dir": str(gdir), "out_dir": str(ROARY / name / "roary_output"),
                    "members": ",".join(a["short_name"] for a in members)})

with open(ROARY / "roary_groups.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(summary[0].keys()), delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(summary)

print(f"{'group':32s} {'role':12s} {'n':>3s}  members")
for s in summary:
    print(f"{s['group']:32s} {s['role']:12s} {s['n']:>3d}  {s['members']}")
if genus_replacement:
    print("\nGenus-level replacement run(s) because ASA3P corepan/ was empty:", ", ".join(sorted(genus_replacement)))
    if "Pectobacterium" in genus_replacement:
        print("  Only isolates that passed the species gate are in this group. For a genus-level core-gene tree, with the "
              "Pectobacterium reference genomes added, use D4b.")
if gate:
    n_in = sum(a["in_analysis"] == "yes" for a in ann_rows)
    print(f"\nSpecies gate: {n_in} of {len(ann_rows)} isolates used; left out:")
    for s_, why in left_out:
        print(f"  {s_:9s} {why[:110]}")
if missing:
    print("\nNo GFF for:", ", ".join(missing), "-> run the fallback Prokka section first.")
if no_fasta:
    print("\nGFF without a ##FASTA section (Roary will reject these):", ", ".join(no_fasta),
          "\n-> annotate these with the fallback Prokka section instead.")
print(f"\nWrote {ROARY/'roary_groups.tsv'} and {DOWNSTREAM/'annotation_files.tsv'}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/prepare_roary_groups.py.new" && mv -f "$SCRIPTS/prepare_roary_groups.py.new" "$SCRIPTS/prepare_roary_groups.py"
echo "wrote $SCRIPTS/prepare_roary_groups.py"

use_env download_env
python "$SCRIPTS/prepare_roary_groups.py"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/run_roary_groups.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Roary (standalone container) for the groups in roary_groups.tsv with a given role:
#   run_roary_groups.sh replacement   genus-level runs for genera whose ASA3P corepan/ step was empty or failed
#   run_roary_groups.sh additional    per-species pan-genomes (outside ASA3P)
#
# Fix vs. first run: the output folder is deleted and NOT re-created before
# Roary starts. If -f points at an existing folder, Roary writes to
# <folder>_<timestamp> instead -- that is where roary_output_container_1778273395
# came from. With the folder absent, Roary writes exactly where we ask.
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

want_role="${1:?role required: replacement or additional}"
GROUPS_TSV="$DOWNSTREAM/roary/roary_groups.tsv"
CPUS="${SLURM_CPUS_PER_TASK:-8}"
failed=()

while IFS=$'\t' read -r group role level ident n gff_dir out_dir _; do
    [ "$role" = "$want_role" ] || continue
    echo "=== $group [$role] ($level, n=$n, -i $ident) ==="
    rm -rf "$out_dir"
    mapfile -t gffs < <(find "$gff_dir" -maxdepth 1 -name '*.gff' | sort)
    if apptainer exec --bind "$ROOT":"$ROOT" "$ROARY_IMG" \
           roary -p "$CPUS" -f "$out_dir" -e -n -i "$ident" "${gffs[@]}" < /dev/null; then
        echo "done: $out_dir"
        grep -E "Core genes|Total genes" "$out_dir/summary_statistics.txt" || true
    else
        echo "WARNING: Roary failed for $group"
        failed+=("$group")
    fi
done < <(tail -n +2 "$GROUPS_TSV")

if [ "${#failed[@]}" -gt 0 ]; then
    echo "Failed groups: ${failed[*]}"
    exit 1
fi
echo "All '$want_role' Roary groups finished."
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_roary_groups.sh.new" && mv -f "$SCRIPTS/run_roary_groups.sh.new" "$SCRIPTS/run_roary_groups.sh"
echo "wrote $SCRIPTS/run_roary_groups.sh"

JOB=$(submit roary_species 16 64G 2-00:00:00 "$SCRIPTS/run_roary_groups.sh" additional); echo "submitted $JOB"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like roary
ls -d "$DOWNSTREAM"/roary/*/roary_output*
for f in "$DOWNSTREAM"/roary/*/roary_output/summary_statistics.txt; do
    echo; echo "== $(basename "$(dirname "$(dirname "$f")")") =="; cat "$f"
done

## D3. Genome-distance tree: Mashtree
The isolates that passed the gate (their standardized contigs, or the filtered genome where one is used) + species references + *Erwinia* outgroup. Root on `OUT_Erwinia_amylovora_ATCC49946`. A second tree has **every** isolate, whatever the species check says, with **bootstrap support** (0–100) on the nodes: `mashtree_ALL_bootstrap.nwk`. This is the tree for the all-isolates figure (Figure 9 of the epidemiology Rmd), with the Mash distances in `mashtree_ALL_distances.tsv`. The isolates that failed the species check are in it (an assembly that is not the expected organism, or is a mixture, can fall anywhere in the tree), so the figure marks every isolate by its ANI. Each bootstrap replicate runs Mashtree again with a different random Mash seed, so the support shows how stable a node is to the sample of k-mers in the sketches, not to resampling of alignment sites; say so in the Methods. `MASHTREE_REPS` sets the number of replicates (default 1000; every replicate re-sketches all genomes, so the run time grows with it). `mashtree_ALL_diagnostic.nwk` is the same genome set from one run, without bootstrap. ASA³P's own FastTreeMP tree (`phylogeny/`) remains the ASA³P result for each genus. For a **core-gene** tree of the whole genus, see D4b.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_mashtree.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Mashtree: species references + Erwinia outgroup + isolates.
#   mashtree_ALL_bootstrap.nwk            EVERY isolate + references + outgroup, whatever the species gate says, with bootstrap
#                                         support (0-100) on the internal nodes. This is the tree for the all-isolates figure.
#   mashtree_ALL_distances.tsv            Mash distance matrix of the same genomes (branch lengths of the trees are Mash distances)
#   mashtree_ALL_bootstrap.info           number of replicates, genomes, Mashtree version (read by the Methods draft)
#   mashtree_ALL_diagnostic.nwk           the same genomes, one run, no bootstrap (same topology as the main tree of the bootstrap run)
#   mashtree_isolates_refs_outgroup.nwk   only the isolates that passed the species gate (D1.2), no bootstrap
# Tip labels are the file names: short names and REF_/OUT_ labels.
# Bootstrap (mashtree_bootstrap.pl): each replicate runs Mashtree again with a different random Mash seed, i.e. on a
# different sample of k-mers in the sketches. The support shows how stable a node is to that sampling, not to a
# resampling of alignment sites. MASHTREE_REPS sets the number of replicates (default 1000).
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

OUT="$DOWNSTREAM/trees"
GATE="${GATE:-$DOWNSTREAM/species_check/analysis_set.tsv}"
REPS="${MASHTREE_REPS:-1000}"
NCPU="${SLURM_CPUS_PER_TASK:-8}"
mkdir -p "$OUT"
# files of an earlier run are removed first: a matrix or bootstrap tree left over from a different set of genomes would be wrong
rm -f "$OUT"/mashtree_ALL_distances.tsv "$OUT"/mashtree_ALL_bootstrap.nwk "$OUT"/mashtree_ALL_bootstrap.nwk.part "$OUT"/mashtree_ALL_bootstrap.info
MT() { apptainer exec --bind "$ROOT":"$ROOT" "$MASHTREE_IMG" "$@"; }
mashtree_run() {   # <output.nwk> <mashtree options and genome files...>
    local out=$1; shift
    MT mashtree --numcpus "$NCPU" "$@" > "$out"
}

mapfile -t refs < <(ls "$REFDIR"/*.fasta)
mapfile -t all_isolates < <(ls "$GENOMES"/*.fna)
[ "${#all_isolates[@]}" -gt 0 ] || { echo "ERROR: no genomes in $GENOMES"; exit 1; }

# ---------------------------------------------------------------- isolates that passed the species gate
if [ -s "$GATE" ]; then
    # include = yes; genome_path is the contig-filtered genome where one is used
    mapfile -t isolates < <(awk -F'\t' 'NR == 1 {for (i = 1; i <= NF; i++) c[$i] = i; next} $c["include"] == "yes" {print $c["genome_path"]}' "$GATE")
else
    echo "WARNING: no species gate ($GATE): using every isolate. Run D1.2 first."
    isolates=("${all_isolates[@]}")
fi
[ "${#isolates[@]}" -gt 0 ] || { echo "ERROR: the gate includes no isolates"; exit 1; }
inputs=("${isolates[@]}" "${refs[@]}")
echo "Genomes in the gated tree: ${#isolates[@]} isolates + ${#refs[@]} references/outgroup"
printf '  %s\n' "${inputs[@]##*/}"
mashtree_run "$OUT/mashtree_isolates_refs_outgroup.nwk" "${inputs[@]}"
echo "Tree: $OUT/mashtree_isolates_refs_outgroup.nwk (root on OUT_Erwinia_amylovora_ATCC49946)"

# ---------------------------------------------------------------- every isolate (+ distance matrix)
all_inputs=("${all_isolates[@]}" "${refs[@]}")
echo "Genomes in the all-isolates tree: ${#all_isolates[@]} isolates + ${#refs[@]} references/outgroup (no species gate)"
h=$(MT mashtree --help 2>&1 || true)
flags=()
if grep -q -- '--outmatrix' <<< "$h"; then flags+=(--outmatrix "$OUT/mashtree_ALL_distances.tsv")
else echo "NOTE: this Mashtree has no --outmatrix: no distance matrix is written"; fi
mashtree_run "$OUT/mashtree_ALL_diagnostic.nwk" ${flags[@]+"${flags[@]}"} "${all_inputs[@]}"
echo "Tree with every isolate: $OUT/mashtree_ALL_diagnostic.nwk"
[ -s "$OUT/mashtree_ALL_distances.tsv" ] && echo "Mash distances: $OUT/mashtree_ALL_distances.tsv"

# ---------------------------------------------------------------- bootstrap
# After the plain trees, so that a problem here never costs them (the job then ends with an error code).
BOOT="$OUT/mashtree_ALL_bootstrap.nwk"
rc=0
if MT bash -c 'command -v mashtree_bootstrap.pl' >/dev/null 2>&1; then
    echo "Bootstrap: $REPS replicates (a different Mash seed for each; this takes a while)"
    if MT mashtree_bootstrap.pl --reps "$REPS" --numcpus "$NCPU" "${all_inputs[@]}" > "$BOOT.part" \
       && [ -s "$BOOT.part" ] && grep -Eq '\)[0-9]+[:;,)]' "$BOOT.part"; then
        mv -f "$BOOT.part" "$BOOT"
        {
            printf 'replicates\t%s\n' "$REPS"
            printf 'isolates\t%s\n' "${#all_isolates[@]}"
            printf 'references_and_outgroup\t%s\n' "${#refs[@]}"
            printf 'mashtree\t%s\n' "$(MT mashtree --version 2>&1 | tail -1)"
        } > "$OUT/mashtree_ALL_bootstrap.info"
        echo "Bootstrap tree: $BOOT"
        grep -oE '\)[0-9]+' "$BOOT" | tr -d ')' | sort -n | awk '{v[NR] = $1} END {
            for (i = 1; i <= NR; i++) {if (v[i] < 70) lt70++; if (v[i] == 100) eq100++}
            printf "support values: %d nodes, minimum %d, median %d, %d below 70, %d equal to 100\n", NR, v[1], v[int((NR + 1) / 2)], lt70, eq100}'
    else
        echo "ERROR: the bootstrap run failed or wrote no support values (see the log above); the trees without bootstrap are fine."
        rm -f "$BOOT.part"; rc=3
    fi
else
    echo "ERROR: mashtree_bootstrap.pl is not in the Mashtree image ($MASHTREE_IMG); the trees without bootstrap are fine."
    rc=3
fi
exit $rc
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_mashtree.sh.new" && mv -f "$SCRIPTS/run_mashtree.sh.new" "$SCRIPTS/run_mashtree.sh"
echo "wrote $SCRIPTS/run_mashtree.sh"

export MASHTREE_REPS=1000      # bootstrap replicates for the all-isolates tree (100 is the least worth reporting)
JOB=$(submit mashtree 16 64G 1-00:00:00 "$SCRIPTS/run_mashtree.sh"); echo "submitted $JOB"

## D4. Per-species core-gene trees (IQ-TREE)
From the D2 core-gene alignments. n < 3: no tree (most species once the gate has removed the failed isolates). n = 3: GTR+G, no support values — an unrooted three-taxon tree has one possible topology, so report branch lengths only. n ≥ 4 (so far only *P. versatile*, 8 isolates after the gate): ModelFinder + 1000 UFBoot + 1000 SH-aLRT.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_iqtree_species.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# IQ-TREE on each species-level Roary core-gene alignment.
#   n < 3   skipped    (no tree possible)
#   n = 3   GTR+G, no support values -- an unrooted 3-taxon tree has only one
#           possible topology, so only the branch lengths carry information
#   n >= 4  ModelFinder + 1000 UFBoot + 1000 SH-aLRT
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"
module load "$IQTREE_MODULE"
IQ=$(command -v iqtree2 || command -v iqtree) || { echo "IQ-TREE not found"; exit 1; }

ROARY="$DOWNSTREAM/roary"
failed=()
while IFS=$'\t' read -r group _ level _ n _ out_dir _; do
    [ "$level" = "species" ] || continue
    aln="$out_dir/core_gene_alignment.aln"
    echo "=== $group (n=$n) ==="
    if [ ! -s "$aln" ]; then echo "no alignment, skipping"; continue; fi
    nseq=$(grep -c '^>' "$aln")
    if [ "$nseq" -lt 3 ]; then echo "only $nseq sequences: no tree"; continue; fi

    tdir="$ROARY/$group/iqtree_core"
    mkdir -p "$tdir"
    cp "$aln" "$tdir/${group}_core.aln"
    cd "$tdir" || exit 1
    if [ "$nseq" -eq 3 ]; then
        "$IQ" -s "${group}_core.aln" -m GTR+G -T AUTO --prefix "${group}_core" --redo
    else
        "$IQ" -s "${group}_core.aln" -m MFP -B 1000 -alrt 1000 -T AUTO --prefix "${group}_core" --redo
    fi || failed+=("$group")
done < <(tail -n +2 "$ROARY/roary_groups.tsv")

[ "${#failed[@]}" -eq 0 ] || { echo "Failed: ${failed[*]}"; exit 1; }
echo "IQ-TREE runs finished."
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_iqtree_species.sh.new" && mv -f "$SCRIPTS/run_iqtree_species.sh.new" "$SCRIPTS/run_iqtree_species.sh"
echo "wrote $SCRIPTS/run_iqtree_species.sh"

JOB=$(submit iqtree_species 16 64G 1-00:00:00 "$SCRIPTS/run_iqtree_species.sh"); echo "submitted $JOB"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like "mashtree|iqtree"
ls -lh "$DOWNSTREAM"/trees/mashtree_*
cat "$DOWNSTREAM/trees/mashtree_ALL_bootstrap.info" 2>/dev/null || echo "no bootstrap tree yet (see the end of the mashtree log in $LOGS)"
find "$DOWNSTREAM/roary" -name "*.treefile" | sort

## D4b. Genus-level core-gene tree (clean isolates + *Pectobacterium* reference genomes)
**Why this step exists.** ASA³P's genus-level Roary run for *Pectobacterium* found **22 core genes**, too few for a tree. The cause is the genome set, not Roary: the core is the set of genes present in ≥ 99 % of the genomes (in **all** of them while there are fewer than 100), and that project holds about seven genomes that are not *Pectobacterium* (the isolates that failed the species check, best fastANI about 78 %). Genes of a genome that far away do not cluster with the others at the identity Roary uses, so a single such genome takes nearly every gene out of the core. (Mixed assemblies add the second genome's genes to the pan-genome; they are left out here as well, because they are not one organism.) Once those genomes are out, only about seven distinct clean isolates and six references are left, which is thin for a genus-wide tree.

**What D4b does.** The tree uses only (1) the isolates that passed the species check and are *Pectobacterium*, and (2) *Pectobacterium* reference genomes: the six species references of A2, plus, if you want them, the type-strain genomes NCBI holds for the other described species (D4b.1). *Dickeya* and *Erwinia* are left out on purpose (at about 78 % ANI they share too few genes with *Pectobacterium* to be in a core alignment); for them, see the Mashtree tree (D3). Every genome is annotated with the same Prokka 1.12 and ASA³P protein database, clustered with Roary (`-i 90`, `-cd 99`), the core genes are aligned with MAFFT, and IQ-TREE infers the tree (ModelFinder within the GTR family, 1,000 UFBoot, 1,000 SH-aLRT, then site concordance factors).

| Step | What | Where it runs |
|---|---|---|
| D4b.1 | Find type-strain genomes at NCBI, download the ones you keep (optional) | login node |
| D4b.2 | Prokka for the references | SLURM array |
| D4b.3 | Choose the genome set | login node |
| D4b.4 | Roary | SLURM |
| D4b.5 | Check the core; which genomes lack the common genes? | login node |
| D4b.6 | IQ-TREE (+ concordance factors), write `core_gene_tree.nwk` | SLURM |

**Run it after D1 (the species gate) and D2.0** (which writes `annotation_files.tsv`). D4b.1 needs nothing else and can be done at any time.

### D4b.1 More *Pectobacterium* reference genomes (optional, login node)
The tree shows where your isolates sit among the described species only if those species are in it, so the more *Pectobacterium* species have a reference, the better. `find_type_strains.py` asks NCBI (Datasets API, no account) for every *Pectobacterium* assembly flagged as **type material**, keeps one per species (RefSeq before GenBank, then the most complete, then the fewest contigs), skips species that already have a reference in `reference_genomes.tsv`, and writes two tables to `metadata/`:

* `core_references_candidates.tsv`: every type-material assembly seen, and why it was or was not chosen.
* `core_references.tsv`: the chosen ones (same columns as `reference_genomes.tsv`, plus the strain, type label and assembly level). An existing `core_references.tsv` is **never overwritten** (the proposal goes to `core_references.proposed.tsv`).

Accessions come only from NCBI's answer. If the request fails (no internet on the node, or NCBI changed its API), the cell writes an empty template and says where to look; fill it in by hand and carry on. Set `NCBI_API_KEY` first if NCBI throttles you.

**Check the list before downloading.** NCBI's species names can differ from the validly published ones, and a type strain in NCBI may carry an older name; compare with the current *Pectobacterium* species list (LPSN, or Ma et al. 2024).

In [2]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/find_type_strains.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Propose Pectobacterium type-strain genomes for the core-gene tree (D4b.1).

Why: the core tree needs more than six reference genomes. A tree of seven clean isolates and six references says
little about where the isolates sit among the described species. The type strain of each species is the usual anchor.

What it does (login node, needs internet; the NCBI Datasets REST API, no account needed):
  1. lists every Pectobacterium assembly in NCBI (GenBank + RefSeq) and keeps those flagged as type material;
  2. keeps one assembly per species (RefSeq before GenBank, then Complete Genome > Chromosome > Scaffold > Contig,
     then fewer contigs and a higher N50), skipping assemblies that are atypical, have more than MAX_CONTIGS contigs
     or fail the CheckM thresholds when NCBI reports them;
  3. leaves out species that already have a reference in reference_genomes.tsv (use --all to list them too);
  4. writes META/core_references_candidates.tsv (every type-material assembly seen and why it was or was not chosen)
     and META/core_references.tsv (the chosen ones; same columns as reference_genomes.tsv plus a few info columns).
     An existing core_references.tsv is never overwritten (the proposal goes to core_references.proposed.tsv),
     because you may have edited it. --force overwrites it.

Accessions come only from NCBI's answer. Nothing is invented: if the request fails, a template with the column
names is written (the exit status is still 0, so "run all" goes on) and you fill the accessions in by hand (NCBI Genome > Pectobacterium > filter "Type material").

Options:
  --json FILE   read a saved dataset_report (the JSON NCBI returns) instead of calling the API
  --all         also list species that already have a reference
  --force       overwrite META/core_references.tsv
Environment: NCBI_API_KEY (optional, raises the rate limit), MAX_CONTIGS (default 500),
             MIN_COMPLETENESS (default 95), MAX_CONTAMINATION (default 5).
Standard library only.
"""
import csv
import json
import os
import re
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

META = Path(os.environ["META"])
GENUS = "Pectobacterium"
API = "https://api.ncbi.nlm.nih.gov/datasets/v2/genome/taxon/{taxon}/dataset_report"
MAX_CONTIGS = int(os.environ.get("MAX_CONTIGS", "500"))
MIN_COMP = float(os.environ.get("MIN_COMPLETENESS", "95"))
MAX_CONT = float(os.environ.get("MAX_CONTAMINATION", "5"))
LEVEL_RANK = {"complete genome": 0, "chromosome": 1, "scaffold": 2, "contig": 3}
TYPE_LABELS = {"TYPE_MATERIAL", "NEOTYPE", "SYNTYPE", "PATHOTYPE"}     # not REFTYPE (reference material, not type)
argv = sys.argv[1:]
force, show_all = "--force" in argv, "--all" in argv
json_file = argv[argv.index("--json") + 1] if "--json" in argv else None


def get(url, tries=4):
    headers = {"Accept": "application/json", "User-Agent": "asap-notebook/5.5"}
    if os.environ.get("NCBI_API_KEY"):
        headers["api-key"] = os.environ["NCBI_API_KEY"]
    for i in range(1, tries + 1):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=120) as r:
                return json.load(r)
        except urllib.error.HTTPError as e:
            if e.code in (400, 404, 422):          # the request itself is wrong: do not repeat it
                raise
            err = e
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            err = e
        print(f"  attempt {i} of {tries} failed ({err}); waiting", file=sys.stderr)
        time.sleep(10 * i)
    raise err


def fetch_reports():
    reports, token = [], None
    # server-side filters are an optimisation only; if NCBI rejects them the unfiltered request is used
    for params in ({"filters.exclude_atypical": "true", "page_size": "1000"}, {"page_size": "1000"}):
        reports, token = [], None
        try:
            while True:
                q = dict(params)
                if token:
                    q["page_token"] = token
                data = get(API.format(taxon=GENUS) + "?" + urllib.parse.urlencode(q))
                reports += data.get("reports", [])
                token = data.get("next_page_token")
                print(f"  {len(reports)} of {data.get('total_count', '?')} assemblies", file=sys.stderr)
                if not token:
                    return reports
        except urllib.error.HTTPError as e:
            print(f"  NCBI rejected the request ({e.code}); trying without the optional filters", file=sys.stderr)
    return reports


def find_key(obj, key):
    """First value stored under `key` anywhere in a nested report (NCBI has moved type_material between versions)."""
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            r = find_key(v, key)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = find_key(v, key)
            if r is not None:
                return r
    return None


def num(x, default=0.0):
    try:
        return float(x)
    except (TypeError, ValueError):
        return default


def species_of(name):
    """'Pectobacterium versatile' -> 'versatile'; None for 'Pectobacterium sp.' and names that are not a species."""
    m = re.match(r"^(?:Candidatus\s+)?" + GENUS + r"\s+([a-z][a-z-]+)\b", name or "")
    if not m or m.group(1) in {"sp", "spp", "cf", "aff"}:
        return None
    return m.group(1)


def row_of(rep):
    org = rep.get("organism") or {}
    name = org.get("organism_name", "")
    strain = (org.get("infraspecific_names") or {}).get("strain", "") or find_key(org, "strain") or ""
    info = rep.get("assembly_info") or {}
    stats = rep.get("assembly_stats") or {}
    tm = find_key(rep, "type_material")
    label = ""
    if isinstance(tm, dict):
        label = tm.get("type_label") or ("TYPE_MATERIAL" if tm.get("type_display_text") else "")
    ani = (rep.get("average_nucleotide_identity") or {})
    chk = rep.get("checkm_info") or {}
    acc = rep.get("accession", "")
    return {
        "accession": acc,
        "source": "RefSeq" if acc.startswith("GCF_") else "GenBank",
        "organism": name,
        "species": species_of(name),
        "subspecies": "subsp." in name,
        "strain": str(strain),
        "type_label": label,
        "level": info.get("assembly_level", ""),
        "status": info.get("assembly_status", "current"),
        "atypical": bool((info.get("atypical") or {}).get("is_atypical")) or bool(rep.get("atypical")),
        "contigs": int(num(stats.get("number_of_contigs"))),
        "n50": int(num(stats.get("contig_n50"))),
        "size_mb": round(num(stats.get("total_sequence_length")) / 1e6, 2),
        "completeness": chk.get("completeness"),
        "contamination": chk.get("contamination"),
        "ani_status": ani.get("taxonomy_check_status", ""),
    }


def reject_reason(r):
    if r["species"] is None:
        return "not a named species"
    if r["status"] and r["status"] not in ("current", ""):
        return f"assembly status {r['status']}"
    if r["atypical"]:
        return "atypical assembly"
    if r["contigs"] and r["contigs"] > MAX_CONTIGS:
        return f"{r['contigs']} contigs (> {MAX_CONTIGS})"
    if r["completeness"] is not None and num(r["completeness"], 100) < MIN_COMP:
        return f"CheckM completeness {r['completeness']}"
    if r["contamination"] is not None and num(r["contamination"], 0) > MAX_CONT:
        return f"CheckM contamination {r['contamination']}"
    if r["ani_status"].lower() in ("failed", "mismatch"):
        return f"NCBI taxonomy check: {r['ani_status']}"
    return ""


def rank(r):
    return (0 if r["source"] == "RefSeq" else 1, 1 if r["subspecies"] else 0, LEVEL_RANK.get(r["level"].lower(), 4),
            r["contigs"] or 10 ** 6, -r["n50"])


# ------------------------------------------------------------------ get the reports
reports = None
try:
    if json_file:
        data = json.load(open(json_file))
        reports = data.get("reports", data) if isinstance(data, dict) else data
    else:
        print(f"Asking NCBI Datasets for all {GENUS} assemblies ...", file=sys.stderr)
        reports = fetch_reports()
    if not reports:
        raise RuntimeError("NCBI returned no assemblies")
except Exception as e:                                         # network, JSON, empty answer
    tpl = META / "core_references.template.tsv"
    META.mkdir(parents=True, exist_ok=True)
    tpl.write_text("label\taccession\tgenus\tspecies\trole\n")
    print(f"\nCould not get the assembly list from NCBI: {e}\n"
          f"Wrote an empty template with the column names: {tpl}\n"
          f"Fill it by hand (one row per type-strain genome; label like REF_Pwasabiae_CFBP3304, accession GCF_/GCA_, genus {GENUS},\n"
          "species the epithet, role reference), save it as core_references.tsv in the same folder, and go on with D4b.1b.\n"
          "Where to look: NCBI Datasets > Genome > Pectobacterium > filter 'Type material' (or `datasets summary genome taxon Pectobacterium --from-type`).")
    sys.exit(0)     # not an error of the notebook: the list is then filled in by hand

rows = [row_of(r) for r in reports]
types = [r for r in rows if r["type_label"] in TYPE_LABELS]
print(f"{len(rows)} {GENUS} assemblies, {len(types)} from type material")
if not types:
    print("NCBI's answer holds no type_material field for any assembly: nothing to propose. "
          "Fill core_references.template.tsv by hand (see above).")
    (META / "core_references.template.tsv").write_text("label\taccession\tgenus\tspecies\trole\n")
    sys.exit(0)     # not an error of the notebook: the list is then filled in by hand

existing_species, existing_labels = set(), set()
ref_path = META / "reference_genomes.tsv"
if ref_path.exists():
    for r in csv.DictReader(open(ref_path, newline=""), delimiter="\t"):
        existing_labels.add(r["label"])
        if r["genus"] == GENUS and r["role"] == "reference":
            existing_species.add(r["species"])

# ------------------------------------------------------------------ choose one per species
by_species = {}
for r in types:
    r["why"] = reject_reason(r)
    if not r["why"]:
        by_species.setdefault(r["species"], []).append(r)
chosen = {}
for sp, lst in by_species.items():
    lst.sort(key=rank)
    chosen[sp] = lst[0]
for r in types:
    if r["why"]:
        continue
    r["why"] = "chosen" if chosen[r["species"]] is r else "another assembly of this species ranks higher"
    if chosen[r["species"]] is r and r["species"] in existing_species and not show_all:
        r["why"] = "species already has a reference in reference_genomes.tsv"

fields = ["accession", "source", "organism", "species", "strain", "type_label", "level", "contigs", "n50", "size_mb",
          "completeness", "contamination", "ani_status", "why"]
cand = META / "core_references_candidates.tsv"
with open(cand, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields, delimiter="\t", extrasaction="ignore", lineterminator="\n")
    w.writeheader()
    for r in sorted(types, key=lambda r: (r["species"] or "~", rank(r))):
        w.writerow({k: ("" if r[k] is None else r[k]) for k in fields})
print(f"Wrote {cand} ({len(types)} type-material assemblies)")

picked = [r for r in chosen.values() if r["why"] == "chosen"]
used = set(existing_labels)
out_rows = []
for r in sorted(picked, key=lambda r: r["species"]):
    strain = re.sub(r"[^A-Za-z0-9]", "", r["strain"]) or r["accession"].replace("_", "").replace(".", "")
    label = f"REF_P{r['species']}_{strain}"
    while label in used:
        label += "x"
    used.add(label)
    out_rows.append({"label": label, "accession": r["accession"], "genus": GENUS, "species": r["species"], "role": "reference",
                     "strain": r["strain"], "type_label": r["type_label"], "level": r["level"], "contigs": r["contigs"],
                     "size_mb": r["size_mb"]})
cols = ["label", "accession", "genus", "species", "role", "strain", "type_label", "level", "contigs", "size_mb"]
target = META / "core_references.tsv"
if target.exists() and not force:
    target = META / "core_references.proposed.tsv"
    print(f"{META / 'core_references.tsv'} exists and is left as it is (use --force to overwrite)")
with open(target, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols, delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(out_rows)

print(f"\nPectobacterium type-strain genomes proposed for the core tree: {len(out_rows)}\n")
print(f"{'label':36s} {'accession':17s} {'level':16s} {'contigs':>7s}  type")
for r in out_rows:
    print(f"{r['label']:36s} {r['accession']:17s} {r['level']:16s} {r['contigs']:>7}  {r['type_label']}")
left = sorted(sp for sp in by_species if sp in existing_species and not show_all)
if left:
    print("\nSpecies that already have a reference (not repeated): " + ", ".join(left))
print(f"\nWrote {target}\nCheck it before D4b.1b: delete rows you do not want, and compare the species with the current "
      "Pectobacterium list (LPSN / Ma et al. 2024). Names in NCBI can differ from the validly published names.")
END_OF_SCRIPT
chmod +x "$SCRIPTS/find_type_strains.py.new" && mv -f "$SCRIPTS/find_type_strains.py.new" "$SCRIPTS/find_type_strains.py"
echo "wrote $SCRIPTS/find_type_strains.py"

# export NCBI_API_KEY=...        # optional
use_env download_env
python "$SCRIPTS/find_type_strains.py"

wrote /home/s94m696/asap_shortname_projects/scripts/find_type_strains.py


Asking NCBI Datasets for all Pectobacterium assemblies ...
  1000 of 1703 assemblies
  1703 of 1703 assemblies


1703 Pectobacterium assemblies, 80 from type material
Wrote /home/s94m696/asap_shortname_projects/metadata/core_references_candidates.tsv (80 type-material assemblies)

Pectobacterium type-strain genomes proposed for the core tree: 12

label                                accession         level            contigs  type
REF_Pactinidiae_KKH3                 GCF_000803315.1   Contig                 3  TYPE_MATERIAL
REF_Paroidearum_NCPPB929             GCF_041228105.1   Complete Genome        1  TYPE_MATERIAL
REF_Pbetavasculorum_NCPPB2795        GCF_000749845.1   Contig                93  TYPE_MATERIAL
REF_Pjejuense_13115                  GCF_026800125.1   Contig                73  TYPE_MATERIAL
REF_Pnicotianae_21LCBS03             GCF_023272715.1   Complete Genome        1  TYPE_MATERIAL
REF_Pparmentieri_RNS08421A           GCF_001742145.1   Complete Genome        1  TYPE_MATERIAL
REF_Pparvum_LMG30828                 GCF_040224335.1   Contig                62  TYPE_MATERIAL
REF_Pperuvien

In [3]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# Edit the list here if needed. Examples (remove the # to use them):
#   mv "$META/core_references.proposed.tsv" "$META/core_references.tsv"     # when an older core_references.tsv was left as it was
#   sed -i '/^REF_Pexamplespecies_STRAIN\t/d' "$META/core_references.tsv"     # drop one reference
if [ -s "$META/core_references.tsv" ]; then
    echo "--- $META/core_references.tsv ---"
    cut -f1,2,4,6-9 "$META/core_references.tsv" | sed 's/\t/   /g' | cut -c1-150
else
    echo "no core_references.tsv (D4b.1 did not write one): the core tree then uses the references of A2 only"
fi
ls "$META"/core_references.* 2>/dev/null

--- /home/s94m696/asap_shortname_projects/metadata/core_references.tsv ---
label   accession   species   strain   type_label   level   contigs
REF_Pactinidiae_KKH3   GCF_000803315.1   actinidiae   KKH3   TYPE_MATERIAL   Contig   3
REF_Paroidearum_NCPPB929   GCF_041228105.1   aroidearum   NCPPB 929   TYPE_MATERIAL   Complete Genome   1
REF_Pbetavasculorum_NCPPB2795   GCF_000749845.1   betavasculorum   NCPPB 2795   TYPE_MATERIAL   Contig   93
REF_Pjejuense_13115   GCF_026800125.1   jejuense   13-115   TYPE_MATERIAL   Contig   73
REF_Pnicotianae_21LCBS03   GCF_023272715.1   nicotianae   21LCBS03   TYPE_MATERIAL   Complete Genome   1
REF_Pparmentieri_RNS08421A   GCF_001742145.1   parmentieri   RNS 08-42-1A   TYPE_MATERIAL   Complete Genome   1
REF_Pparvum_LMG30828   GCF_040224335.1   parvum   LMG 30828   TYPE_MATERIAL   Contig   62
REF_Pperuviense_IFB5232   GCF_002847345.1   peruviense   IFB5232   TYPE_MATERIAL   Contig   60
REF_Ppolonicum_LMG31077   GCF_043964385.1   polonicum   LMG 31077

**D4b.1b Download them** (login node; skips what is already there). The FASTA files go to `references/core/`, with the same short headers as the references of A2.

In [4]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/download_core_references.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Download the extra Pectobacterium reference genomes of META/core_references.tsv into REFDIR/core/ (D4b.1b).

Columns used: label, accession, genus, species, role (any other column is ignored).
Writes REFDIR/core/<label>.fasta with short headers (<label>_contig_000001 ...), the same format as download_references.py.
Already-present FASTA files are skipped. Run it on the login node (compute nodes may have no internet access).
A download that fails is reported at the end and does not stop the others; the exit status is then 1.
"""
import csv
import os
import shutil
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

META = Path(os.environ["META"])
REFDIR = Path(os.environ["REFDIR"]) / "core"
DL = REFDIR / "downloads"
REFDIR.mkdir(parents=True, exist_ok=True)
DL.mkdir(parents=True, exist_ok=True)
table = META / "core_references.tsv"
if not table.exists():
    sys.exit(f"ERROR: {table} not found -- run D4b.1 first (it proposes the table), or write it by hand.")


def read_fasta(path):
    header, seq = None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq)
                header, seq = line[1:], []
            else:
                seq.append(line)
    if header is not None:
        yield header, "".join(seq)


def write_wrapped(out, seq, width=80):
    for i in range(0, len(seq), width):
        out.write(seq[i:i + width] + "\n")


with open(table, newline="") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))
if not rows:
    sys.exit(f"ERROR: {table} has no rows")

failed = []
for r in rows:
    label, acc = r["label"].strip(), r["accession"].strip()
    if not label or not acc.startswith(("GCF_", "GCA_")):
        failed.append((label or "?", f"not an assembly accession: {acc!r}"))
        continue
    final = REFDIR / f"{label}.fasta"
    if final.exists() and final.stat().st_size > 0:
        print(f"exists, skipping: {final.name}")
        continue
    url = (f"https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/{acc}"
           f"/download?include_annotation_type=GENOME_FASTA")
    zpath, xdir = DL / f"{label}_{acc}.zip", DL / f"{label}_{acc}"
    if xdir.exists():
        shutil.rmtree(xdir)
    print(f"downloading {label} ({acc}) ...")
    try:
        for attempt in (1, 2, 3):
            try:
                urllib.request.urlretrieve(url, zpath)
                break
            except Exception as e:
                if attempt == 3:
                    raise
                print(f"  attempt {attempt} failed ({e}); waiting")
                time.sleep(15 * attempt)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(xdir)
        fnas = sorted(xdir.rglob("*.fna"))
        if not fnas:
            raise RuntimeError("no .fna in the download")
        n, total = 0, 0
        with final.open("w") as out:
            for fna in fnas:
                for _, seq in read_fasta(fna):
                    n += 1
                    total += len(seq)
                    out.write(f">{label}_contig_{n:06d}\n")
                    write_wrapped(out, seq)
        print(f"  wrote {final.name}: {n} sequences, {total:,} bp")
    except Exception as e:
        failed.append((label, str(e)))
        final.unlink(missing_ok=True)

print("Extra references in", REFDIR)
if failed:
    print("\nFAILED:")
    for lab, why in failed:
        print(f"  {lab}: {why}")
    sys.exit(1)
END_OF_SCRIPT
chmod +x "$SCRIPTS/download_core_references.py.new" && mv -f "$SCRIPTS/download_core_references.py.new" "$SCRIPTS/download_core_references.py"
echo "wrote $SCRIPTS/download_core_references.py"

if [ ! -s "$META/core_references.tsv" ]; then
    echo "No core_references.tsv: skipping (the core tree will use the references of A2 only)."; exit 0
fi
use_env download_env
python "$SCRIPTS/download_core_references.py"
ls -lh "$REFDIR"/core/*.fasta

wrote /home/s94m696/asap_shortname_projects/scripts/download_core_references.py
downloading REF_Pactinidiae_KKH3 (GCF_000803315.1) ...
  wrote REF_Pactinidiae_KKH3.fasta: 3 sequences, 4,922,167 bp
downloading REF_Paroidearum_NCPPB929 (GCF_041228105.1) ...
  wrote REF_Paroidearum_NCPPB929.fasta: 1 sequences, 5,281,458 bp
downloading REF_Pbetavasculorum_NCPPB2795 (GCF_000749845.1) ...
  wrote REF_Pbetavasculorum_NCPPB2795.fasta: 93 sequences, 4,685,213 bp
downloading REF_Pjejuense_13115 (GCF_026800125.1) ...
  wrote REF_Pjejuense_13115.fasta: 73 sequences, 5,069,478 bp
downloading REF_Pnicotianae_21LCBS03 (GCF_023272715.1) ...
  wrote REF_Pnicotianae_21LCBS03.fasta: 1 sequences, 4,927,525 bp
downloading REF_Pparmentieri_RNS08421A (GCF_001742145.1) ...
  wrote REF_Pparmentieri_RNS08421A.fasta: 1 sequences, 5,030,841 bp
downloading REF_Pparvum_LMG30828 (GCF_040224335.1) ...
  wrote REF_Pparvum_LMG30828.fasta: 62 sequences, 4,654,067 bp
downloading REF_Pperuviense_IFB5232 (GCF_002847345.1) 

### D4b.2 Annotate the references with Prokka
The same Prokka 1.12 and ASA³P protein database as the isolates (RefSeq annotations come from a different pipeline, and genomes annotated differently cluster less cleanly). One reference per SLURM array task; only references without a finished annotation are submitted. `REDO=yes` annotates all of them again. Only *Pectobacterium* references are used here: *Dickeya* references and the *Erwinia* outgroup are skipped on purpose.

In [5]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/make_refs_todo.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""List the Pectobacterium reference genomes that still need a Prokka annotation (D4b.2).

The core-gene tree (D4b) needs reference genomes annotated with the SAME Prokka 1.12 + ASA3P protein database
as the isolates, so that Roary compares like with like (RefSeq annotations come from another pipeline).

References used:
  * META/reference_genomes.tsv   rows with genus Pectobacterium and role reference (FASTA: REFDIR/<label>.fasta)
  * META/core_references.tsv     optional extra Pectobacterium reference genomes for the core tree only
                                 (same columns; FASTA: REFDIR/core/<label>.fasta, from download_core_references.py)
Dickeya references and the Erwinia outgroup are NOT used: at about 78% ANI they share too few genes with
Pectobacterium to keep a core (the genus-level run with such genomes had 22 core genes).
  -> DOWNSTREAM/prokka_refs/todo.tsv   (label, genus, species, fasta, locustag)
--redo lists every reference again, also those already annotated.
Prints the --array range to pass to sbatch.
"""
import csv
import os
import sys
import zlib
from pathlib import Path

META = Path(os.environ["META"])
REFDIR = Path(os.environ["REFDIR"])
DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
OUT = DOWNSTREAM / "prokka_refs"
redo = "--redo" in sys.argv


def read_tsv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


def has_fasta_section(path):
    with open(path) as f:
        return any(line.startswith("##FASTA") for line in f)


refs = []
for r in read_tsv(META / "reference_genomes.tsv"):
    if r["genus"] == "Pectobacterium" and r["role"] == "reference":
        refs.append((r["label"], r["genus"], r["species"], REFDIR / f"{r['label']}.fasta"))
extra = META / "core_references.tsv"
if extra.exists():
    for r in read_tsv(extra):
        if r["genus"] != "Pectobacterium":
            print(f"NOTE: {r['label']} is {r['genus']}, not Pectobacterium: not used in the core tree")
            continue
        if r["role"] == "outgroup":
            print(f"NOTE: {r['label']} has role outgroup: not used (too distant for a core alignment)")
            continue
        refs.append((r["label"], r["genus"], r["species"], REFDIR / "core" / f"{r['label']}.fasta"))
if not refs:
    raise SystemExit("ERROR: no Pectobacterium reference in reference_genomes.tsv / core_references.tsv")
labels = [x[0] for x in refs]
if len(set(labels)) != len(labels):
    raise SystemExit("ERROR: a reference label appears twice: " + ", ".join(sorted({x for x in labels if labels.count(x) > 1})))

OUT.mkdir(parents=True, exist_ok=True)
todo, done, missing = [], [], []
for label, genus, species, fasta in refs:
    if not (fasta.exists() and fasta.stat().st_size > 0):
        missing.append((label, fasta))
        continue
    gff = OUT / label / f"{label}.gff"
    if gff.exists() and has_fasta_section(gff) and not redo:
        done.append(label)
        continue
    tag = "RF" + format(zlib.crc32(label.encode()) & 0xFFFFFF, "06X")      # 8 characters, stable for a label
    todo.append((label, genus, species, str(fasta), tag))

with open(OUT / "todo.tsv", "w") as f:
    f.write("label\tgenus\tspecies\tfasta\tlocustag\n")
    for t in todo:
        f.write("\t".join(t) + "\n")

print(f"{len(refs)} Pectobacterium reference(s): {len(done)} already annotated, {len(todo)} to annotate, {len(missing)} without a FASTA")
for t in todo:
    print(f"  to annotate: {t[0]:34s} {t[1]} {t[2]}")
for label, fasta in missing:
    print(f"  NO FASTA:    {label}  (expected {fasta}) -> run D4b.1 (or A2 for the original references)")
print(f"ARRAY=1-{len(todo)}" if todo else "ARRAY=")
END_OF_SCRIPT
chmod +x "$SCRIPTS/make_refs_todo.py.new" && mv -f "$SCRIPTS/make_refs_todo.py.new" "$SCRIPTS/make_refs_todo.py"
echo "wrote $SCRIPTS/make_refs_todo.py"

wrote /home/s94m696/asap_shortname_projects/scripts/make_refs_todo.py


In [6]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/make_refs_todo.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""List the Pectobacterium reference genomes that still need a Prokka annotation (D4b.2).

The core-gene tree (D4b) needs reference genomes annotated with the SAME Prokka 1.12 + ASA3P protein database
as the isolates, so that Roary compares like with like (RefSeq annotations come from another pipeline).

References used:
  * META/reference_genomes.tsv   rows with genus Pectobacterium and role reference (FASTA: REFDIR/<label>.fasta)
  * META/core_references.tsv     optional extra Pectobacterium reference genomes for the core tree only
                                 (same columns; FASTA: REFDIR/core/<label>.fasta, from download_core_references.py)
Dickeya references and the Erwinia outgroup are NOT used: at about 78% ANI they share too few genes with
Pectobacterium to keep a core (the genus-level run with such genomes had 22 core genes).
  -> DOWNSTREAM/prokka_refs/todo.tsv   (label, genus, species, fasta, locustag)
--redo lists every reference again, also those already annotated.
Prints the --array range to pass to sbatch.
"""
import csv
import os
import sys
import zlib
from pathlib import Path

META = Path(os.environ["META"])
REFDIR = Path(os.environ["REFDIR"])
DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
OUT = DOWNSTREAM / "prokka_refs"
redo = "--redo" in sys.argv


def read_tsv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


def has_fasta_section(path):
    with open(path) as f:
        return any(line.startswith("##FASTA") for line in f)


refs = []
for r in read_tsv(META / "reference_genomes.tsv"):
    if r["genus"] == "Pectobacterium" and r["role"] == "reference":
        refs.append((r["label"], r["genus"], r["species"], REFDIR / f"{r['label']}.fasta"))
extra = META / "core_references.tsv"
if extra.exists():
    for r in read_tsv(extra):
        if r["genus"] != "Pectobacterium":
            print(f"NOTE: {r['label']} is {r['genus']}, not Pectobacterium: not used in the core tree")
            continue
        if r["role"] == "outgroup":
            print(f"NOTE: {r['label']} has role outgroup: not used (too distant for a core alignment)")
            continue
        refs.append((r["label"], r["genus"], r["species"], REFDIR / "core" / f"{r['label']}.fasta"))
if not refs:
    raise SystemExit("ERROR: no Pectobacterium reference in reference_genomes.tsv / core_references.tsv")
labels = [x[0] for x in refs]
if len(set(labels)) != len(labels):
    raise SystemExit("ERROR: a reference label appears twice: " + ", ".join(sorted({x for x in labels if labels.count(x) > 1})))

OUT.mkdir(parents=True, exist_ok=True)
todo, done, missing = [], [], []
for label, genus, species, fasta in refs:
    if not (fasta.exists() and fasta.stat().st_size > 0):
        missing.append((label, fasta))
        continue
    gff = OUT / label / f"{label}.gff"
    if gff.exists() and has_fasta_section(gff) and not redo:
        done.append(label)
        continue
    tag = "RF" + format(zlib.crc32(label.encode()) & 0xFFFFFF, "06X")      # 8 characters, stable for a label
    todo.append((label, genus, species, str(fasta), tag))

with open(OUT / "todo.tsv", "w") as f:
    f.write("label\tgenus\tspecies\tfasta\tlocustag\n")
    for t in todo:
        f.write("\t".join(t) + "\n")

print(f"{len(refs)} Pectobacterium reference(s): {len(done)} already annotated, {len(todo)} to annotate, {len(missing)} without a FASTA")
for t in todo:
    print(f"  to annotate: {t[0]:34s} {t[1]} {t[2]}")
for label, fasta in missing:
    print(f"  NO FASTA:    {label}  (expected {fasta}) -> run D4b.1 (or A2 for the original references)")
print(f"ARRAY=1-{len(todo)}" if todo else "ARRAY=")
END_OF_SCRIPT
chmod +x "$SCRIPTS/make_refs_todo.py.new" && mv -f "$SCRIPTS/make_refs_todo.py.new" "$SCRIPTS/make_refs_todo.py"
echo "wrote $SCRIPTS/make_refs_todo.py"

cat > "$SCRIPTS/prokka_refs_array.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Prokka for the Pectobacterium reference genomes of the core-gene tree (D4b.2), one reference per SLURM array task.
# Same Prokka 1.12, same ASA3P protein database and same options as the isolates' fallback annotation (C1), so the
# references and the isolates are annotated alike. Contigs are renamed <locustag>_<n> (short IDs; Prokka stops on long ones).
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

TODO="${TODO_FILE:-$DOWNSTREAM/prokka_refs/todo.tsv}"
line=$(awk -v n="$SLURM_ARRAY_TASK_ID" 'NR == n + 1' "$TODO")   # +1 skips the header
if [ -z "$line" ]; then
    echo "No line for array task $SLURM_ARRAY_TASK_ID in $TODO"; exit 1
fi
IFS=$'\t' read -r label genus species fasta locustag <<< "$line"
outdir="$DOWNSTREAM/prokka_refs/${label}"
echo "Prokka (reference): $label ($genus $species), locus tag $locustag"
test -s "$fasta" || { echo "Missing input $fasta"; exit 1; }

mkdir -p "$DOWNSTREAM/prokka_refs/_input"
input="$DOWNSTREAM/prokka_refs/_input/${label}.fna"
awk -v p="$locustag" '/^>/ {n++; print ">" p "_" n; next} {print}' "$fasta" > "$input"

apptainer exec \
    --bind "$ASAPDIR":/asap:ro \
    --bind "$ROOT":"$ROOT" \
    "$PROKKA_IMG" \
    bash -c 'export LC_ALL=C LANG=C
             export PATH=/asap/share/prokka-1.12/binaries/linux:/asap/share/prokka-1.12/bin:/asap/share/barrnap-0.8/bin:$PATH
             exec /asap/share/prokka/bin/prokka "$@"' _ \
        --kingdom Bacteria \
        --genus "$genus" \
        --species "$species" \
        --strain "$label" \
        --prefix "$label" \
        --locustag "$locustag" \
        --centre "$CENTRE" \
        --cpus "${SLURM_CPUS_PER_TASK:-8}" \
        --outdir "$outdir" \
        --proteins /asap/db/sequences/asap-proteins.faa \
        --force \
        --addgenes \
        --rfam \
        --rawproduct \
        --compliant \
        "$input" < /dev/null

ls -lh "$outdir/${label}.gff" "$outdir/${label}.gbk"
END_OF_SCRIPT
chmod +x "$SCRIPTS/prokka_refs_array.sh.new" && mv -f "$SCRIPTS/prokka_refs_array.sh.new" "$SCRIPTS/prokka_refs_array.sh"
echo "wrote $SCRIPTS/prokka_refs_array.sh"

REDO=no          # yes = annotate every reference again

use_env download_env
if [ "$REDO" = yes ]; then out=$(python "$SCRIPTS/make_refs_todo.py" --redo)
else                       out=$(python "$SCRIPTS/make_refs_todo.py"); fi
echo "$out"
ARRAY=$(echo "$out" | sed -n 's/^ARRAY=//p')
if [ -n "$ARRAY" ]; then
    JOB=$(SBATCH_EXTRA="--array=$ARRAY" submit prokka_refs 8 32G 08:00:00 "$SCRIPTS/prokka_refs_array.sh")
    echo "submitted array job $JOB ($ARRAY)"
else
    echo "Nothing to submit: every reference is annotated (or has no FASTA, see above)."
fi

wrote /home/s94m696/asap_shortname_projects/scripts/make_refs_todo.py
wrote /home/s94m696/asap_shortname_projects/scripts/prokka_refs_array.sh
18 Pectobacterium reference(s): 0 already annotated, 18 to annotate, 0 without a FASTA
  to annotate: REF_Patrosepticum_SCRI1043         Pectobacterium atrosepticum
  to annotate: REF_Ppolaris_NIBIO1392             Pectobacterium polaris
  to annotate: REF_Pversatile_CFBP6051            Pectobacterium versatile
  to annotate: REF_Pcarotovorum                   Pectobacterium carotovorum
  to annotate: REF_Pbrasiliense                   Pectobacterium brasiliense
  to annotate: REF_Podoriferum                    Pectobacterium odoriferum
  to annotate: REF_Pactinidiae_KKH3               Pectobacterium actinidiae
  to annotate: REF_Paroidearum_NCPPB929           Pectobacterium aroidearum
  to annotate: REF_Pbetavasculorum_NCPPB2795      Pectobacterium betavasculorum
  to annotate: REF_Pjejuense_13115                Pectobacterium jejuense
  to ann

In [9]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like prokka_refs
find "$DOWNSTREAM/prokka_refs" -name "*.gff" | sort
grep -l -i "error" "$LOGS"/prokka_refs_*.err 2>/dev/null || echo "no errors in the reference logs"

               JobID                        JobName      State ExitCode    Elapsed 
            917034_1                    prokka_refs  COMPLETED      0:0   00:05:03 
            917034_2                    prokka_refs  COMPLETED      0:0   00:04:20 
            917034_3                    prokka_refs  COMPLETED      0:0   00:06:35 
            917034_4                    prokka_refs  COMPLETED      0:0   00:04:15 
            917034_5                    prokka_refs  COMPLETED      0:0   00:04:17 
            917034_6                    prokka_refs  COMPLETED      0:0   00:04:28 
            917034_7                    prokka_refs  COMPLETED      0:0   00:03:54 
            917034_8                    prokka_refs  COMPLETED      0:0   00:04:24 
            917034_9                    prokka_refs  COMPLETED      0:0   00:05:05 
           917034_10                    prokka_refs  COMPLETED      0:0   00:04:11 
           917034_11                    prokka_refs  COMPLETED      0:0   00

In [10]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"
grep -i -h "error" "$LOGS"/prokka_refs_917034_*.err | sort | uniq -c | sort -rn | head -20

      1 [08:52:27] Deleting unwanted file: /home/s94m696/asap_shortname_projects/downstream/prokka_refs/REF_Pversatile_CFBP6051/errorsummary.val
      1 [08:51:15] Deleting unwanted file: /home/s94m696/asap_shortname_projects/downstream/prokka_refs/REF_Pperuviense_IFB5232/errorsummary.val
      1 [08:51:01] Deleting unwanted file: /home/s94m696/asap_shortname_projects/downstream/prokka_refs/REF_Pparvum_LMG30828/errorsummary.val
      1 [08:50:57] Deleting unwanted file: /home/s94m696/asap_shortname_projects/downstream/prokka_refs/REF_Pbetavasculorum_NCPPB2795/errorsummary.val
      1 [08:50:54] Deleting unwanted file: /home/s94m696/asap_shortname_projects/downstream/prokka_refs/REF_Patrosepticum_SCRI1043/errorsummary.val
      1 [08:50:40] Deleting unwanted file: /home/s94m696/asap_shortname_projects/downstream/prokka_refs/REF_Pwasabiae_CFBP3304/errorsummary.val
      1 [08:50:31] Deleting unwanted file: /home/s94m696/asap_shortname_projects/downstream/prokka_refs/REF_Ppolonicum_LMG310

In [8]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"
while squeue -u $USER -h -n prokka_refs | grep -q .; do
    date +%T
    squeue -u $USER -n prokka_refs
    sleep 30
done
echo "no prokka_refs jobs left in the queue"
jobs_like prokka_refs

no prokka_refs jobs left in the queue
               JobID                        JobName      State ExitCode    Elapsed 
            917034_1                    prokka_refs  COMPLETED      0:0   00:05:03 
            917034_2                    prokka_refs  COMPLETED      0:0   00:04:20 
            917034_3                    prokka_refs  COMPLETED      0:0   00:06:35 
            917034_4                    prokka_refs  COMPLETED      0:0   00:04:15 
            917034_5                    prokka_refs  COMPLETED      0:0   00:04:17 
            917034_6                    prokka_refs  COMPLETED      0:0   00:04:28 
            917034_7                    prokka_refs  COMPLETED      0:0   00:03:54 
            917034_8                    prokka_refs  COMPLETED      0:0   00:04:24 
            917034_9                    prokka_refs  COMPLETED      0:0   00:05:05 
           917034_10                    prokka_refs  COMPLETED      0:0   00:04:11 
           917034_11                  

### D4b.3 Choose the genome set
Collects the genomes of the tree: the isolates with `in_analysis = yes` that are *Pectobacterium* (`annotation_files.tsv`, from D2.0), plus every annotated *Pectobacterium* reference. The GFFs are copied to `core_tree/gff/`, so Roary's sample names are the isolate short names and the reference labels. The cell prints the set, who was left out and why, and which isolates are clones (≥ 99.9 % ANI to each other: they are kept, since identical genomes give identical tips, but they add nothing to the core).

`CORE_EXCLUDE` leaves genomes out by name; use it after D4b.5 if one genome costs the core most of its genes. **Re-run D2.0 first after any change to the gate.**

In [11]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/select_core_set.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Choose the genomes of the genus-level core-gene tree and build the Roary input folder (D4b.3).

Included:
  * isolates that passed the species gate (D1.2) and are Pectobacterium (annotation_files.tsv, made by D2.0 -- re-run D2.0
    after any change to the gate). Isolates that failed the gate (no species match, MIXED assemblies) are never used:
    genomes at ~78% ANI share almost no genes with Pectobacterium at the identity Roary clusters at, and one such genome
    is enough to remove nearly the whole core (the first genus-level run: 22 core genes).
  * the Pectobacterium reference genomes annotated in D4b.2 (prokka_refs/).
Environment:
  CORE_EXCLUDE   labels to leave out (space or comma separated), e.g. after the leave-one-out table (D4b.5)
Writes DOWNSTREAM/core_tree/core_set.tsv and DOWNSTREAM/core_tree/gff/<label>.gff (copies, so Roary's sample names are the labels).
Clones (isolates >= 99.9% ANI to each other, from the D1.3 all-vs-all fastANI) are listed but kept: identical genomes
give identical tips, which is harmless for the tree.
"""
import csv
import os
import re
import shutil
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
META = Path(os.environ["META"])
CORE = DOWNSTREAM / "core_tree"
exclude = {x for x in re.split(r"[ ,]+", os.environ.get("CORE_EXCLUDE", "")) if x}


def read_tsv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


def has_fasta_section(path):
    with open(path) as f:
        return any(line.startswith("##FASTA") for line in f)


ann_path = DOWNSTREAM / "annotation_files.tsv"
if not ann_path.exists():
    raise SystemExit("ERROR: annotation_files.tsv is missing -- run D2.0 (after the species gate, D1.2) first.")
if not (DOWNSTREAM / "species_check" / "analysis_set.tsv").exists():
    raise SystemExit("ERROR: no species gate (species_check/analysis_set.tsv) -- run D1 first; without it failed isolates would be used.")

rows, problems, skipped = [], [], []
for a in read_tsv(ann_path):
    s = a["short_name"]
    if a["in_analysis"] != "yes":
        skipped.append((s, "failed the species gate"))
    elif a["genus"] != "Pectobacterium":
        skipped.append((s, f"genus {a['genus']}"))
    elif s in exclude:
        skipped.append((s, "CORE_EXCLUDE"))
    elif not a["gff"]:
        problems.append(f"{s}: no GFF (run the fallback Prokka section, C1)")
    elif not has_fasta_section(a["gff"]):
        problems.append(f"{s}: GFF has no ##FASTA section (annotate it in C1)")
    else:
        rows.append({"label": s, "kind": "isolate", "genus": a["genus"], "species": a["species"], "source": a["source"], "gff": a["gff"]})

ref_tables = [(r["label"], r["genus"], r["species"]) for r in read_tsv(META / "reference_genomes.tsv")
              if r["genus"] == "Pectobacterium" and r["role"] == "reference"]
if (META / "core_references.tsv").exists():
    ref_tables += [(r["label"], r["genus"], r["species"]) for r in read_tsv(META / "core_references.tsv")
                   if r["genus"] == "Pectobacterium" and r["role"] != "outgroup"]
for label, genus, species in ref_tables:
    gff = DOWNSTREAM / "prokka_refs" / label / f"{label}.gff"
    if label in exclude:
        skipped.append((label, "CORE_EXCLUDE"))
    elif not gff.exists() or not has_fasta_section(gff):
        problems.append(f"{label}: not annotated yet (run D4b.2)")
    else:
        rows.append({"label": label, "kind": "reference", "genus": genus, "species": species, "source": "prokka_refs", "gff": str(gff)})

if problems:
    print("Not usable yet:")
    for p in problems:
        print("  " + p)
n_iso = sum(r["kind"] == "isolate" for r in rows)
n_ref = sum(r["kind"] == "reference" for r in rows)
if n_iso < 2 or n_ref < 2 or len(rows) < 4:
    raise SystemExit(f"ERROR: {n_iso} isolate(s) and {n_ref} reference(s) are usable; the core-gene tree needs at least 4 genomes "
                     "(2 isolates and 2 references). Fix the problems above and re-run.")

if (CORE / "gff").exists():
    shutil.rmtree(CORE / "gff")
(CORE / "gff").mkdir(parents=True)
for r in rows:
    shutil.copyfile(r["gff"], CORE / "gff" / f"{r['label']}.gff")
with open(CORE / "core_set.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["label", "kind", "genus", "species", "source", "gff"], delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)

print(f"\n{len(rows)} genomes for the core-gene tree: {n_iso} isolates + {n_ref} references\n")
print(f"{'label':36s} {'kind':10s} species")
for r in sorted(rows, key=lambda r: (r["kind"], r["species"], r["label"])):
    print(f"{r['label']:36s} {r['kind']:10s} {r['species']}")
print("\nLeft out:")
for s, why in skipped:
    print(f"  {s:12s} {why}")

# clones among the isolates (ANI >= 99.9 in at least one direction)
fa = DOWNSTREAM / "species_check" / "fastani_allvsall.tsv"
if fa.exists():
    names = {r["label"] for r in rows if r["kind"] == "isolate"}
    parent = {n: n for n in names}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    with open(fa) as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) < 3:
                continue
            q, r_ = Path(p[0]).name.rsplit(".", 1)[0], Path(p[1]).name.rsplit(".", 1)[0]
            if q in names and r_ in names and q != r_ and float(p[2]) >= 99.9:
                parent[find(q)] = find(r_)
    groups = {}
    for n in names:
        groups.setdefault(find(n), []).append(n)
    clones = [sorted(g) for g in groups.values() if len(g) > 1]
    print(f"\nDistinct isolate genomes at 99.9% ANI: {len(groups)} of {len(names)}")
    for g in sorted(clones):
        print("  clones: " + ", ".join(g))
print(f"\nWrote {CORE/'core_set.tsv'} and {CORE/'gff'}/")
END_OF_SCRIPT
chmod +x "$SCRIPTS/select_core_set.py.new" && mv -f "$SCRIPTS/select_core_set.py.new" "$SCRIPTS/select_core_set.py"
echo "wrote $SCRIPTS/select_core_set.py"

export CORE_EXCLUDE=""          # e.g. CORE_EXCLUDE="PI012 REF_Pxxx_YYY" (space-separated)
use_env download_env
python "$SCRIPTS/select_core_set.py"

wrote /home/s94m696/asap_shortname_projects/scripts/select_core_set.py

30 genomes for the core-gene tree: 12 isolates + 18 references

label                                kind       species
PI004                                isolate    atrosepticum
PI039                                isolate    atrosepticum
PI027                                isolate    carotovorum
PI009                                isolate    polaris
PI019                                isolate    versatile
PI020                                isolate    versatile
PI022                                isolate    versatile
PI025                                isolate    versatile
PI041                                isolate    versatile
PI043                                isolate    versatile
PI044                                isolate    versatile
PI051                                isolate    versatile
REF_Pactinidiae_KKH3                 reference  actinidiae
REF_Paroidearum_NCPPB929             reference 

### D4b.4 Roary on the set
`-i 90` (the BLASTp identity below which two proteins are put in different groups) and `-cd 99` (a core gene is in ≥ 99 % of the genomes, so in all of them below 100 genomes); `-e -n` aligns the core genes with MAFFT. Roary's default identity is 95; a lower value is the usual choice for a set spanning several species, because orthologs of species that are further apart are then not split. A lower value also merges more paralogs, so for a comparison set `CORE_IDENT=95` and look at both. The output folder is deleted first and not created, because Roary writes to `<folder>_<timestamp>` when it already exists (the source of `roary_output_container_1778273395`).

In [12]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_roary_core.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Roary on the genomes of the genus-level core-gene tree (D4b.4): the clean Pectobacterium isolates + the annotated
# Pectobacterium reference genomes (core_set.tsv, GFFs in core_tree/gff/).
#   CORE_IDENT  minimum BLASTp identity, default 90. Roary's default is 95; a lower value is the usual choice for a set that
#               spans several species, because orthologs of species that are further apart are then not split into separate
#               groups. A lower value also merges more paralogs; CORE_IDENT=95 gives the strict version for comparison.
#   CORE_CD     core threshold in percent of genomes, default 99 (Roary's default: in all genomes while there are < 100).
# Core genes are aligned with MAFFT (-e -n), as in the per-species runs. The output folder is deleted and NOT created
# first: if -f points at an existing folder Roary writes to <folder>_<timestamp> instead (see run_roary_groups.sh).
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

CORE="$DOWNSTREAM/core_tree"
IDENT="${CORE_IDENT:-90}"
CD="${CORE_CD:-99}"
CPUS="${SLURM_CPUS_PER_TASK:-8}"
out="$CORE/roary_output"

mapfile -t gffs < <(find "$CORE/gff" -maxdepth 1 -name '*.gff' 2>/dev/null | sort)
if [ "${#gffs[@]}" -lt 4 ]; then
    echo "ERROR: ${#gffs[@]} GFF file(s) in $CORE/gff -- run D4b.3 first (needs at least 4)"; exit 1
fi
echo "Roary: ${#gffs[@]} genomes, -i $IDENT, core threshold $CD%"
printf '  %s\n' "${gffs[@]##*/}"
rm -rf "$out"
if apptainer exec --bind "$ROOT":"$ROOT" "$ROARY_IMG" \
       roary -p "$CPUS" -f "$out" -e -n -i "$IDENT" -cd "$CD" "${gffs[@]}" < /dev/null; then
    printf 'identity\t%s\ncore_threshold\t%s\ngenomes\t%s\n' "$IDENT" "$CD" "${#gffs[@]}" > "$CORE/roary_settings.tsv"
    echo "done: $out"
    cat "$out/summary_statistics.txt"
    core=$(awk -F'\t' '/^Core genes/ {print $3}' "$out/summary_statistics.txt")
    if [ "${core:-0}" -lt 500 ]; then
        echo "WARNING: only ${core:-0} core genes, far fewer than one genus should have. Run D4b.5: it shows which genomes lack the common genes."
    fi
else
    echo "ERROR: Roary failed"; exit 1
fi
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_roary_core.sh.new" && mv -f "$SCRIPTS/run_roary_core.sh.new" "$SCRIPTS/run_roary_core.sh"
echo "wrote $SCRIPTS/run_roary_core.sh"

export CORE_IDENT=90      # minimum BLASTp identity
export CORE_CD=99         # core = present in at least this % of the genomes
JOB=$(submit roary_core 16 64G 1-00:00:00 "$SCRIPTS/run_roary_core.sh"); echo "submitted $JOB"

wrote /home/s94m696/asap_shortname_projects/scripts/run_roary_core.sh
submitted 917675


In [14]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"
while squeue -u $USER -h -n roary_core | grep -q .; do
    date +%T
    squeue -u $USER -n roary_core
    sleep 60
done
echo "roary_core is no longer in the queue"
jobs_like roary_core
tail -n 15 "$LOGS/roary_core_917675.out"

roary_core is no longer in the queue
               JobID                        JobName      State ExitCode    Elapsed 
              917675                     roary_core  COMPLETED      0:0   00:40:21 
  REF_Pwasabiae_CFBP3304.gff
  REF_Pzantedeschiae_9M.gff

Please cite Roary if you use any of the results it produces:
    Andrew J. Page, Carla A. Cummins, Martin Hunt, Vanessa K. Wong, Sandra Reuter, Matthew T. G. Holden, Maria Fookes, Daniel Falush, Jacqueline A. Keane, Julian Parkhill,
	"Roary: Rapid large-scale prokaryote pan genome analysis", Bioinformatics, 2015 Nov 15;31(22):3691-3693
    doi: http://doi.org/10.1093/bioinformatics/btv421
	Pubmed: 26198102

done: /home/s94m696/asap_shortname_projects/downstream/core_tree/roary_output
Core genes	(99% <= strains <= 100%)	2428
Soft core genes	(95% <= strains < 99%)	261
Shell genes	(15% <= strains < 95%)	2551
Cloud genes	(0% <= strains < 15%)	11389
Total genes	(0% <= strains <= 100%)	16629


### D4b.5 Check the core (and find out what broke the old one)
Run this when the Roary job has finished. It prints Roary's summary and then the **leave-one-out table**: for every genome, how many core genes there would be without it, how many genes it alone lacks, and how many common genes (present in at least half of the genomes) it lacks. If one genome lacks most of the common genes, it is not like the others: put its label in `CORE_EXCLUDE` (D4b.3) and repeat D4b.3–D4b.4. The second table removes genomes one at a time (always the one that lacks the most common genes) and shows the core after each step, for the case where several genomes are to blame and no single removal helps.

The second cell does the same for **ASA³P's own genus-level run** (the one with 22 core genes), so you can see which genomes caused it. It is a diagnostic only; nothing in the cell changes that run.

In [15]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/core_gene_loo.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Which genomes keep a pan-genome from having a core? (leave-one-out on a Roary gene_presence_absence.Rtab)

For every genome: how many core genes there would be without it, how many genes are present in ALL other genomes but
missing in this one, and how many COMMON genes (present in at least half of the genomes) it lacks. One outlier genome is
often not the whole story: with several outliers, removing any single genome gains nothing. So the second table removes
genomes one at a time, always the one that lacks the most common genes, and shows the core after each removal, until the
core reaches --target genes (default 1000) or only 4 genomes are left.
Core = present in at least 99% of the genomes (Roary's default), i.e. in all of them while there are fewer than 100 genomes.

  core_gene_loo.py <gene_presence_absence.Rtab | roary output folder> [--target 1000] [--core 0.99] [--out core_loo.tsv]

Run it on the old genus-level run to see the cause of its 22 core genes, and on the new core-tree run to check it.
Needs only the Python standard library.
"""
import argparse
import csv
import math
import sys
from pathlib import Path

ap = argparse.ArgumentParser()
ap.add_argument("rtab")
ap.add_argument("--target", type=int, default=1000)
ap.add_argument("--core", type=float, default=0.99)
ap.add_argument("--out", default="")
a = ap.parse_args()

p = Path(a.rtab)
if p.is_dir():
    p = p / "gene_presence_absence.Rtab"
if not p.exists():
    sys.exit(f"ERROR: {p} not found")
with open(p) as f:
    rd = csv.reader(f, delimiter="\t")
    header = next(rd)
    genomes = header[1:]
    genes = [[int(x) for x in row[1:]] for row in rd if row]
n = len(genomes)
if n < 4:
    sys.exit(f"ERROR: only {n} genomes in {p}")


def need(m):                     # genes present in at least this many of m genomes are core
    return max(1, math.ceil(a.core * m - 1e-9))


def core_count(cols):
    k = need(len(cols))
    return sum(1 for g in genes if sum(g[c] for c in cols) >= k)


all_cols = list(range(n))
core_all = core_count(all_cols)
print(f"{p}\n{n} genomes, {len(genes)} gene groups, core genes (>= {a.core:.0%}): {core_all}\n")

COMMON = 0.5


def lacks_common(cols):
    """for each genome in cols: number of genes present in >= 50% of cols that the genome lacks"""
    k = math.ceil(COMMON * len(cols) - 1e-9)
    lack = {c: 0 for c in cols}
    for g in genes:
        if sum(g[c] for c in cols) >= k:
            for c in cols:
                if g[c] == 0:
                    lack[c] += 1
    return lack


lack_all = lacks_common(all_cols)
rows = []
for c in all_cols:
    rest = [x for x in all_cols if x != c]
    rows.append({"genome": genomes[c], "genes_present": sum(g[c] for g in genes),
                 "core_without_it": core_count(rest),
                 "genes_only_it_lacks": sum(1 for g in genes if g[c] == 0 and all(g[x] == 1 for x in rest)),
                 "common_genes_it_lacks": lack_all[c]})
for r in rows:
    r["core_gain"] = r["core_without_it"] - core_all
rows.sort(key=lambda r: (-r["common_genes_it_lacks"], -r["core_gain"], r["genome"]))
print(f"{'genome':30s} {'genes':>7s} {'core w/o it':>12s} {'gain':>6s} {'only it lacks':>14s} {'lacks common':>13s}")
for r in rows:
    print(f"{r['genome']:30s} {r['genes_present']:7d} {r['core_without_it']:12d} {r['core_gain']:6d} {r['genes_only_it_lacks']:14d} {r['common_genes_it_lacks']:13d}")

# greedy removal
cols, order = list(all_cols), []
cur = core_all
print(f"\nRemoving the genome that lacks the most common genes, one at a time (stop at {a.target} core genes or 4 genomes):")
while cur < a.target and len(cols) > 4:
    lack = lacks_common(cols)
    best = max(cols, key=lambda c: (lack[c], -sum(g[c] for g in genes), -c))
    cols = [x for x in cols if x != best]
    cur = core_count(cols)
    order.append((genomes[best], cur))
    print(f"  remove {genomes[best]:28s} -> {len(cols)} genomes, {cur} core genes")
if cur >= a.target:
    print(f"\nRemoving {len(order)} genome(s) gives {cur} core genes: " + ", ".join(g for g, _ in order))
else:
    print(f"\nOnly {cur} core genes are left when {len(cols)} genomes remain: this set has no core worth a tree.")

out = Path(a.out) if a.out else p.parent / "core_loo.tsv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["genome", "genes_present", "core_without_it", "core_gain", "genes_only_it_lacks", "common_genes_it_lacks"], delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)
print(f"\nWrote {out}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/core_gene_loo.py.new" && mv -f "$SCRIPTS/core_gene_loo.py.new" "$SCRIPTS/core_gene_loo.py"
echo "wrote $SCRIPTS/core_gene_loo.py"

jobs_like roary_core
echo; cat "$DOWNSTREAM/core_tree/roary_output/summary_statistics.txt"
use_env download_env
echo; echo "=== core tree run: leave-one-out ==="
python "$SCRIPTS/core_gene_loo.py" "$DOWNSTREAM/core_tree/roary_output" --out "$DOWNSTREAM/core_tree/core_loo.tsv"

wrote /home/s94m696/asap_shortname_projects/scripts/core_gene_loo.py
               JobID                        JobName      State ExitCode    Elapsed 
              917675                     roary_core  COMPLETED      0:0   00:40:21 

Core genes	(99% <= strains <= 100%)	2428
Soft core genes	(95% <= strains < 99%)	261
Shell genes	(15% <= strains < 95%)	2551
Cloud genes	(0% <= strains < 15%)	11389
Total genes	(0% <= strains <= 100%)	16629

=== core tree run: leave-one-out ===
/home/s94m696/asap_shortname_projects/downstream/core_tree/roary_output/gene_presence_absence.Rtab
30 genomes, 16629 gene groups, core genes (>= 99%): 2428

genome                           genes  core w/o it   gain  only it lacks  lacks common
REF_Pbetavasculorum_NCPPB2795     4212         2511     83             83           649
REF_Pparmentieri_RNS08421A        4528         2443     15             15           602
REF_Pwasabiae_CFBP3304            4627         2465     37             37           598
REF_Ppunj

In [16]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/core_gene_loo.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Which genomes keep a pan-genome from having a core? (leave-one-out on a Roary gene_presence_absence.Rtab)

For every genome: how many core genes there would be without it, how many genes are present in ALL other genomes but
missing in this one, and how many COMMON genes (present in at least half of the genomes) it lacks. One outlier genome is
often not the whole story: with several outliers, removing any single genome gains nothing. So the second table removes
genomes one at a time, always the one that lacks the most common genes, and shows the core after each removal, until the
core reaches --target genes (default 1000) or only 4 genomes are left.
Core = present in at least 99% of the genomes (Roary's default), i.e. in all of them while there are fewer than 100 genomes.

  core_gene_loo.py <gene_presence_absence.Rtab | roary output folder> [--target 1000] [--core 0.99] [--out core_loo.tsv]

Run it on the old genus-level run to see the cause of its 22 core genes, and on the new core-tree run to check it.
Needs only the Python standard library.
"""
import argparse
import csv
import math
import sys
from pathlib import Path

ap = argparse.ArgumentParser()
ap.add_argument("rtab")
ap.add_argument("--target", type=int, default=1000)
ap.add_argument("--core", type=float, default=0.99)
ap.add_argument("--out", default="")
a = ap.parse_args()

p = Path(a.rtab)
if p.is_dir():
    p = p / "gene_presence_absence.Rtab"
if not p.exists():
    sys.exit(f"ERROR: {p} not found")
with open(p) as f:
    rd = csv.reader(f, delimiter="\t")
    header = next(rd)
    genomes = header[1:]
    genes = [[int(x) for x in row[1:]] for row in rd if row]
n = len(genomes)
if n < 4:
    sys.exit(f"ERROR: only {n} genomes in {p}")


def need(m):                     # genes present in at least this many of m genomes are core
    return max(1, math.ceil(a.core * m - 1e-9))


def core_count(cols):
    k = need(len(cols))
    return sum(1 for g in genes if sum(g[c] for c in cols) >= k)


all_cols = list(range(n))
core_all = core_count(all_cols)
print(f"{p}\n{n} genomes, {len(genes)} gene groups, core genes (>= {a.core:.0%}): {core_all}\n")

COMMON = 0.5


def lacks_common(cols):
    """for each genome in cols: number of genes present in >= 50% of cols that the genome lacks"""
    k = math.ceil(COMMON * len(cols) - 1e-9)
    lack = {c: 0 for c in cols}
    for g in genes:
        if sum(g[c] for c in cols) >= k:
            for c in cols:
                if g[c] == 0:
                    lack[c] += 1
    return lack


lack_all = lacks_common(all_cols)
rows = []
for c in all_cols:
    rest = [x for x in all_cols if x != c]
    rows.append({"genome": genomes[c], "genes_present": sum(g[c] for g in genes),
                 "core_without_it": core_count(rest),
                 "genes_only_it_lacks": sum(1 for g in genes if g[c] == 0 and all(g[x] == 1 for x in rest)),
                 "common_genes_it_lacks": lack_all[c]})
for r in rows:
    r["core_gain"] = r["core_without_it"] - core_all
rows.sort(key=lambda r: (-r["common_genes_it_lacks"], -r["core_gain"], r["genome"]))
print(f"{'genome':30s} {'genes':>7s} {'core w/o it':>12s} {'gain':>6s} {'only it lacks':>14s} {'lacks common':>13s}")
for r in rows:
    print(f"{r['genome']:30s} {r['genes_present']:7d} {r['core_without_it']:12d} {r['core_gain']:6d} {r['genes_only_it_lacks']:14d} {r['common_genes_it_lacks']:13d}")

# greedy removal
cols, order = list(all_cols), []
cur = core_all
print(f"\nRemoving the genome that lacks the most common genes, one at a time (stop at {a.target} core genes or 4 genomes):")
while cur < a.target and len(cols) > 4:
    lack = lacks_common(cols)
    best = max(cols, key=lambda c: (lack[c], -sum(g[c] for g in genes), -c))
    cols = [x for x in cols if x != best]
    cur = core_count(cols)
    order.append((genomes[best], cur))
    print(f"  remove {genomes[best]:28s} -> {len(cols)} genomes, {cur} core genes")
if cur >= a.target:
    print(f"\nRemoving {len(order)} genome(s) gives {cur} core genes: " + ", ".join(g for g, _ in order))
else:
    print(f"\nOnly {cur} core genes are left when {len(cols)} genomes remain: this set has no core worth a tree.")

out = Path(a.out) if a.out else p.parent / "core_loo.tsv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["genome", "genes_present", "core_without_it", "core_gain", "genes_only_it_lacks", "common_genes_it_lacks"], delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)
print(f"\nWrote {out}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/core_gene_loo.py.new" && mv -f "$SCRIPTS/core_gene_loo.py.new" "$SCRIPTS/core_gene_loo.py"
echo "wrote $SCRIPTS/core_gene_loo.py"

use_env download_env
old=$(ls -dt "$(project_dir Pectobacterium)"/corepan/roary_output* 2>/dev/null | head -1)
if [ -z "$old" ] || [ ! -s "$old/gene_presence_absence.Rtab" ]; then
    echo "No ASA3P genus-level Roary run found in $(project_dir Pectobacterium)/corepan (nothing to diagnose)."
else
    echo "=== ASA3P genus-level run: $old ==="
    cat "$old/summary_statistics.txt"
    python "$SCRIPTS/core_gene_loo.py" "$old" --out "$DOWNSTREAM/core_tree/core_loo_ASA3P_project.tsv"
fi

wrote /home/s94m696/asap_shortname_projects/scripts/core_gene_loo.py
No ASA3P genus-level Roary run found in /home/s94m696/asap_shortname_projects/ASA3P_Pectobacterium/corepan (nothing to diagnose).


### D4b.6 IQ-TREE on the core-gene alignment
ModelFinder restricted to the GTR family (`CORE_MSET=ALL` tests every model; it takes hours longer on an alignment of this size and nearly always ends on a GTR model anyway), 1,000 ultrafast bootstrap replicates (with `-bnni`, so that model violations do not inflate them) and 1,000 SH-aLRT replicates. A second run adds **site concordance factors** (sCF, 100 quartets per branch): the share of alignment sites that support a branch. A branch with high UFBoot but low sCF is supported by few sites, which is common on long alignments, so look at both before drawing conclusions from a deep branch.

The tree is **unrooted**. *Dickeya* and *Erwinia* are not in the alignment; root it at the midpoint, or on a clade you can justify, and say which in the legend. The labels of `core_gene_tree.nwk` are `SH-aLRT/UFBoot`; `core_gene_tree_scf.nwk` carries the concordance factors.

In [17]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/write_core_info.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Collect the core-gene tree and its settings in DOWNSTREAM/trees/ (end of D4b.6).
  core_gene_tree.nwk        IQ-TREE tree; node labels are SH-aLRT/UFBoot (e.g. 97.2/100), unrooted
  core_gene_tree_scf.nwk    the same tree with the site concordance factors as node labels (if the sCF run worked)
  core_gene_tree.info       settings and counts, read by the Methods draft (E2)
Standard library only (runs in download_env or without any environment).
"""
import csv
import os
import re
import shutil
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
CORE = DOWNSTREAM / "core_tree"
TREES = DOWNSTREAM / "trees"
TREES.mkdir(parents=True, exist_ok=True)
tdir = CORE / "iqtree"


def read_text(p):
    return Path(p).read_text(errors="replace") if Path(p).exists() else ""


info = {}
rows = list(csv.DictReader(open(CORE / "core_set.tsv", newline=""), delimiter="\t"))
info["genomes"] = len(rows)
info["isolates"] = sum(r["kind"] == "isolate" for r in rows)
info["references"] = sum(r["kind"] == "reference" for r in rows)
for line in read_text(CORE / "roary_settings.tsv").splitlines():
    k, _, v = line.partition("\t")
    if k:
        info["roary_" + k] = v
m = re.search(r"^Core genes\t(?:[^\t]*\t)?(\d+)\s*$", read_text(CORE / "roary_output" / "summary_statistics.txt"), re.M)
if m:
    info["core_genes"] = m.group(1)
m = re.search(r"^Total genes\t(?:[^\t]*\t)?(\d+)\s*$", read_text(CORE / "roary_output" / "summary_statistics.txt"), re.M)
if m:
    info["total_genes"] = m.group(1)
aln = tdir / "core.aln"
if aln.exists():
    n, length = 0, 0
    with open(aln) as f:
        for line in f:
            if line.startswith(">"):
                n += 1
            elif n == 1:
                length += len(line.strip())
    info["alignment_length_bp"] = length
iq = read_text(tdir / "core.iqtree")
m = re.search(r"Best-fit model according to BIC:\s*(\S+)", iq)
if m:
    info["model"] = m.group(1)
m = re.search(r"IQ-TREE.*?version\s+([0-9.]+)", read_text(tdir / "core.log"))
if m:
    info["iqtree"] = m.group(1)
info["ufboot_replicates"] = 1000
info["shalrt_replicates"] = 1000

if not (tdir / "core.treefile").exists() or (tdir / "core.treefile").stat().st_size == 0:
    raise SystemExit("ERROR: core.treefile not found -- the IQ-TREE run did not finish")
shutil.copyfile(tdir / "core.treefile", TREES / "core_gene_tree.nwk")
if (tdir / "core_scf.cf.tree").exists():
    shutil.copyfile(tdir / "core_scf.cf.tree", TREES / "core_gene_tree_scf.nwk")
    info["scf"] = "yes"
with open(TREES / "core_gene_tree.info", "w") as f:
    for k, v in info.items():
        f.write(f"{k}\t{v}\n")
print("Wrote", TREES / "core_gene_tree.nwk")
for k, v in info.items():
    print(f"  {k:20s} {v}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/write_core_info.py.new" && mv -f "$SCRIPTS/write_core_info.py.new" "$SCRIPTS/write_core_info.py"
echo "wrote $SCRIPTS/write_core_info.py"

cat > "$SCRIPTS/run_iqtree_core.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# IQ-TREE on the core-gene alignment of the genus-level core tree (D4b.6).
#   model        ModelFinder restricted to the GTR family (+I, +G, +R rate variants); the full model set on a ~3 Mb alignment
#                costs hours and the best model for core genes of closely related genomes is nearly always GTR+...
#                (CORE_MSET overrides, e.g. CORE_MSET=ALL for the full set)
#   support      1000 ultrafast bootstrap (UFBoot2, with -bnni so that model violations do not inflate it) and 1000 SH-aLRT
#   concordance  site concordance factors (sCF, 100 quartets per branch) on the final tree: the share of alignment
#                sites that support each branch; a branch with high UFBoot but low sCF is supported by few sites.
# The tree is unrooted. Dickeya and Erwinia are too distant to share a core with these genomes, so they are not in the
# alignment; root it at the midpoint or on a clade you can justify (the Mash tree of D3 roots on Erwinia).
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"
module load "$IQTREE_MODULE"
IQ=$(command -v iqtree2 || command -v iqtree) || { echo "IQ-TREE not found"; exit 1; }

CORE="$DOWNSTREAM/core_tree"
aln="$CORE/roary_output/core_gene_alignment.aln"
[ -s "$aln" ] || { echo "ERROR: $aln missing -- run D4b.4 first"; exit 1; }
nseq=$(grep -c '^>' "$aln")
[ "$nseq" -ge 4 ] || { echo "ERROR: only $nseq sequences"; exit 1; }

MSET="${CORE_MSET:-GTR}"
tdir="$CORE/iqtree"
mkdir -p "$tdir"
cp "$aln" "$tdir/core.aln"
cd "$tdir" || exit 1
mset=()
[ "$MSET" = "ALL" ] || mset=(-mset "$MSET")
"$IQ" -s core.aln -m MFP "${mset[@]}" -B 1000 -alrt 1000 -bnni -T AUTO --prefix core --redo || { echo "ERROR: IQ-TREE failed"; exit 1; }
"$IQ" -s core.aln -t core.treefile --scf 100 -T AUTO --prefix core_scf --redo \
    || echo "WARNING: the concordance-factor run failed; the tree without sCF is fine"

# the job has no python of its own: the helper runs in download_env (stdlib only)
( use_env download_env && python "$SCRIPTS/write_core_info.py" )
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_iqtree_core.sh.new" && mv -f "$SCRIPTS/run_iqtree_core.sh.new" "$SCRIPTS/run_iqtree_core.sh"
echo "wrote $SCRIPTS/run_iqtree_core.sh"

export CORE_MSET=GTR      # ModelFinder candidate set; ALL = every model (hours longer)
JOB=$(submit iqtree_core 16 64G 2-00:00:00 "$SCRIPTS/run_iqtree_core.sh"); echo "submitted $JOB"

wrote /home/s94m696/asap_shortname_projects/scripts/write_core_info.py
wrote /home/s94m696/asap_shortname_projects/scripts/run_iqtree_core.sh
submitted 918415


In [1]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"
while squeue -u $USER -h -n iqtree_core | grep -q .; do
    date +%T
    squeue -u $USER -n iqtree_core
    sleep 60
done
echo "iqtree_core is no longer in the queue"
jobs_like iqtree_core
tail -n 25 "$LOGS/iqtree_core_918415.out"

iqtree_core is no longer in the queue
               JobID                        JobName      State ExitCode    Elapsed 
              918415                    iqtree_core  COMPLETED      0:0   08:55:34 
IQ-TREE multicore version 2.2.2.6 COVID-edition for Linux 64-bit built Nov 11 2023
Developed by Bui Quang Minh, James Barbetti, Nguyen Lam Tung,
Olga Chernomor, Heiko Schmidt, Dominik Schrempf, Michael Woodhams, Ly Trong Nhan.

Host:    epyc001 (AVX2, FMA3, 1007 GB RAM)
Command: /mnt/shared/moduleapps/eb/IQ-TREE/2.2.2.6-gompi-2022b/bin/iqtree2 -s core.aln -t core.treefile --scf 100 -T AUTO --prefix core_scf --redo
Seed:    266692 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Tue Oct 27 18:58:54 2026
Kernel:  AVX+FMA - auto-detect threads (16 CPU cores detected)

Wrote /home/s94m696/asap_shortname_projects/downstream/trees/core_gene_tree.nwk
  genomes              30
  isolates             12
  references           18
  roary_identity       90
  roary_core_thresho

In [2]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like iqtree_core
cat "$DOWNSTREAM/trees/core_gene_tree.info" 2>/dev/null || echo "no core-gene tree yet (see the iqtree_core log in $LOGS)"
ls -lh "$DOWNSTREAM"/trees/core_gene_tree* 2>/dev/null || true

               JobID                        JobName      State ExitCode    Elapsed 
              918415                    iqtree_core  COMPLETED      0:0   08:55:34 
genomes	30
isolates	12
references	18
roary_identity	90
roary_core_threshold	99
roary_genomes	30
core_genes	2428
total_genes	16629
alignment_length_bp	2431676
model	GTR+F+I+R9
iqtree	2.2.2.6
ufboot_replicates	1000
shalrt_replicates	1000
-rw-r--r-- 1 s94m696 tempest-users  236 Oct 27 18:58 /home/s94m696/asap_shortname_projects/downstream/trees/core_gene_tree.info
-rw-r--r-- 1 s94m696 tempest-users 1.5K Oct 27 18:58 /home/s94m696/asap_shortname_projects/downstream/trees/core_gene_tree.nwk


## D4c. Which genus are the isolates that failed the species check? (optional, GTDB-Tk)
fastANI says only that these genomes are about 78–79 % ANI from every *Pectobacterium* and *Dickeya* reference. **GTDB-Tk** places each genome in the Genome Taxonomy Database reference tree (120 marker genes) and returns its genus, and its species where the database has a close genome, without needing a reference list: the answer to "what are they?" that the Mash tree cannot give. The isolates that passed the species check are classified too, as a control: each should come back with the genus of its label.

**Cost.** The database is a one-time download of about 57 GB (compressed; it is larger unpacked, so the setup cell prints the free space first), and the placement step needs a lot of memory. Skip D4c if you only need the *Pectobacterium* core tree. GTDB names are not NCBI names (GTDB splits and renames some genera and species); report the GTDB genus as "GTDB genus".

* **D4c.1** creates `gtdbtk_env` (bioconda `gtdbtk`, newest version) and downloads the database (`GTDBTK_DB`, default `software/gtdbtk_db`). It is a job because of the download size. `GTDBTK_DB_URL` can point at a specific release if the newest data needs a newer GTDB-Tk than conda offers.
* **D4c.2** classifies every isolate (placement only, `--skip_ani_screen`), one job.
* **D4c.3** shows the table `gtdbtk/gtdbtk_classification.tsv`: GTDB genus and species per isolate, next to the gate status and fastANI.

In [3]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/setup_gtdbtk.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# One-time setup for the optional genus identification of the isolates that failed the species check (D4c.1).
# Run as a submitted job: the GTDB reference package is a ~57 GB download (the "latest" release at the time of writing).
#   * conda environment gtdbtk_env (GTDB-Tk from bioconda; the newest version, which matches the newest GTDB data)
#   * GTDB-Tk reference data in $GTDBTK_DB (default: $SOFT/gtdbtk_db). Needs about 2 x 57 GB while it unpacks.
# Safe to run again: a finished environment and database are left alone; an interrupted download resumes (curl -C -).
# GTDBTK_DB_URL overrides the download address, e.g. a fixed release if the newest one needs a newer GTDB-Tk than conda has:
#   https://data.gtdb.ecogenomic.org/releases/latest/auxillary_files/gtdbtk_package/full_package/gtdbtk_data.tar.gz
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"
GTDBTK_DB="${GTDBTK_DB:-$SOFT/gtdbtk_db}"
GTDBTK_DB_URL="${GTDBTK_DB_URL:-https://data.gtdb.ecogenomic.org/releases/latest/auxillary_files/gtdbtk_package/full_package/gtdbtk_data.tar.gz}"
mkdir -p "$SOFT" "$GTDBTK_DB" "$LOGS"
module load "$CONDA_MODULE"
set +u; source "$(conda info --base)/etc/profile.d/conda.sh"; set -u

if conda env list | awk '{print $1}' | grep -qx gtdbtk_env && ( use_env gtdbtk_env && command -v gtdbtk >/dev/null ); then
    echo "[env] gtdbtk_env exists"
else
    conda env remove -y -n gtdbtk_env >/dev/null 2>&1
    echo "[env] creating gtdbtk_env"
    conda create -y -n gtdbtk_env -c conda-forge -c bioconda gtdbtk || { echo "ERROR: could not create gtdbtk_env"; exit 1; }
    ( use_env gtdbtk_env && command -v gtdbtk >/dev/null ) || { echo "ERROR: gtdbtk not found in gtdbtk_env"; exit 1; }
fi
( use_env gtdbtk_env && gtdbtk --version 2>&1 | head -2 )

if [ -s "$GTDBTK_DB/metadata/metadata.txt" ]; then
    echo "[db] already in $GTDBTK_DB:"; cat "$GTDBTK_DB/metadata/metadata.txt"
    exit 0
fi
tarball="$GTDBTK_DB/gtdbtk_data.tar.gz"
echo "[db] downloading $GTDBTK_DB_URL"
for i in 1 2 3 4 5; do
    curl -L -C - --fail --retry 3 --retry-delay 30 -o "$tarball" "$GTDBTK_DB_URL" && break
    echo "[db] attempt $i failed"; sleep $(( 60 * i ))
done
[ -s "$tarball" ] || { echo "ERROR: download failed"; exit 1; }
gzip -t "$tarball" || { echo "ERROR: $tarball is damaged (partial download); delete it and run again"; exit 1; }
echo "[db] unpacking"
tar -xzf "$tarball" -C "$GTDBTK_DB" --strip-components=1 || { echo "ERROR: unpacking failed"; exit 1; }
if [ -s "$GTDBTK_DB/metadata/metadata.txt" ]; then
    rm -f "$tarball"
    echo "[db] ready:"; cat "$GTDBTK_DB/metadata/metadata.txt"
else
    echo "ERROR: no metadata/metadata.txt after unpacking; look at $GTDBTK_DB (the tarball is kept)"; ls "$GTDBTK_DB"; exit 1
fi
END_OF_SCRIPT
chmod +x "$SCRIPTS/setup_gtdbtk.sh.new" && mv -f "$SCRIPTS/setup_gtdbtk.sh.new" "$SCRIPTS/setup_gtdbtk.sh"
echo "wrote $SCRIPTS/setup_gtdbtk.sh"

df -h "$SOFT" | tail -1
# export GTDBTK_DB=...        # where the database goes (default: $SOFT/gtdbtk_db)
JOB=$(submit gtdbtk_setup 4 16G 2-00:00:00 "$SCRIPTS/setup_gtdbtk.sh"); echo "submitted $JOB"

wrote /home/s94m696/asap_shortname_projects/scripts/setup_gtdbtk.sh
10.226.112.1:/home/s94m696  3.0T  2.2T  837G  73% /home/s94m696
submitted 930293


In [11]:
%%bash
source /home/s94m696/asap_shortname_projects/config.sh
squeue -u "$USER" -j 930293 -o "%.10i %.14j %.3t %.10M %.10l %R"
jobs_like gtdbtk_setup
tail -n 5 "$LOGS"/gtdbtk_setup_930293.out
du -sh "$SOFT/gtdbtk_db" 2>/dev/null

slurm_load_jobs error: Invalid job id specified


               JobID                        JobName      State ExitCode    Elapsed 
              930293                   gtdbtk_setup  COMPLETED      0:0   02:26:50 
[db] ready:
RED_DIST_BAC_DICT={"d__": 0.00, "p__":0.3399759686386926,"c__":0.4764466993167481,"o__":0.6467334973890564,"f__":0.7972969419385083,"g__":0.9438360803986985}
RED_DIST_ARC_DICT={"d__":0.00,"p__":0.2859284763413424,"c__":0.42852344844350987,"o__":0.5692329378979698,"f__":0.758762614031466,"g__":0.9250910192387802}
VERSION_DATA=r232

94G	/home/s94m696/asap_shortname_projects/software/gtdbtk_db


In [12]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/summarize_gtdbtk.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Merge the GTDB-Tk classification with the species gate (end of D4c.2 / D4c.3).
Reads DOWNSTREAM/gtdbtk/classify/gtdbtk.{bac120,ar53}.summary.tsv (+ gtdbtk.failed_genomes.tsv if present) and the gate,
writes DOWNSTREAM/gtdbtk/gtdbtk_classification.tsv:
  short_name, initial_label, qc_status, include, best_ani_fastani, gtdb_genus, gtdb_species, gtdb_classification,
  gtdb_closest_reference, gtdb_ani, gtdb_af, method, note
and prints the result per gate status, then, for isolates that failed the gate, the genus found.
The GTDB taxonomy is not the NCBI taxonomy (GTDB re-defines some genera and species and adds letter suffixes such as
'_A' to split species), so compare genus names between the two and species names only with care.
Standard library only.
"""
import csv
import os
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
OUT = DOWNSTREAM / "gtdbtk"
CL = OUT / "classify"


def read_tsv(p):
    p = Path(p)
    if not p.exists():
        return []
    with open(p, newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


gtdb = {}
for name in ("gtdbtk.bac120.summary.tsv", "gtdbtk.ar53.summary.tsv"):
    for r in read_tsv(CL / name):
        gtdb[r["user_genome"]] = r
failed = {r.get("user_genome") or r.get("name", ""): r for r in read_tsv(CL / "gtdbtk.failed_genomes.tsv")}
if not gtdb and not failed:
    raise SystemExit(f"ERROR: no GTDB-Tk summary in {CL} -- the classification did not finish")


def level(tax, prefix):
    for part in (tax or "").split(";"):
        if part.startswith(prefix):
            return part[len(prefix):]
    return ""


rows = []
for g in read_tsv(DOWNSTREAM / "species_check" / "analysis_set.tsv"):
    s = g["short_name"]
    r = gtdb.get(s)
    row = {"short_name": s, "initial_label": f"{g.get('genus_meta', '')} {g.get('species_meta', '')}".strip(),
           "qc_status": g.get("qc_status", ""), "include": g.get("include", ""), "best_ani_fastani": g.get("best_ani", "")}
    if r:
        tax = r.get("classification", "")
        ani_ref = r.get("fastani_reference", "")
        row.update({
            "gtdb_genus": level(tax, "g__") or "(unclassified at genus level)",
            "gtdb_species": level(tax, "s__"),
            "gtdb_classification": tax,
            "gtdb_closest_reference": ani_ref if ani_ref not in ("", "N/A") else r.get("closest_placement_reference", ""),
            "gtdb_ani": r.get("fastani_ani", "") if ani_ref not in ("", "N/A") else r.get("closest_placement_ani", ""),
            "gtdb_af": r.get("fastani_af", "") if ani_ref not in ("", "N/A") else r.get("closest_placement_af", ""),
            "method": r.get("classification_method", ""), "note": r.get("note", "") or r.get("warnings", "")})
    else:
        why = failed.get(s, {}).get("reason", "") if s in failed else "not in the GTDB-Tk output"
        row.update({"gtdb_genus": "FAILED", "gtdb_species": "", "gtdb_classification": "", "gtdb_closest_reference": "",
                    "gtdb_ani": "", "gtdb_af": "", "method": "", "note": why})
    rows.append(row)

cols = ["short_name", "initial_label", "qc_status", "include", "best_ani_fastani", "gtdb_genus", "gtdb_species",
        "gtdb_classification", "gtdb_closest_reference", "gtdb_ani", "gtdb_af", "method", "note"]
OUT.mkdir(parents=True, exist_ok=True)
with open(OUT / "gtdbtk_classification.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols, delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)

print(f"{'isolate':10s} {'initial label':28s} {'gate':11s} {'GTDB genus':22s} {'GTDB species':30s} {'ANI':>6s}")
for r in sorted(rows, key=lambda r: (r["include"] == "yes", r["qc_status"], r["short_name"])):
    print(f"{r['short_name']:10s} {r['initial_label'][:27]:28s} {r['qc_status']:11s} {r['gtdb_genus'][:21]:22s} "
          f"{r['gtdb_species'][:29]:30s} {r['gtdb_ani'][:6]:>6s}")
bad = [r for r in rows if r["include"] != "yes"]
if bad:
    print("\nIsolates that failed the species check, by GTDB genus:")
    by = {}
    for r in bad:
        by.setdefault(r["gtdb_genus"], []).append(r["short_name"])
    for gname, members in sorted(by.items()):
        print(f"  {gname:28s} {', '.join(sorted(members))}")
ctrl = [r for r in rows if r["include"] == "yes" and r["initial_label"].split(" ")[0] != r["gtdb_genus"]]
if ctrl:
    print("\nCONTROL FAILED -- isolates that passed the species check but have a different GTDB genus than their label:")
    for r in ctrl:
        print(f"  {r['short_name']}: {r['initial_label']} -> {r['gtdb_genus']}")
print(f"\nWrote {OUT / 'gtdbtk_classification.tsv'}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/summarize_gtdbtk.py.new" && mv -f "$SCRIPTS/summarize_gtdbtk.py.new" "$SCRIPTS/summarize_gtdbtk.py"
echo "wrote $SCRIPTS/summarize_gtdbtk.py"

cat > "$SCRIPTS/run_gtdbtk.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# GTDB-Tk classify_wf on every isolate genome (D4c.2): a genus (and species, where GTDB has one) for the isolates that
# failed the species check, with the isolates that passed as a control (they should come back as Pectobacterium or Dickeya).
# Input: the genome file the gate used for each isolate (analysis_set.tsv, column genome_path; the raw standardized
# contigs, because GTDB-Tk is being asked what the assembly IS -- a MIXED assembly is classified as a whole and the
# result for it is only a guide).
# --skip_ani_screen: placement in the reference tree only, so the Mash database is not needed; the genus comes from
# the placement and, where a reference genome is close enough, from the ANI to it.
# Memory: pplacer needs about 100 GB of RAM per process; --pplacer_cpus 1 keeps it there.
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"
GTDBTK_DB="${GTDBTK_DB:-$SOFT/gtdbtk_db}"
[ -s "$GTDBTK_DB/metadata/metadata.txt" ] || { echo "ERROR: GTDB-Tk database missing in $GTDBTK_DB -- run D4c.1 first"; exit 1; }
[ -s "$GATE" ] || { echo "ERROR: $GATE missing -- run D1 first"; exit 1; }
export GTDBTK_DATA_PATH="$GTDBTK_DB"

OUT="$DOWNSTREAM/gtdbtk"
IN="$OUT/genomes"
rm -rf "$IN" "$OUT/classify"
mkdir -p "$IN"
use_env gtdbtk_env          # python 3 and GTDB-Tk (the job has no python of its own)
python - "$GATE" "$IN" <<'PYEOF'
import csv, os, sys
gate, dest = sys.argv[1], sys.argv[2]
n = 0
for r in csv.DictReader(open(gate, newline=""), delimiter="\t"):
    src = r.get("genome_path", "")
    # the standardized contigs of the isolate, not a contig-filtered copy
    raw = os.path.join(os.environ["GENOMES"], r["short_name"] + ".fna")
    src = raw if os.path.exists(raw) else src
    if not src or not os.path.exists(src):
        print("no genome file for", r["short_name"]); continue
    os.symlink(src, os.path.join(dest, r["short_name"] + ".fna"))
    n += 1
print(n, "genomes linked into", dest)
PYEOF
ls "$IN" | head -3 >/dev/null || exit 1
gtdbtk --version 2>&1 | head -2 | tee "$OUT/gtdbtk_version.txt"
gtdbtk classify_wf --genome_dir "$IN" --extension fna --out_dir "$OUT/classify" \
    --cpus "${SLURM_CPUS_PER_TASK:-16}" --pplacer_cpus 1 --skip_ani_screen || { echo "ERROR: GTDB-Tk failed (see above)"; exit 1; }
cp "$GTDBTK_DB/metadata/metadata.txt" "$OUT/gtdbtk_db_metadata.txt" 2>/dev/null
python "$SCRIPTS/summarize_gtdbtk.py"
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_gtdbtk.sh.new" && mv -f "$SCRIPTS/run_gtdbtk.sh.new" "$SCRIPTS/run_gtdbtk.sh"
echo "wrote $SCRIPTS/run_gtdbtk.sh"

# memory: GTDB-Tk's placement step is the heavy one (its documentation asks for roughly 100 GB or more);
# lower the number only if the nodes of your partition are smaller than this
JOB=$(submit gtdbtk 16 160G 1-00:00:00 "$SCRIPTS/run_gtdbtk.sh"); echo "submitted $JOB"

wrote /home/s94m696/asap_shortname_projects/scripts/summarize_gtdbtk.py
wrote /home/s94m696/asap_shortname_projects/scripts/run_gtdbtk.sh
submitted 931308


In [16]:
%%bash
source /home/s94m696/asap_shortname_projects/config.sh
sed -i 's/ --pplacer_cpus 1 --skip_ani_screen/ --pplacer_cpus 1/' "$SCRIPTS/run_gtdbtk.sh"
grep -n "gtdbtk classify_wf" -A1 "$SCRIPTS/run_gtdbtk.sh"
JOB=$(submit gtdbtk 16 160G 1-00:00:00 "$SCRIPTS/run_gtdbtk.sh"); echo "submitted $JOB"

39:gtdbtk classify_wf --genome_dir "$IN" --extension fna --out_dir "$OUT/classify" \
40-    --cpus "${SLURM_CPUS_PER_TASK:-16}" --pplacer_cpus 1 || { echo "ERROR: GTDB-Tk failed (see above)"; exit 1; }
submitted 931384


In [17]:
%%bash
source /home/s94m696/asap_shortname_projects/config.sh
squeue -u "$USER" -j 931384 -o "%.10i %.14j %.3t %.10M %.10l %R"
jobs_like gtdbtk
tail -n 15 "$LOGS"/gtdbtk_931308.out
ls "$DOWNSTREAM"/gtdbtk/classify 2>/dev/null

     JOBID           NAME  ST       TIME TIME_LIMIT NODELIST(REASON)
               JobID                        JobName      State ExitCode    Elapsed 
              930293                   gtdbtk_setup  COMPLETED      0:0   02:26:50 
              931308                         gtdbtk     FAILED      1:0   00:00:06 
              931323                         gtdbtk  COMPLETED      0:0   00:01:18 
              931384                         gtdbtk  COMPLETED      0:0   00:01:26 
25 genomes linked into /home/s94m696/asap_shortname_projects/downstream/gtdbtk/genomes
gtdbtk: version 2.7.1 Copyright 2017 Pierre-Alain Chaumeil, Aaron Mussig and Donovan Parks
There is a newer version of GTDB-Tk available: v2.8.0
ERROR: GTDB-Tk failed (see above)
classify
gtdbtk.bac120.summary.tsv
gtdbtk.json
gtdbtk.log
gtdbtk.warnings.log


In [18]:
%%bash
source /home/s94m696/asap_shortname_projects/config.sh
tail -n 45 "$LOGS"/gtdbtk_931384.out
echo ----
column -t -s$'\t' "$DOWNSTREAM"/gtdbtk/gtdbtk_classification.tsv | cut -c1-230

[2026-10-28 11:53:35] INFO: Calculating all vs all ANI with skani v0.3.1.
[2026-10-28 11:54:46] INFO: Capturing skani results.
[2026-10-28 11:54:52] INFO: Summary of results saved to: /home/s94m696/asap_shortname_projects/downstream/gtdbtk/classify/classify/ani_screen/gtdbtk.bac120.ani_summary.tsv
[2026-10-28 11:54:52] INFO: 25 genome(s) have been classified using the ANI pre-screening step.
[2026-10-28 11:54:52] INFO: Done.
[2026-10-28 11:54:52] INFO: All genomes have been classified by the ANI screening step. Identify and Align steps will be skipped.
[2026-10-28 11:54:53] INFO: Note that Tk classification mode is insufficient for publication of new taxonomic designations. New designations should be based on one or more de novo trees, an example of which can be produced by Tk in de novo mode.
[2026-10-28 11:54:53] INFO: Done.
[2026-10-28 11:54:53] INFO: Removing intermediate files.
[2026-10-28 11:54:53] INFO: Intermediate files removed.
[2026-10-28 11:54:53] INFO: Done.
isolate    ini

In [19]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/summarize_gtdbtk.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Merge the GTDB-Tk classification with the species gate (end of D4c.2 / D4c.3).
Reads DOWNSTREAM/gtdbtk/classify/gtdbtk.{bac120,ar53}.summary.tsv (+ gtdbtk.failed_genomes.tsv if present) and the gate,
writes DOWNSTREAM/gtdbtk/gtdbtk_classification.tsv:
  short_name, initial_label, qc_status, include, best_ani_fastani, gtdb_genus, gtdb_species, gtdb_classification,
  gtdb_closest_reference, gtdb_ani, gtdb_af, method, note
and prints the result per gate status, then, for isolates that failed the gate, the genus found.
The GTDB taxonomy is not the NCBI taxonomy (GTDB re-defines some genera and species and adds letter suffixes such as
'_A' to split species), so compare genus names between the two and species names only with care.
Standard library only.
"""
import csv
import os
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
OUT = DOWNSTREAM / "gtdbtk"
CL = OUT / "classify"


def read_tsv(p):
    p = Path(p)
    if not p.exists():
        return []
    with open(p, newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


gtdb = {}
for name in ("gtdbtk.bac120.summary.tsv", "gtdbtk.ar53.summary.tsv"):
    for r in read_tsv(CL / name):
        gtdb[r["user_genome"]] = r
failed = {r.get("user_genome") or r.get("name", ""): r for r in read_tsv(CL / "gtdbtk.failed_genomes.tsv")}
if not gtdb and not failed:
    raise SystemExit(f"ERROR: no GTDB-Tk summary in {CL} -- the classification did not finish")


def level(tax, prefix):
    for part in (tax or "").split(";"):
        if part.startswith(prefix):
            return part[len(prefix):]
    return ""


rows = []
for g in read_tsv(DOWNSTREAM / "species_check" / "analysis_set.tsv"):
    s = g["short_name"]
    r = gtdb.get(s)
    row = {"short_name": s, "initial_label": f"{g.get('genus_meta', '')} {g.get('species_meta', '')}".strip(),
           "qc_status": g.get("qc_status", ""), "include": g.get("include", ""), "best_ani_fastani": g.get("best_ani", "")}
    if r:
        tax = r.get("classification", "")
        ani_ref = r.get("fastani_reference", "")
        row.update({
            "gtdb_genus": level(tax, "g__") or "(unclassified at genus level)",
            "gtdb_species": level(tax, "s__"),
            "gtdb_classification": tax,
            "gtdb_closest_reference": ani_ref if ani_ref not in ("", "N/A") else r.get("closest_placement_reference", ""),
            "gtdb_ani": r.get("fastani_ani", "") if ani_ref not in ("", "N/A") else r.get("closest_placement_ani", ""),
            "gtdb_af": r.get("fastani_af", "") if ani_ref not in ("", "N/A") else r.get("closest_placement_af", ""),
            "method": r.get("classification_method", ""), "note": r.get("note", "") or r.get("warnings", "")})
    else:
        why = failed.get(s, {}).get("reason", "") if s in failed else "not in the GTDB-Tk output"
        row.update({"gtdb_genus": "FAILED", "gtdb_species": "", "gtdb_classification": "", "gtdb_closest_reference": "",
                    "gtdb_ani": "", "gtdb_af": "", "method": "", "note": why})
    rows.append(row)

cols = ["short_name", "initial_label", "qc_status", "include", "best_ani_fastani", "gtdb_genus", "gtdb_species",
        "gtdb_classification", "gtdb_closest_reference", "gtdb_ani", "gtdb_af", "method", "note"]
OUT.mkdir(parents=True, exist_ok=True)
with open(OUT / "gtdbtk_classification.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols, delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)

print(f"{'isolate':10s} {'initial label':28s} {'gate':11s} {'GTDB genus':22s} {'GTDB species':30s} {'ANI':>6s}")
for r in sorted(rows, key=lambda r: (r["include"] == "yes", r["qc_status"], r["short_name"])):
    print(f"{r['short_name']:10s} {r['initial_label'][:27]:28s} {r['qc_status']:11s} {r['gtdb_genus'][:21]:22s} "
          f"{r['gtdb_species'][:29]:30s} {r['gtdb_ani'][:6]:>6s}")
bad = [r for r in rows if r["include"] != "yes"]
if bad:
    print("\nIsolates that failed the species check, by GTDB genus:")
    by = {}
    for r in bad:
        by.setdefault(r["gtdb_genus"], []).append(r["short_name"])
    for gname, members in sorted(by.items()):
        print(f"  {gname:28s} {', '.join(sorted(members))}")
ctrl = [r for r in rows if r["include"] == "yes" and r["initial_label"].split(" ")[0] != r["gtdb_genus"]]
if ctrl:
    print("\nCONTROL FAILED -- isolates that passed the species check but have a different GTDB genus than their label:")
    for r in ctrl:
        print(f"  {r['short_name']}: {r['initial_label']} -> {r['gtdb_genus']}")
print(f"\nWrote {OUT / 'gtdbtk_classification.tsv'}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/summarize_gtdbtk.py.new" && mv -f "$SCRIPTS/summarize_gtdbtk.py.new" "$SCRIPTS/summarize_gtdbtk.py"
echo "wrote $SCRIPTS/summarize_gtdbtk.py"

jobs_like "gtdbtk"
tail -n 3 "$LOGS"/gtdbtk_setup_*.out 2>/dev/null | tail -n 6
if [ -s "$DOWNSTREAM/gtdbtk/gtdbtk_classification.tsv" ]; then
    use_env download_env
    python "$SCRIPTS/summarize_gtdbtk.py"
else
    echo "no GTDB-Tk classification yet (D4c.2 has not finished)"
fi

wrote /home/s94m696/asap_shortname_projects/scripts/summarize_gtdbtk.py
               JobID                        JobName      State ExitCode    Elapsed 
              930293                   gtdbtk_setup  COMPLETED      0:0   02:26:50 
              931308                         gtdbtk     FAILED      1:0   00:00:06 
              931323                         gtdbtk  COMPLETED      0:0   00:01:18 
              931384                         gtdbtk  COMPLETED      0:0   00:01:26 
RED_DIST_ARC_DICT={"d__":0.00,"p__":0.2859284763413424,"c__":0.42852344844350987,"o__":0.5692329378979698,"f__":0.758762614031466,"g__":0.9250910192387802}
VERSION_DATA=r232

isolate    initial label                gate        GTDB genus             GTDB species                      ANI
PI012      Pectobacterium versatile     MIXED       Pantoea                Pantoea agglomerans               N/A
PI053      Pectobacterium versatile     MIXED       Pectobacterium         Pectobacterium versatile        

## D5. *P. versatile*: environment vs. diseased-plant gene content (descriptive)
After the gate (PI012 and PI053 are `MIXED`) the comparison is 5 plant vs. 3 environmental isolates, so the smallest possible Fisher p is 1/56 = 0.018 and no per-gene test can survive multiple-testing correction. Every lineage comes from a single source (L1/L2 plant, L3 environment), so source-splitting genes cannot be told apart from lineage markers. The table lists every accessory cluster with presence by source and by lineage; use it to pick regions to inspect, not as evidence of source association.

> The CAZyme / PCWDE columns in `source_metadata_correlated.tsv` come from two different methods (rows 1–14 ≈ 240–258 CAZymes, rows 15–25 ≈ 1,215–1,285; e.g. *P. polaris* PI009 = 255 vs. PI008 = 1,255). Do not use them: Part F recomputes them with one dbCAN run for every genome (`functional/tables/functional_summary.tsv`).
>
> This step needs `source_metadata_correlated.tsv` in `metadata/` (the table with `workflow_short_name`, `lineage`, `source_class`, ...). The cell stops with a message if it is not there.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/pversatile_gene_partition.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Descriptive environment vs. diseased-plant gene partition, P. versatile.

Replaces the first run's Fisher + BH "association" test. With 3 environmental
vs. 7 plant isolates the smallest possible two-sided Fisher p is 1/120 = 0.0083,
so after BH correction across thousands of accessory clusters nothing can reach
significance by construction. In addition, every lineage here comes from a
single source (L1/L2 plant, L3 environment), so any gene that splits by source
also splits by lineage -- source and lineage cannot be told apart.

This script therefore reports, for every accessory cluster (present in some but
not all isolates):
  * presence counts in plant vs. environment isolates
  * presence pattern per lineage
  * source_split   = env_only / plant_only when the cluster perfectly separates sources
  * lineage_consistent = presence is identical within every multi-isolate lineage
  * fisher_p       = kept only as a ranking aid, not as evidence

  -> DOWNSTREAM/pversatile_source/Pversatile_gene_partition.tsv
"""
import csv
import os
from math import comb
from pathlib import Path

META = Path(os.environ["META"])
DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
roary_dir = DOWNSTREAM / "roary" / "Pectobacterium_versatile" / "roary_output"
out_dir = DOWNSTREAM / "pversatile_source"
out_dir.mkdir(parents=True, exist_ok=True)


def hp(a, r1, c1, n):
    return comb(c1, a) * comb(n - c1, r1 - a) / comb(n, r1)


def fisher_two_sided(a, b, c, d):
    r1, c1, n = a + b, a + c, a + b + c + d
    obs = hp(a, r1, c1, n)
    lo, hi = max(0, r1 - (n - c1)), min(r1, c1)
    return min(1.0, sum(hp(x, r1, c1, n) for x in range(lo, hi + 1) if hp(x, r1, c1, n) <= obs + 1e-12))


src, lin = {}, {}
with open(META / "source_metadata_correlated.tsv", newline="") as f:
    for r in csv.DictReader(f, delimiter="\t"):
        if r["genus"] == "Pectobacterium" and r["species_workflow"] == "versatile" \
                and r["source_class"] in {"environment", "diseased_plant"}:
            src[r["workflow_short_name"]] = r["source_class"]
            lin[r["workflow_short_name"]] = r["lineage"]

annot = {}
with open(roary_dir / "gene_presence_absence.csv", newline="") as f:
    for r in csv.DictReader(f):
        annot[r["Gene"]] = r.get("Annotation", "")

with open(roary_dir / "gene_presence_absence.Rtab") as f:
    header = f.readline().rstrip("\n").split("\t")[1:]
    samples = [s for s in header if s in src]
    idx = {s: header.index(s) for s in samples}
    env = sorted(s for s in samples if src[s] == "environment")
    plant = sorted(s for s in samples if src[s] == "diseased_plant")
    not_in_rtab = sorted(set(src) - set(header))
    lineages = {}
    for s in samples:   # each singleton is its own group, labelled by isolate
        key = f"singleton_{s}" if lin[s].lower() == "singleton" else lin[s]
        lineages.setdefault(key, []).append(s)
    multi = {L: m for L, m in lineages.items() if len(m) > 1}

    rows = []
    for line in f:
        parts = line.rstrip("\n").split("\t")
        gene, vals = parts[0], parts[1:]
        pres = {s: int(vals[idx[s]]) for s in samples}
        total = sum(pres.values())
        if total in (0, len(samples)):
            continue                                   # core or absent: not informative
        a = sum(pres[s] for s in plant)
        c = sum(pres[s] for s in env)
        b, d = len(plant) - a, len(env) - c
        if a == len(plant) and c == 0:
            split = "plant_only"
        elif c == len(env) and a == 0:
            split = "env_only"
        else:
            split = ""
        lin_consistent = all(len({pres[s] for s in m}) == 1 for m in multi.values())
        rows.append({
            "gene_cluster": gene, "annotation": annot.get(gene, ""),
            "plant_present": f"{a}/{len(plant)}", "env_present": f"{c}/{len(env)}",
            "per_lineage": "; ".join(f"{L}:{sum(pres[s] for s in m)}/{len(m)}" for L, m in sorted(lineages.items())),
            "source_split": split, "lineage_consistent": "yes" if lin_consistent else "no",
            "fisher_p": f"{fisher_two_sided(a, b, c, d):.4g}",
        })

rows.sort(key=lambda r: (r["source_split"] == "", float(r["fisher_p"])))
out = out_dir / "Pversatile_gene_partition.tsv"
with out.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["gene_cluster"], delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)

source_per_lineage = {L: {src[s] for s in m} for L, m in multi.items()}
confounded = all(len(v) == 1 for v in source_per_lineage.values())
n_split = sum(1 for r in rows if r["source_split"])
print("Plant isolates:", plant)
print("Environmental isolates:", env)
if not_in_rtab:
    print("In metadata but not in the Roary table:", not_in_rtab)
print("Lineages:", {L: f"{len(m)} ({'/'.join(sorted({src[s] for s in m}))})" for L, m in sorted(lineages.items())})
print(f"Accessory clusters: {len(rows)}; clusters that perfectly split source: {n_split}")
print(f"Smallest achievable Fisher p with {len(plant)} vs {len(env)}: "
      f"{fisher_two_sided(0, len(plant), len(env), 0):.4f}")
if confounded:
    print("NOTE: every lineage comes from a single source, so source-splitting clusters are "
          "indistinguishable from lineage markers. Treat this table as descriptive.")
print("Wrote", out)
END_OF_SCRIPT
chmod +x "$SCRIPTS/pversatile_gene_partition.py.new" && mv -f "$SCRIPTS/pversatile_gene_partition.py.new" "$SCRIPTS/pversatile_gene_partition.py"
echo "wrote $SCRIPTS/pversatile_gene_partition.py"

if [ ! -s "$META/source_metadata_correlated.tsv" ]; then
    echo "STOP: $META/source_metadata_correlated.tsv is missing."
    echo "Copy your source_metadata_correlated.tsv into $META (it has the columns workflow_short_name, lineage, source_class, ...), then re-run this cell."
    ls -l "$META"
    exit 1
fi
use_env download_env
python "$SCRIPTS/pversatile_gene_partition.py"
head -n 25 "$DOWNSTREAM/pversatile_source/Pversatile_gene_partition.tsv" | cut -f1-8

## D6. Genome maps (*P. versatile*)
Environmental: PI019, PI020 (L3), PI025 (singleton). Diseased plant: PI022 (L1), PI053 (L1, only if it is in the analysis set), PI041, PI044 (L2). Isolates that are not in the analysis set are skipped (PI053 is `MIXED`; add PI043 or PI051 if you want another L2 isolate). GenBank files come from `annotation_files.tsv` (ASA³P's annotation, or C1's where it replaced it). Edit `order.txt` to change isolates or order.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

MAP="$DOWNSTREAM/genome_maps/Pversatile_env_vs_plant"
mkdir -p "$MAP"/{gbk,results}
printf '%s\n' PI019 PI020 PI025 PI022 PI053 PI041 PI044 > "$MAP/order.wanted"
: > "$MAP/order.txt"
while read -r s; do
    if awk -F'\t' -v s="$s" 'NR == 1 {for (i = 1; i <= NF; i++) c[$i] = i; next} $c["short_name"] == s && $c["include"] == "yes" {f = 1} END {exit !f}' "$GATE"; then
        echo "$s" >> "$MAP/order.txt"
    else
        echo "skipping $s: not in the analysis set (see D1.2)"
    fi
done < "$MAP/order.wanted"
while read -r s; do
    gbk=$(awk -F'\t' -v s="$s" '$1 == s {print $5}' "$DOWNSTREAM/annotation_files.tsv")
    if [ -s "$gbk" ]; then cp -f "$gbk" "$MAP/gbk/${s}.gbk"; else echo "WARNING: no GBK for $s"; fi
done < "$MAP/order.txt"
ls -lh "$MAP/gbk"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_pgv_maps.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Whole-genome synteny map (pyGenomeViz + MUMmer) for the selected P. versatile
# isolates. GBKs come from annotation_files.tsv, i.e. raw-contig annotations,
# so contig order reflects the isolate's own assembly, not a reference.
set -euo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"
use_env genomeviz_env
# Jupyter sets MPLBACKEND to its inline backend and a submitted job inherits it; matplotlib cannot use that in a batch job
export MPLBACKEND=Agg

MAP="$DOWNSTREAM/genome_maps/Pversatile_env_vs_plant"
cd "$MAP"
mapfile -t gbks < <(awk 'NF' order.txt | sed 's#^#gbk/#; s#$#.gbk#')
echo "Map order: ${gbks[*]}"
pgv-mummer "${gbks[@]}" -o results/pgv_mummer
find results/pgv_mummer -maxdepth 2 -type f | sort
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_pgv_maps.sh.new" && mv -f "$SCRIPTS/run_pgv_maps.sh.new" "$SCRIPTS/run_pgv_maps.sh"
echo "wrote $SCRIPTS/run_pgv_maps.sh"

JOB=$(submit pgv_maps 8 64G 12:00:00 "$SCRIPTS/run_pgv_maps.sh"); echo "submitted $JOB"

Gene-cluster view with clinker (**optional**). clinker aligns **gene clusters** (a locus of up to a few hundred kb); a whole 5 Mb genome gives an enormous, slow page, so the cell does nothing unless you set `RUN_CLINKER=yes` and put extracted locus GenBank files in `genome_maps/Pversatile_env_vs_plant/clusters/`. The whole-genome comparison is the pyGenomeViz map above.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

RUN_CLINKER=no     # yes = run clinker on the locus files in $MAP/clusters/
MAP="$DOWNSTREAM/genome_maps/Pversatile_env_vs_plant"
if [ "$RUN_CLINKER" != yes ]; then echo "clinker skipped (optional; see the note above)."; exit 0; fi
conda env list | awk '{print $1}' | grep -qx clinker_env || { echo "clinker_env does not exist: re-run A0.3 (package clinker-py)."; exit 1; }
ls "$MAP"/clusters/*.gbk >/dev/null 2>&1 || { echo "Put locus GenBank files in $MAP/clusters/ first."; exit 1; }
use_env clinker_env
clinker "$MAP"/clusters/*.gbk -p "$MAP/results/pversatile_clinker.html"
ls -lh "$MAP/results"

# Part F — Functional screens of all genomes
**`[Outside ASA³P]`** — ASA³P does not run these tools. Part F sits **before Part E** because E (versions, Methods draft, output map) reports its results.

It screens **every isolate that has an annotation, including the ones that failed the species check**, so the thesis can show all 25 genomes. Every result row carries the isolate's `gate_status` and its genome-based identification (from D1.2 and D1.3). For an isolate that did not pass, the numbers describe whatever organism or mixture is in the assembly: report it as "re-sequencing pending" and keep it out of the soft-rot *Pectobacteriaceae* comparisons.

| Question | Tool | Runs on | Where it appears |
|---|---|---|---|
| CAZymes and plant-cell-wall-degrading enzyme (PCWDE) families | dbCAN v5 (HMMER, dbCAN-sub, DIAMOND) | proteins | `cazymes_*`, `pcwde_*` |
| Secretion systems | MacSyFinder v2 with the TXSScan models | proteins | `secretion_systems_*`, `T2SS` … `T6SS` |
| Prophages and plasmids | geNomad | contigs | `genomad_*` |
| Plasmid replicons, relaxases, mobility | MOB-suite (`mob_recon`), PlasmidFinder (ABRicate) | contigs | `mob_*`, `plasmidfinder_*` |
| Virulence factors | VFDB (ABRicate) | contigs | `vfdb_*` |
| Metal and biocide resistance | BacMet2 (ABRicate) | contigs | `bacmet2_*` |
| Type III effector candidates | EffectiveT3 (a soft step, see below) | proteins | `effector_candidates` |

**Order:** D1.2 and D1.3 first (the species gate and the identity table; Part F runs without them, but the identity columns are then empty), C4 (it collects the annotations), then F0 → F1 → F2 → F3 → F4 → F5.

**Reading the numbers.** A tool that did not finish for an isolate gives `NA`, never 0. The PCWDE classes (pectin, cellulose, hemicellulose) are assigned by CAZy family (`pcwde_family_map.tsv`); activity varies within a family, so they are candidates, not measured enzyme activities. Type III effector prediction and secretion-system detection are sequence-based predictions.

**EffectiveT3** is a standalone Java program that cannot be tested ahead of time on your cluster. It runs as its own step: if it fails, that step is recorded as `SKIPPED` with the reason and the other five tools are not affected. EffectiveDB is a web server and is not run here.

### F0. Install the tools and databases (once)
One submitted job. It creates four conda environments (`dbcan_env`, `macsy_env`, `genomad_env`, `mobsuite_env`, one per tool so their dependencies cannot clash), downloads the databases (dbCAN, geNomad, MOB-suite, the TXSScan models; several GB) and the EffectiveT3 jar files, and checks that the ABRicate image from A0.4 has VFDB, BacMet2 and PlasmidFinder. The compute node needs internet (the container pulls in A0.4 already do). Expect 1–3 hours, mostly conda solving and downloads.

**Safe to re-run:** an environment is kept if its main command works (otherwise it is rebuilt), and a database is kept once its `.done_<name>` marker exists, so a re-run only repeats what failed. Downloads are retried; dbCAN is fetched from its AWS S3 release first and from the university server (long time-out, resumable) if that fails; MOB-suite falls back from bioconda to `pip` (with BLAST+ and Mash from conda). Every item keeps its own log in `functional/setup_logs/`, and the cell below prints the end of the log of each item that failed.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/setup_functional.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Part F, one-time setup (run as a submitted job: it downloads several GB).
#   * four conda environments (one per tool, so their dependencies cannot clash)
#   * databases: dbCAN, geNomad, MOB-suite, MacSyFinder TXSScan models
#   * EffectiveT3 1.0.1 jars (soft step: a failure here does not stop the rest)
# Safe to run again: whatever is already installed is left alone.
# Each item writes its own log to functional/setup_logs/<kind>_<name>.log (F0 shows the end of the
# log of every item that failed).
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"
SLOG="$FUNC/setup_logs"
mkdir -p "$FUNC_DB" "$ET3_DIR/module" "$FUNC" "$SLOG"
STATUS="$FUNC/setup_status.tsv"
: > "$STATUS"
note() { printf '%s\t%s\t%s\t%s\n' "$1" "$2" "${3:-}" "${4:-}" | tee -a "$STATUS"; }

module load "$CONDA_MODULE"
set +u; source "$(conda info --base)/etc/profile.d/conda.sh"; set -u

# A download that can time out is repeated (waiting a little longer each time)
retry() {   # retry <tries> <command> [args]
    local n=$1 i; shift
    for i in $(seq 1 "$n"); do
        "$@" && return 0
        echo "[retry] attempt $i of $n failed: $*"
        [ "$i" -lt "$n" ] && sleep $(( ${RETRY_WAIT:-30} * i ))
    done
    return 1
}

# ---------------------------------------------------------------- environments
env_ok() {   # env_ok <env> <command>: the environment's main command is found once it is activated
    ( use_env "$1" && command -v "$2" >/dev/null 2>&1 )
}
# bioconda first; if its package is missing or does not solve, a python + pip environment instead
# (<extra conda packages> are what the pip package needs from outside Python; leave "" for none)
ensure_env() {   # ensure_env <env> <command> <pip package> "<extra conda packages>" <bioconda package> [...]
    local env=$1 cmd=$2 pippkg=$3 extra=$4; shift 4
    if conda env list | awk '{print $1}' | grep -qx "$env"; then
        if env_ok "$env" "$cmd"; then echo "[env] $env exists and $cmd is found"; return 0; fi
        echo "[env] $env exists but $cmd is not found in it (an interrupted install?): rebuilding"
        conda env remove -y -n "$env" >/dev/null 2>&1
    fi
    echo "[env] creating $env from bioconda: $*"
    if conda create -y -n "$env" -c conda-forge -c bioconda "$@" && env_ok "$env" "$cmd"; then return 0; fi
    echo "[env] the bioconda route failed for $env"
    [ -n "$pippkg" ] || return 1
    echo "[env] trying python + pip instead (pip install $pippkg; conda supplies: $extra)"
    conda env remove -y -n "$env" >/dev/null 2>&1
    # shellcheck disable=SC2086
    conda create -y -n "$env" -c conda-forge -c bioconda python=3.11 pip $extra || return 1
    ( use_env "$env" && pip install "$pippkg" ) || return 1
    env_ok "$env" "$cmd" || { echo "[env] $cmd is still not found in $env"; return 1; }
}
envstep() {   # envstep <env> <command> <pip package> "<extra conda packages>" <bioconda package> [...]
    local env=$1
    if ensure_env "$@" 2>&1 | tee "$SLOG/env_$env.log"; then note env "$env" OK
    else note env "$env" FAILED "log: $SLOG/env_$env.log"; fi
}
envstep dbcan_env    run_dbcan   dbcan       "hmmer diamond"       "dbcan>=5" diamond hmmer
envstep macsy_env    macsyfinder macsyfinder "hmmer"               macsyfinder hmmer
envstep genomad_env  genomad     ""          ""                    genomad
# MOB-suite by pip needs BLAST+ and Mash from conda, and pycurl from conda (so that it is not compiled)
envstep mobsuite_env mob_recon   mob_suite   "blast mash pycurl"   mob_suite

# ------------------------------------------------------------------- databases
# A marker file (.done_<name>) is written only after a download finished and was checked, so an
# interrupted download is repeated and a finished one is not. Part F's steps look for the marker.
dbstep() {   # dbstep <name> <function>
    local name=$1; shift
    if [ -e "$FUNC_DB/.done_$name" ]; then note db "$name" present; return 0; fi
    if "$@" 2>&1 | tee "$SLOG/db_$name.log"; then touch "$FUNC_DB/.done_$name"; note db "$name" OK
    else note db "$name" FAILED "log: $SLOG/db_$name.log"; fi
}

need_env() {   # need_env <env> <command>: activate the environment; stop at once if its tool is not there
    use_env "$1"
    command -v "$2" >/dev/null 2>&1 || { echo "$2 is not available in $1 (see the environment step above)"; return 1; }
}

# dbCAN. The university server (pro.unl.edu) answers slowly and run_dbcan's default is to give up after
# 30 s without data, which is what produced "Read timed out". So: the pinned AWS S3 release first
# (the same files, matching the installed version), then the university server; both with long time-outs,
# resumable. Only the CAZyme files are fetched (--no-cgc): Part F annotates proteins, not gene clusters.
# Flags are only used if this dbCAN version has them.
dl_dbcan() {
    need_env dbcan_env run_dbcan || return 1
    local db="$FUNC_DB/dbcan_db" h flags=()
    mkdir -p "$db"
    h=$(COLUMNS=250 run_dbcan database --help 2>&1) || true     # wide, so that the help text does not wrap an option name
    grep -q -- '--no-cgc'  <<< "$h" && flags+=(--no-cgc)
    grep -q -- '--resume'  <<< "$h" && flags+=(--resume)
    grep -q -- '--retries' <<< "$h" && flags+=(--retries 5)
    grep -q -- '--timeout' <<< "$h" && flags+=(--timeout 600)
    echo "[dbcan] options: ${flags[*]-}"
    if grep -q -- '--aws_s3' <<< "$h"; then
        retry 3 run_dbcan database --db_dir "$db" ${flags[@]+"${flags[@]}"} --aws_s3 && dbcan_ok "$db" && return 0
        echo "[dbcan] the AWS S3 release could not be completed: trying the university server"
    fi
    retry 3 run_dbcan database --db_dir "$db" ${flags[@]+"${flags[@]}"} && dbcan_ok "$db"
}
dbcan_ok() {   # the files a protein-mode annotation needs are present
    local db=$1 f
    for f in CAZy.dmnd; do [ -s "$db/$f" ] || { echo "[dbcan] $db/$f is missing"; return 1; }; done
    ls "$db"/*.hmm >/dev/null 2>&1 || { echo "[dbcan] no .hmm files in $db"; return 1; }
}
dl_genomad() { need_env genomad_env  genomad  && ( cd "$FUNC_DB" && retry 3 genomad download-database . ); }       # creates genomad_db/
dl_mob()     { need_env mobsuite_env mob_init && retry 2 mob_init -d "$FUNC_DB/mob_db"; }                          # downloads and sketches
dl_txss()    { need_env macsy_env    msf_data && retry 2 msf_data install TXSScan --target "$FUNC_DB/macsy_models" && [ -d "$FUNC_DB/macsy_models/TXSScan" ]; }
dbstep dbcan    dl_dbcan
dbstep genomad  dl_genomad
dbstep mobsuite dl_mob
dbstep txsscan  dl_txss

# ----------------------------------------------------------------- EffectiveT3
# jars from the Galaxy software depot (the university download page needs a browser)
DEPOT="https://depot.galaxyproject.org/software"
get() {   # get <url> <file>
    [ -s "$2" ] && return 0
    curl -fsSL --retry 3 -o "$2.part" "$1" && mv "$2.part" "$2"
}
if get "$DEPOT/TTSS_GUI/TTSS_GUI_1.0.1_src_all.jar" "$ET3_DIR/TTSS_GUI-1.0.1.jar" \
   && get "$DEPOT/TTSS_STD/TTSS_STD_2.0.2_src_all.jar" "$ET3_DIR/module/TTSS_STD-2.0.2.jar" \
   && get "$DEPOT/TTSS_STD/TTSS_STD_1.0.1_src_all.jar" "$ET3_DIR/module/TTSS_STD-1.0.1.jar" \
   && get "$DEPOT/TTSS_PLANT/TTSS_PLANT_1.0.1_src_all.jar" "$ET3_DIR/module/TTSS_PLANT-1.0.1.jar"; then
    note db effectiveT3 OK
else
    note db effectiveT3 FAILED "download from the Galaxy depot failed; Part F will skip EffectiveT3"
fi

# ----------------------------------------------------- ABRicate databases check
for db in vfdb bacmet2 plasmidfinder; do
    if apptainer exec --bind "$ROOT":"$ROOT" "$ABRICATE_IMG" abricate --list 2>/dev/null | awk '{print $1}' | grep -qx "$db"; then
        note abricate "$db" present
    else
        note abricate "$db" MISSING
    fi
done

echo
echo "===== setup summary ====="
awk -F'\t' '{printf "%-10s %-14s %-8s %s\n", $1, $2, $3, $4}' "$STATUS"
if grep -qE "FAILED|MISSING" "$STATUS"; then
    echo "Some items failed: Part F will skip the tools that depend on them (F3 shows which)."
    exit 1
fi
echo "All Part F tools and databases are ready."
END_OF_SCRIPT
chmod +x "$SCRIPTS/setup_functional.sh.new" && mv -f "$SCRIPTS/setup_functional.sh.new" "$SCRIPTS/setup_functional.sh"
echo "wrote $SCRIPTS/setup_functional.sh"

JOB=$(submit func_setup 8 32G 12:00:00 "$SCRIPTS/setup_functional.sh"); echo "submitted $JOB"

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like func_setup
echo
if [ -s "$FUNC/setup_status.tsv" ]; then awk -F'\t' '{printf "%-10s %-14s %-8s %s\n", $1, $2, $3, $4}' "$FUNC/setup_status.tsv"; else echo "no status yet (job not finished or not started)"; fi
# the end of the log of every item that failed
awk -F'\t' '$3 == "FAILED" {print $1 "_" $2}' "$FUNC/setup_status.tsv" 2>/dev/null | while read -r item; do
    [ -s "$FUNC/setup_logs/$item.log" ] || continue
    echo; echo "--- end of $FUNC/setup_logs/$item.log"
    tail -n 12 "$FUNC/setup_logs/$item.log"
done

Expect `OK` or `present` on every line. `FAILED` for an environment or a database means Part F will report that tool as not run (the end of its log is printed above); once the job has finished, re-run the submit cell to repeat only what failed. `FAILED` for `effectiveT3` only affects the effector step.

### F1. Protein files and the isolate list
Reads the protein translations from each isolate's GenBank annotation (ASA³P's, or Prokka's where C1 replaced it) into `functional/proteins/<short>.faa`, and writes `functional/isolates.tsv` with the number of genes, the species-check status and the genome-based identification. Expect one row per isolate; an isolate with no annotation is listed under "Left out" (run C1 and C4 for it).

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/make_identity_table.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Two-column identification table for the supplement (and for Part F).

Initial identification (from the metadata) next to the genome-based identification:
  PASS / RELABELLED  fastANI species (>=95% ANI over >=50% of the assembly)
  MIXED              that species + "second organism (mixed assembly)"
  NO_MATCH           ASA3P k-mer classification (closest database genome), else "mixed culture, no single-genome match"
Needs D1.2 (analysis_set.tsv) and, for NO_MATCH isolates, asap_taxonomy.tsv (D1.3).
  -> DOWNSTREAM/species_check/supplement_isolate_identity.tsv
"""
import csv
import os
from pathlib import Path

OUT = Path(os.environ["DOWNSTREAM"]) / "species_check"
rd = lambda p: {r["short_name"]: r for r in csv.DictReader(open(p, newline=""), delimiter="\t")}
gate = rd(OUT / "analysis_set.tsv")
tax = rd(OUT / "asap_taxonomy.tsv") if (OUT / "asap_taxonomy.tsv").exists() else {}
if not tax:
    print("NOTE: asap_taxonomy.tsv not found (run the ASA3P taxonomy summary in D1.3 first): NO_MATCH isolates will read 'mixed culture, no single-genome match'.")

rows = []
for s in sorted(gate):
    g, t = gate[s], tax.get(s, {})
    status = g["qc_status"]
    initial = f"{g['genus_meta']} {g['species_meta']}"
    if status in ("PASS", "RELABELLED"):
        final = f"{g['genus_final']} {g['species_final']}"
        ani, aln = g["best_ani"], f"{100 * float(g['aligned_fraction']):.0f}"
        basis = "fastANI to species reference (>=95% ANI, >=50% aligned)"
    elif status == "MIXED":
        final = f"{g['genus_final']} {g['species_final']} + second organism (mixed assembly)"
        ani, aln = g["best_ani"], f"{100 * float(g['aligned_fraction']):.0f}"
        basis = "fastANI to species reference; <50% of the assembly aligns"
    else:  # NO_MATCH
        final = t.get("asap_closest_genome") or "mixed culture, no single-genome match"
        ani, aln = t.get("kmer_ani_pct", ""), t.get("kmer_conserved_dna_pct", "")
        basis = "ASA3P k-mer classification (closest database genome)"
    rows.append({"isolate": s, "initial_identification": initial, "genome_based_identification": final,
                 "ani_pct": ani, "genome_aligned_pct": aln, "basis": basis,
                 "gate_status": status, "in_analysis": g["include"]})

out = OUT / "supplement_isolate_identity.tsv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]), delimiter="\t", lineterminator="\n"); w.writeheader(); w.writerows(rows)
for r in rows:
    print(f"{r['isolate']:8s} {r['initial_identification'][:28]:28s} -> {r['genome_based_identification'][:62]:62s} {r['gate_status']}")
print(f"\nWrote {out}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/make_identity_table.py.new" && mv -f "$SCRIPTS/make_identity_table.py.new" "$SCRIPTS/make_identity_table.py"
echo "wrote $SCRIPTS/make_identity_table.py"
cat > "$SCRIPTS/functional_inputs.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Inputs for Part F (comparative-genomic screens): one protein FASTA per isolate.

Why this exists: dbCAN, MacSyFinder/TXSScan and EffectiveT3 work on PROTEINS, and
every isolate's annotation lives in a GenBank file (ASA3P's own, or Prokka's from
C1; annotation_files.tsv says which). The proteins are read from the /translation
of each CDS, so they are exactly the genes of the annotation used elsewhere in
this notebook.

Protein IDs are "<short>c<contig number>_<gene number>" (for example PI019c0007_00012):
MacSyFinder's "gembase" format needs <replicon>_<number>, with every contig as a
replicon, and the same ID is then used by all tools. <short>.idmap.tsv translates
an ID back to the annotation's locus tag, contig, position and product.

Every isolate with an annotation is included whatever its species-check status:
the status and the genome-based identification travel with the results as columns.

  -> DOWNSTREAM/functional/proteins/<short>.faa and <short>.idmap.tsv
  -> DOWNSTREAM/functional/isolates.tsv
"""
import csv
import os
import re
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
GENOMES = Path(os.environ["GENOMES"])
FUNC = Path(os.environ.get("FUNC") or DOWNSTREAM / "functional")
CHECK = DOWNSTREAM / "species_check"


def read_tsv(path):
    if not Path(path).exists():
        return []
    with open(path, newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


def parse_qualifiers(lines):
    """Qualifier lines of one feature -> dict (first value wins; flags map to True)."""
    q, key, val = {}, None, []

    def flush():
        if key is not None and key not in q:
            v = " ".join(val).strip()
            if v.startswith('"'):
                v = v[1:]
            if v.endswith('"'):
                v = v[:-1]
            q[key] = v if v != "" or val else True
    for ln in lines:
        m = re.match(r"^/([A-Za-z_0-9]+)(?:=(.*))?$", ln)
        if m:
            flush()
            key, val = m.group(1), ([m.group(2)] if m.group(2) is not None else [])
            if m.group(2) is None:
                q.setdefault(key, True)
                key, val = None, []
        elif key is not None:
            val.append(ln.strip())
    flush()
    return q


def parse_gbk(path):
    """Yield (contig index, contig name, CDS dict) in file order."""
    contig_idx, contig = 0, ""
    in_feat, cur = False, None
    out = []

    def flush():
        nonlocal cur
        if cur is not None:
            loc = cur["loc"]
            nums = [int(x) for x in re.findall(r"\d+", loc)]
            q = parse_qualifiers(cur["q"])
            out.append((contig_idx, contig, {
                "start": min(nums) if nums else 0, "end": max(nums) if nums else 0,
                "strand": "-" if loc.startswith("complement") else "+",
                "locus_tag": q.get("locus_tag", "") if q.get("locus_tag") is not True else "",
                "product": q.get("product", "") if q.get("product") is not True else "",
                "gene": q.get("gene", "") if q.get("gene") is not True else "",
                "pseudo": bool(q.get("pseudo") or q.get("pseudogene")),
                "translation": re.sub(r"[^A-Za-z]", "", q.get("translation", "")) if isinstance(q.get("translation"), str) else ""}))
        cur = None

    with open(path) as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("LOCUS"):
                flush()
                contig_idx += 1
                parts = line.split()
                contig = parts[1] if len(parts) > 1 else f"contig{contig_idx}"
                in_feat = False
            elif line.startswith("FEATURES"):
                in_feat = True
            elif line.startswith("ORIGIN") or line.startswith("//"):
                flush()
                in_feat = False
            elif in_feat:
                if re.match(r"^ {5}\S", line):
                    flush()
                    key, loc = line[5:21].strip(), line[21:].strip()
                    cur = {"loc": loc, "q": []} if key == "CDS" else None
                elif cur is not None and line.startswith(" " * 21):
                    txt = line[21:]
                    if not cur["q"] and not txt.startswith("/"):
                        cur["loc"] += txt.strip()          # location continued on the next line
                    elif txt.startswith("/"):
                        cur["q"].append(txt)
                    else:
                        cur["q"][-1] += " " + txt.strip()   # wrapped qualifier value
    flush()
    return out


def wrap(seq, n=60):
    return "\n".join(seq[i:i + n] for i in range(0, len(seq), n))


ann = read_tsv(DOWNSTREAM / "annotation_files.tsv")
if not ann:
    raise SystemExit("ERROR: downstream/annotation_files.tsv is missing or empty: run C4 (it collects the annotations).")
gate = {g["short_name"]: g for g in read_tsv(CHECK / "analysis_set.tsv")}
ident = {r["isolate"]: r for r in read_tsv(CHECK / "supplement_isolate_identity.tsv")}
if not gate:
    print("NOTE: no species gate yet (D1.2): status columns will be empty.")
if gate and not ident:
    print("NOTE: no supplement_isolate_identity.tsv (D1.3) -- 'genome-based identification' falls back to the gate's species.")

(FUNC / "proteins").mkdir(parents=True, exist_ok=True)
rows, problems = [], []
for a in ann:
    s = a["short_name"]
    gbk = a.get("gbk", "")
    fna = GENOMES / f"{s}.fna"
    if not gbk or not Path(gbk).is_file():
        problems.append(f"{s}: no GenBank file in annotation_files.tsv")
        continue
    if not fna.is_file():
        problems.append(f"{s}: {fna} is missing (run B1)")
        continue
    cds = parse_gbk(gbk)
    faa, idmap, counter = [], [], {}
    for ci, cname, c in cds:
        if c["pseudo"] or len(c["translation"]) < 10:
            continue
        counter[ci] = counter.get(ci, 0) + 1
        pid = f"{s}c{ci:04d}_{counter[ci]:05d}"
        prod = re.sub(r"\s+", " ", c["product"]).strip()
        faa.append(f">{pid} {c['locus_tag']} {prod}\n{wrap(c['translation'])}\n")
        idmap.append([pid, c["locus_tag"], cname, c["start"], c["end"], c["strand"], c["gene"], prod])
    if not faa:
        problems.append(f"{s}: no CDS with a translation in {gbk}")
        continue
    (FUNC / "proteins" / f"{s}.faa").write_text("".join(faa))
    with open(FUNC / "proteins" / f"{s}.idmap.tsv", "w", newline="") as f:
        w = csv.writer(f, delimiter="\t", lineterminator="\n")
        w.writerow(["protein_id", "locus_tag", "contig", "start", "end", "strand", "gene", "product"])
        w.writerows(idmap)
    g = gate.get(s, {})
    initial = f"{g.get('genus_meta', a.get('genus', ''))} {g.get('species_meta', a.get('species', ''))}".strip()
    if s in ident:
        genome_id = ident[s]["genome_based_identification"]
    elif g.get("genus_final"):
        genome_id = f"{g['genus_final']} {g['species_final']}"
    else:
        genome_id = "unresolved (not matched by the species check)" if g else ""
    rows.append({"short_name": s, "genome_fna": str(fna), "proteins_faa": str(FUNC / "proteins" / f"{s}.faa"),
                 "n_cds": len(faa), "n_contigs": len(counter), "annotation_source": a.get("source", ""),
                 "gate_status": g.get("qc_status", ""), "in_analysis_set": g.get("include", ""),
                 "initial_identification": initial, "genome_based_identification": genome_id})

if not rows:
    raise SystemExit("ERROR: no isolate has both an annotation and a standardized genome.\n  " + "\n  ".join(problems))
rows.sort(key=lambda r: r["short_name"])
with open(FUNC / "isolates.tsv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()), delimiter="\t", lineterminator="\n")
    w.writeheader()
    w.writerows(rows)

print(f"{'isolate':8s} {'CDS':>6s} {'contigs':>7s}  {'gate':9s}  genome-based identification")
for r in rows:
    print(f"{r['short_name']:8s} {r['n_cds']:>6d} {r['n_contigs']:>7d}  {r['gate_status']:9s}  {r['genome_based_identification'][:70]}")
if problems:
    print("\nLeft out:")
    for p in problems:
        print("  " + p)
print(f"\n{len(rows)} isolates -> {FUNC/'isolates.tsv'}   (array size for the next cell: 1-{len(rows)})")
END_OF_SCRIPT
chmod +x "$SCRIPTS/functional_inputs.py.new" && mv -f "$SCRIPTS/functional_inputs.py.new" "$SCRIPTS/functional_inputs.py"
echo "wrote $SCRIPTS/functional_inputs.py"

use_env download_env
if [ -s "$GATE" ]; then
    python "$SCRIPTS/make_identity_table.py" > /dev/null && echo "identification table: $DOWNSTREAM/species_check/supplement_isolate_identity.tsv"
else
    echo "NOTE: the species gate (D1.2) has not been run: gate_status and identification will be empty in the Part F tables."
fi
python "$SCRIPTS/functional_inputs.py"

### F2. Run the screens (one SLURM array task per isolate)
Each task runs six steps for one isolate (`dbcan`, `genomad`, `mob`, `txss`, `abricate`, `et3`) and records `DONE`, `FAILED` or `SKIPPED` for each, so one broken tool never costs the others. 6 isolates run at a time; allow roughly 1–3 hours per isolate (geNomad and dbCAN take most of it). Re-running is cheap: finished steps are skipped. To redo only some steps, change `TOOLS` (for example `TOOLS="et3"`); `FORCE=1` also redoes steps that are `DONE`.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/run_functional.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Part F array task: the comparative-genomic screens for ONE isolate (row $SLURM_ARRAY_TASK_ID
# of functional/isolates.tsv). Every tool is its own step; a step that fails is recorded and the
# next one still runs, so one broken tool never costs the other results.
#
#   dbcan     dbCAN v5 CAZyme annotation (proteins)             -> functional/dbcan/<isolate>/overview.tsv
#   genomad   geNomad: viruses / proviruses and plasmids        -> functional/genomad/<isolate>/
#   mob       MOB-suite mob_recon: replicons, relaxases, mobility -> functional/mobsuite/<isolate>/
#   txss      MacSyFinder + TXSScan: secretion systems          -> functional/macsyfinder/<isolate>/
#   abricate  ABRicate: VFDB, BacMet2, PlasmidFinder            -> functional/abricate/<isolate>.<db>.tsv
#   et3       EffectiveT3 type III effector candidates (soft)   -> functional/effectiveT3/<isolate>.raw.tsv
#
# Re-running skips steps already DONE. Environment switches (set them in the submit cell):
#   TOOLS=dbcan,txss   only these steps          FORCE=1   redo steps even if DONE
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"

IDX="${SLURM_ARRAY_TASK_ID:?run this as an array job (see F2)}"
CPUS="${SLURM_CPUS_PER_TASK:-8}"
S=$(awk -F'\t' -v n="$IDX" 'NR == n + 1 {print $1}' "$FUNC/isolates.tsv")
[ -n "$S" ] || { echo "ERROR: no row $IDX in $FUNC/isolates.tsv"; exit 2; }
FNA="$GENOMES/$S.fna"
FAA="$FUNC/proteins/$S.faa"
[ -s "$FNA" ] && [ -s "$FAA" ] || { echo "ERROR: $FNA or $FAA missing (re-run F1)"; exit 2; }
STAT="$FUNC/status"; mkdir -p "$STAT"
TOOLS="${TOOLS:-dbcan,genomad,mob,txss,abricate,et3}"
echo "Part F: $S  (array task $IDX, $CPUS cpus, steps: $TOOLS)  $(date)"

TOOL=""
why()  { printf '%s\n' "$*" > "$STAT/$S.$TOOL.why"; }
need() { command -v "$1" >/dev/null 2>&1 || { why "$1 not found after activating $2: run F0"; return 11; }; }

# ---------------------------------------------------------------------- steps
step_dbcan() {
    use_env dbcan_env; need run_dbcan dbcan_env || return 11
    [ -e "$FUNC_DB/.done_dbcan" ] || { why "dbCAN database not complete: run F0 again (it shows what failed)"; return 11; }
    local out="$FUNC/dbcan/$S"; rm -rf "$out"; mkdir -p "$out"
    run_dbcan CAZyme_annotation --input_raw_data "$FAA" --mode protein \
        --output_dir "$out" --db_dir "$FUNC_DB/dbcan_db" --threads "$CPUS" || return $?
    [ -s "$out/overview.tsv" ] || { why "dbCAN finished but wrote no overview.tsv"; return 12; }
}

step_genomad() {
    use_env genomad_env; need genomad genomad_env || return 11
    [ -e "$FUNC_DB/.done_genomad" ] || { why "geNomad database not complete: run F0 again (it shows what failed)"; return 11; }
    local out="$FUNC/genomad/$S"; rm -rf "$out"
    genomad end-to-end --cleanup --splits 4 --threads "$CPUS" "$FNA" "$out" "$FUNC_DB/genomad_db" || return $?
    [ -e "$out/${S}_summary/${S}_virus_summary.tsv" ] || { why "geNomad finished but wrote no virus summary"; return 12; }
}

step_mob() {
    use_env mobsuite_env; need mob_recon mobsuite_env || return 11
    [ -e "$FUNC_DB/.done_mobsuite" ] || { why "MOB-suite database not complete: run F0 again (it shows what failed)"; return 11; }
    local out="$FUNC/mobsuite/$S"; rm -rf "$out"
    mob_recon --infile "$FNA" --outdir "$out" --sample_id "$S" --num_threads "$CPUS" \
        --database_directory "$FUNC_DB/mob_db" || return $?
    [ -e "$out/contig_report.txt" ] || { why "MOB-suite finished but wrote no contig_report.txt"; return 12; }
}

step_txss() {
    use_env macsy_env; need macsyfinder macsy_env || return 11; need hmmsearch macsy_env || return 11
    [ -e "$FUNC_DB/.done_txsscan" ] && [ -d "$FUNC_DB/macsy_models/TXSScan" ] || { why "TXSScan models not complete: run F0 again (it shows what failed)"; return 11; }
    local out="$FUNC/macsyfinder/$S"; rm -rf "$out"; mkdir -p "$FUNC/macsyfinder"
    rm -f "$FAA".idx "$FAA".*.idx            # MacSyFinder's sequence index must match the current protein file
    # gembase = one replicon per contig, genes in annotation order (IDs are <contig>_<gene number>)
    macsyfinder --sequence-db "$FAA" --db-type gembase --replicon-topology linear \
        --models-dir "$FUNC_DB/macsy_models" --models TXSScan all \
        -o "$out" --worker "$CPUS" --mute || return $?
    [ -e "$out/best_solution.tsv" ] || { why "MacSyFinder finished but wrote no best_solution.tsv"; return 12; }
}

step_abricate() {
    local out="$FUNC/abricate"; mkdir -p "$out"
    local spec db mi mc failed=""
    for spec in "vfdb:$ABR_MINID:$ABR_MINCOV" "bacmet2:$ABR_MINID:$ABR_MINCOV" "plasmidfinder:$PLF_MINID:$PLF_MINCOV"; do
        IFS=: read -r db mi mc <<< "$spec"
        if apptainer exec --bind "$ROOT":"$ROOT" "$ABRICATE_IMG" \
               abricate --db "$db" --minid "$mi" --mincov "$mc" --threads "$CPUS" "$FNA" > "$out/$S.$db.tsv.part"; then
            mv "$out/$S.$db.tsv.part" "$out/$S.$db.tsv"
        else
            rm -f "$out/$S.$db.tsv.part"; failed="$failed $db"          # the other databases still run
        fi
    done
    [ -z "$failed" ] || { why "abricate failed for:$failed (is the database in the image? F0 checks)"; return 21; }
}

# Java: the host's if there is one, otherwise the one inside the ASA3P image (which ASA3P itself uses)
run_java() {
    if command -v java >/dev/null 2>&1; then
        java "$@"
    else
        apptainer exec --bind "$ROOT":"$ROOT" "$ASAP_IMG" bash -c 'java "$@"' java "$@"
    fi
}

step_et3() {
    local jar="$ET3_DIR/TTSS_GUI-1.0.1.jar"
    if [ ! -s "$jar" ] || [ ! -s "$ET3_DIR/module/$ET3_MODEL" ]; then
        why "EffectiveT3 is not installed in $ET3_DIR (F0 downloads it)"; return 77
    fi
    local out="$FUNC/effectiveT3"; mkdir -p "$out"
    local raw="$out/$S.raw.tsv" log="$out/$S.log" model rc=1
    : > "$log"
    # EffectiveT3 must run from the folder that contains module/. Its model argument is the jar's
    # file name in the Galaxy wrapper; the other spellings are tried in case a version wants them.
    for model in "$ET3_MODEL" "module/$ET3_MODEL" "${ET3_MODEL%.jar}"; do
        rm -f "$raw"
        ( cd "$ET3_DIR" && run_java -jar "$jar" -f "$FAA" -m "$model" -t "$ET3_CUTOFF" -o "$raw" -q ) >> "$log" 2>&1
        rc=$?
        if [ $rc -eq 0 ] && [ -s "$raw" ]; then
            echo "model argument that worked: -m $model" >> "$log"; return 0
        fi
        echo "attempt -m $model: rc=$rc, output $( [ -s "$raw" ] && echo present || echo empty )" >> "$log"
    done
    why "EffectiveT3 did not run with any model argument (see $log)"; return 77   # soft: counts as SKIPPED
}

# ------------------------------------------------------------------- driver
ANYFAIL=0
run_step() {   # run_step <tool-key> <function> <required: 1|0>
    local key=$1 fn=$2 req=$3
    case ",$TOOLS," in *",$key,"*) ;; *) return 0 ;; esac
    if [ "${FORCE:-0}" != 1 ] && grep -q '^DONE' "$STAT/$S.$key" 2>/dev/null; then
        echo "=== [$S] $key: already DONE, skipped"; return 0
    fi
    TOOL=$key
    rm -f "$STAT/$S.$key" "$STAT/$S.$key.why"
    echo "=== [$S] $key  $(date '+%F %T')"
    ( "$fn" ); local rc=$?
    local reason=""; [ -s "$STAT/$S.$key.why" ] && reason=$(head -c 300 "$STAT/$S.$key.why")
    if [ $rc -ne 0 ] && [ $rc -ne 77 ] && [ -z "$reason" ]; then
        reason="the tool exited with an error; see logs/func_${SLURM_ARRAY_JOB_ID:-${SLURM_JOB_ID:-?}}_${IDX}.err"
    fi
    if   [ $rc -eq 0 ];  then printf 'DONE\t%s\n' "$(date '+%F %T')" > "$STAT/$S.$key"
    elif [ $rc -eq 77 ]; then printf 'SKIPPED\t%s\n' "$reason" > "$STAT/$S.$key"
    else printf 'FAILED\trc=%s\t%s\n' "$rc" "$reason" > "$STAT/$S.$key"; [ "$req" = 1 ] && ANYFAIL=1
    fi
    echo "=== [$S] $key -> $(cut -f1 "$STAT/$S.$key")  $reason"
}

run_step dbcan    step_dbcan    1
run_step genomad  step_genomad  1
run_step mob      step_mob      1
run_step txss     step_txss     1
run_step abricate step_abricate 1
run_step et3      step_et3      0

echo "Part F finished: $S  (any required step failed: $ANYFAIL)  $(date)"
exit "$ANYFAIL"
END_OF_SCRIPT
chmod +x "$SCRIPTS/run_functional.sh.new" && mv -f "$SCRIPTS/run_functional.sh.new" "$SCRIPTS/run_functional.sh"
echo "wrote $SCRIPTS/run_functional.sh"

[ -s "$FUNC/setup_status.tsv" ] || { echo "Run F0 first (no $FUNC/setup_status.tsv)."; exit 1; }
[ -s "$FUNC/isolates.tsv" ]     || { echo "Run F1 first (no $FUNC/isolates.tsv)."; exit 1; }
grep -q -E "FAILED|MISSING" "$FUNC/setup_status.tsv" && echo "NOTE: some setup items failed (F0); the tools that need them will be recorded as FAILED or SKIPPED."

TOOLS="dbcan,genomad,mob,txss,abricate,et3"     # e.g. TOOLS="et3" to run one step only
FORCE=0                                          # 1 = redo steps that are already DONE
export TOOLS FORCE

N=$(( $(wc -l < "$FUNC/isolates.tsv") - 1 ))
JOB=$(SBATCH_EXTRA="--array=1-$N%6" submit func 8 48G 12:00:00 "$SCRIPTS/run_functional.sh")
echo "submitted array job $JOB: $N isolates, steps: $TOOLS"

### F3. Monitor
One row per isolate, one column per tool: `DONE`, `FAILED`, `SKIPPED`, or `-` (not run yet). The list underneath gives the reason for every step that is not `DONE`. A log for each task is in `logs/func_<jobid>_<task>.out`.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

jobs_like " func " | tail -n 30
echo
tools="dbcan genomad mob txss abricate et3"
printf '%-9s' isolate; for t in $tools; do printf '%-10s' "$t"; done; echo
tail -n +2 "$FUNC/isolates.tsv" | cut -f1 | while read -r s; do
    printf '%-9s' "$s"
    for t in $tools; do
        f="$FUNC/status/$s.$t"
        if [ -s "$f" ]; then printf '%-10s' "$(cut -f1 "$f")"; else printf '%-10s' "-"; fi
    done
    echo
done
echo
echo "Not DONE (reason):"
n=0
for f in "$FUNC"/status/*; do
    case "$f" in *.why) continue ;; esac
    [ -e "$f" ] || continue
    [ "$(cut -f1 "$f")" = DONE ] && continue
    echo "  $(basename "$f"): $(tr '\t' ' ' < "$f")"; n=$((n + 1))
done
[ "$n" -gt 0 ] || echo "  (none)"

### F4. Tables for the thesis
Run when F3 shows every step finished (steps that are not `DONE` simply give `NA`). Writes `functional/tables/`: `functional_summary.tsv` (**one row per isolate**: the numbers to quote), `functional_status.tsv`, and the long tables behind it (CAZyme genes and families, secretion systems, prophages, plasmids, VFDB and BacMet2 hits, effector candidates). Re-run after any F2 re-run.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/summarize_functional.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Part F: turn the per-isolate tool outputs into tables for the thesis.

Reads  DOWNSTREAM/functional/{status,dbcan,macsyfinder,genomad,mobsuite,abricate,effectiveT3}
Writes DOWNSTREAM/functional/tables/*.tsv

Rules that keep the numbers honest:
  * A tool that did not finish for an isolate gives NA for that isolate, never 0.
  * Every table carries the isolate's species-check status and genome-based identification,
    so isolates that did not pass the species check can be flagged or dropped at plotting time.
  * Parsers read column names from each file's header and ignore the rest, so a new tool
    version that adds a column does not break them.

Tables
  functional_status.tsv        isolate x tool: DONE / FAILED / SKIPPED / NOT_RUN (+ reason)
  functional_summary.tsv       ONE ROW PER ISOLATE: the numbers to quote in Results
  cazyme_families_long.tsv     isolate, CAZy family, number of genes
  cazyme_genes.tsv             every CAZyme gene with its families, PCWDE class and product
  pcwde_family_map.tsv         the family -> pectin / cellulose / hemicellulose assignment used
  secretion_systems_long.tsv   one row per system found by MacSyFinder/TXSScan
  secretion_systems_matrix.tsv isolate x system type (number of complete systems)
  genomad_viruses.tsv / genomad_plasmids.tsv   per-sequence geNomad calls
  mob_contigs.tsv              per-contig MOB-suite replicon assignment
  plasmidfinder_hits.tsv       PlasmidFinder replicon hits (ABRicate)
  vfdb_hits.tsv, bacmet2_hits.tsv + *_matrix.tsv   ABRicate hits and gene presence/absence
  effector_candidates.tsv      EffectiveT3 positives, with the annotation's locus tag and product
"""
import csv
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
FUNC = Path(os.environ.get("FUNC") or DOWNSTREAM / "functional")
META = Path(os.environ.get("META", DOWNSTREAM.parent / "metadata"))
OUT = FUNC / "tables"
OUT.mkdir(parents=True, exist_ok=True)
NA = "NA"
TOOLS = ["dbcan", "txss", "genomad", "mob", "abricate", "et3"]
TOOL_LABEL = {"dbcan": "dbCAN", "txss": "MacSyFinder/TXSScan", "genomad": "geNomad",
              "mob": "MOB-suite", "abricate": "ABRicate (VFDB, BacMet2, PlasmidFinder)", "et3": "EffectiveT3"}

# CAZy families -> plant-cell-wall-degrading class. A family-level assignment: activity differs between
# members of one family, so these are candidates ("PCWDE-family genes"), not proven enzymes.
PCWDE = {
    "pectin": "PL1 PL2 PL3 PL4 PL9 PL10 PL11 PL22 PL26 PL42 GH28 GH78 GH88 GH105 GH106 CE8 CE12 CE13".split(),
    "cellulose": "GH5 GH6 GH8 GH9 GH12 GH44 GH45 GH48 GH74 AA9 AA10".split(),
    "hemicellulose": "GH10 GH11 GH26 GH30 GH39 GH43 GH51 GH53 GH54 GH62 GH67 GH115 GH120 CE1 CE2 CE3 CE4 CE5 CE6 CE7".split(),
}
FAM2CLASS = {fam: cls for cls, fams in PCWDE.items() for fam in fams}


def read_rows(path, key=None):
    """Header-driven TSV reader. Lines starting with '#' are comments, except a '#'-prefixed header
    (ABRicate writes '#FILE'). With key=<column name>, the header is the first line that has that column.
    Only the first field loses a leading '#': a column called '#ofTools' (dbCAN) keeps its name."""
    rows, header = [], None
    with open(path, newline="") as f:
        for line in f:
            line = line.rstrip("\r\n")
            if not line.strip():
                continue
            fields = [h.strip() for h in line.split("\t")]
            first_stripped = [fields[0].lstrip("#").strip()] + fields[1:]
            if header is None:
                if key is not None:
                    if key in first_stripped:
                        header = first_stripped
                elif len(fields) >= 2 or not line.startswith("#"):
                    header = first_stripped
                continue
            if line.startswith("#") or first_stripped == header:
                continue
            parts = line.split("\t")
            rows.append(dict(zip(header, parts + [""] * (len(header) - len(parts)))))
    return rows


def col(row, *names):
    """First matching column, ignoring case and '(s)' suffixes."""
    norm = {re.sub(r"\(s\)$", "", k).strip().lower(): k for k in row}
    for n in names:
        k = norm.get(n.lower())
        if k is not None:
            return row[k].strip()
    return ""


def num(x, default=0.0):
    try:
        return float(x)
    except (TypeError, ValueError):
        return default


def write(name, header, rows):
    with open(OUT / name, "w", newline="") as f:
        w = csv.writer(f, delimiter="\t", lineterminator="\n")
        w.writerow(header)
        w.writerows(rows)
    return OUT / name


# ------------------------------------------------------------------ isolates
iso_path = FUNC / "isolates.tsv"
if not iso_path.exists():
    sys.exit(f"ERROR: {iso_path} not found: run F1 first.")
isolates = read_rows(iso_path)
names = [r["short_name"] for r in isolates]
ident = {r["short_name"]: r for r in isolates}

extra = {}   # optional context from the metadata table (lineage, source class); no grower or location columns
for cand in (META / "source_metadata_correlated.tsv",):
    if cand.exists():
        for r in read_rows(cand):
            extra[r.get("workflow_short_name", "")] = {k: r.get(k, "") for k in ("lineage", "source_class", "environment_type")}


def idcols(s):
    r = ident[s]
    e = extra.get(s, {})
    return [r.get("initial_identification", ""), r.get("genome_based_identification", ""), r.get("gate_status", ""),
            e.get("lineage", ""), e.get("source_class", ""), e.get("environment_type", "")]


IDHEAD = ["isolate", "initial_identification", "genome_based_identification", "gate_status", "lineage", "source_class", "environment_type"]

# -------------------------------------------------------------------- status
status = {}
for s in names:
    for t in TOOLS:
        p = FUNC / "status" / f"{s}.{t}"
        if p.exists():
            parts = p.read_text().strip().split("\t")
            status[(s, t)] = (parts[0], " ".join(parts[1:]).strip())
        else:
            status[(s, t)] = ("NOT_RUN", "")
done = lambda s, t: status[(s, t)][0] == "DONE"
write("functional_status.tsv", IDHEAD + [TOOL_LABEL[t] for t in TOOLS] + ["notes"],
      [[s] + idcols(s) + [status[(s, t)][0] for t in TOOLS] +
       ["; ".join(f"{t}: {status[(s, t)][1]}" for t in TOOLS if status[(s, t)][0] in ("FAILED", "SKIPPED") and status[(s, t)][1])]
       for s in names])

summary = {s: {} for s in names}   # isolate -> column -> value
warnings = []

# --------------------------------------------------------------------- dbCAN
write("pcwde_family_map.tsv", ["family", "pcwde_class", "note"],
      [[fam, cls, "family-level assignment; activity varies within a family"] for cls, fams in PCWDE.items() for fam in fams])
FAMRE = re.compile(r"(GH|GT|PL|CE|AA|CBM)(\d+)")
fam_rows, gene_rows = [], []
CLASSES = ["GH", "GT", "PL", "CE", "AA", "CBM"]
for s in names:
    ov = FUNC / "dbcan" / s / "overview.tsv"
    if not (done(s, "dbcan") and ov.exists()):
        continue
    idmap = {}
    p = FUNC / "proteins" / f"{s}.idmap.tsv"
    if p.exists():
        idmap = {r["protein_id"]: r for r in read_rows(p)}
    genes, fam_count = [], Counter()
    for r in read_rows(ov, key="Gene ID"):
        gid = col(r, "Gene ID")
        ntools = int(num(col(r, "#ofTools"), 0))
        rec = col(r, "Recommend Results")
        if rec in ("", "-"):
            rec = " ".join(col(r, k) for k in ("dbCAN_hmm", "dbCAN_sub", "DIAMOND") if col(r, k) not in ("", "-"))
        fams = sorted({m.group(1) + m.group(2) for m in FAMRE.finditer(rec)})
        genes.append((gid, ntools, fams))
    cazy = [g for g in genes if g[1] >= 2 and g[2]]        # dbCAN's own recommendation: supported by >= 2 of the 3 tools
    anyc = [g for g in genes if g[2]]
    cls_count = Counter()
    pc = {"pectin": set(), "cellulose": set(), "hemicellulose": set()}
    for gid, nt, fams in cazy:
        for c in {re.match(r"[A-Z]+", f).group(0) for f in fams}:
            cls_count[c] += 1
        for f in fams:
            fam_count[f] += 1
        gclasses = sorted({FAM2CLASS[f] for f in fams if f in FAM2CLASS})
        for c in gclasses:
            pc[c].add(gid)
        gene_rows.append([s, gid, nt, "+".join(fams), ",".join(gclasses), idmap.get(gid, {}).get("locus_tag", ""),
                          idmap.get(gid, {}).get("contig", ""), idmap.get(gid, {}).get("product", "")])
    for f, n in sorted(fam_count.items()):
        fam_rows.append([s, f, n])
    d = summary[s]
    d["cazymes_total"] = len(cazy)
    d["cazymes_any_tool"] = len(anyc)
    for c in CLASSES:
        d[f"cazy_{c}"] = cls_count.get(c, 0)
    for c in ("pectin", "cellulose", "hemicellulose"):
        d[f"pcwde_{c}"] = len(pc[c])
    d["pcwde_total"] = len(pc["pectin"] | pc["cellulose"] | pc["hemicellulose"])
write("cazyme_families_long.tsv", ["isolate", "cazy_family", "n_genes"], fam_rows)
write("cazyme_genes.tsv", ["isolate", "protein_id", "n_tools", "cazy_families", "pcwde_class", "locus_tag", "contig", "product"], gene_rows)

# ------------------------------------------------- MacSyFinder / TXSScan
sys_rows = []
types = set()
for s in names:
    bs = FUNC / "macsyfinder" / s / "best_solution.tsv"
    if not (done(s, "txss") and bs.exists()):
        continue
    found = {}
    for r in read_rows(bs, key="sys_id"):
        sid = col(r, "sys_id")
        if not sid:
            continue
        fqn = col(r, "model_fqn")
        typ = fqn.rstrip("/").split("/")[-1] if fqn else "unknown"
        d = found.setdefault(sid, {"type": typ, "replicon": col(r, "replicon"), "wholeness": col(r, "sys_wholeness"),
                                   "score": col(r, "sys_score"), "genes": []})
        d["genes"].append(col(r, "gene_name"))
    for sid, d in sorted(found.items()):
        sys_rows.append([s, d["type"], sid, d["replicon"], d["wholeness"], d["score"], len(d["genes"]), ",".join(d["genes"])])
        types.add(d["type"])
    summary[s]["secretion_systems_total"] = len(found)
    t3 = [d for d in found.values() if d["type"].upper().startswith("T3SS")]
    summary[s]["T3SS_systems"] = len(t3)
    summary[s]["T6SS_systems"] = sum(1 for d in found.values() if d["type"].upper().startswith("T6SS"))
    summary[s]["T2SS_systems"] = sum(1 for d in found.values() if d["type"].upper().startswith("T2SS"))
    summary[s]["T4SS_systems"] = sum(1 for d in found.values() if "T4SS" in d["type"].upper())
    summary[s]["_systems"] = Counter(d["type"] for d in found.values())
write("secretion_systems_long.tsv", ["isolate", "system_type", "system_id", "replicon_contig", "wholeness", "score", "n_genes", "genes"], sys_rows)
tl = sorted(types)
write("secretion_systems_matrix.tsv", IDHEAD + tl,
      [[s] + idcols(s) + [summary[s]["_systems"].get(t, 0) if "_systems" in summary[s] else NA for t in tl] for s in names])

# ------------------------------------------------------------------ geNomad
vir_rows, pla_rows = [], []
for s in names:
    base = FUNC / "genomad" / s / f"{s}_summary"
    vf, pf = base / f"{s}_virus_summary.tsv", base / f"{s}_plasmid_summary.tsv"
    if not (done(s, "genomad") and vf.exists()):
        continue
    nprov = nvir = vbp = 0
    for r in read_rows(vf):
        name = col(r, "seq_name")
        prov = "|provirus_" in name
        nprov += prov
        nvir += not prov
        vbp += int(num(col(r, "length")))
        vir_rows.append([s, name, "provirus" if prov else "virus contig", col(r, "length"), col(r, "topology"),
                         col(r, "virus_score"), col(r, "n_hallmarks"), col(r, "taxonomy")])
    npla = pbp = 0
    if pf.exists():
        for r in read_rows(pf):
            npla += 1
            pbp += int(num(col(r, "length")))
            pla_rows.append([s, col(r, "seq_name"), col(r, "length"), col(r, "topology"), col(r, "plasmid_score"),
                             col(r, "conjugation_genes"), col(r, "amr_genes")])
    d = summary[s]
    d["genomad_proviruses"], d["genomad_virus_contigs"], d["genomad_virus_bp"] = nprov, nvir, vbp
    d["genomad_plasmid_seqs"], d["genomad_plasmid_bp"] = npla, pbp
write("genomad_viruses.tsv", ["isolate", "seq_name", "kind", "length", "topology", "virus_score", "n_hallmarks", "taxonomy"], vir_rows)
write("genomad_plasmids.tsv", ["isolate", "seq_name", "length", "topology", "plasmid_score", "conjugation_genes", "amr_genes"], pla_rows)

# ---------------------------------------------------------------- MOB-suite
mob_rows = []
for s in names:
    cr = FUNC / "mobsuite" / s / "contig_report.txt"
    if not (done(s, "mob") and cr.exists()):
        continue
    pl = [r for r in read_rows(cr) if col(r, "molecule_type").lower() == "plasmid"]
    clusters = {col(r, "primary_cluster_id") for r in pl if col(r, "primary_cluster_id") not in ("", "-")}
    mob = Counter(col(r, "predicted_mobility").lower() or "unknown" for r in pl)
    d = summary[s]
    d["mob_plasmid_contigs"] = len(pl)
    d["mob_plasmid_clusters"] = len(clusters)
    d["mob_plasmid_bp"] = int(sum(num(col(r, "size")) for r in pl))
    d["mob_conjugative"] = mob.get("conjugative", 0)
    d["mob_mobilizable"] = mob.get("mobilizable", 0)
    for r in read_rows(cr):
        mob_rows.append([s, col(r, "contig_id"), col(r, "molecule_type"), col(r, "primary_cluster_id"), col(r, "size"),
                         col(r, "rep_type"), col(r, "relaxase_type"), col(r, "predicted_mobility"), col(r, "mash_nearest_neighbor")])
write("mob_contigs.tsv", ["isolate", "contig", "molecule_type", "plasmid_cluster", "size", "rep_type", "relaxase_type", "predicted_mobility", "mash_nearest_neighbor"], mob_rows)

# ------------------------------------------------------------------ ABRicate
def read_abricate(s, db):
    p = FUNC / "abricate" / f"{s}.{db}.tsv"
    return read_rows(p) if p.exists() else None


ab_rows = {"vfdb": [], "bacmet2": [], "plasmidfinder": []}
ab_genes = {"vfdb": defaultdict(set), "bacmet2": defaultdict(set)}
ab_ok = {db: set() for db in ab_rows}           # isolates whose result file for that database exists
for s in names:
    for db in ab_rows:                           # per database: one failing database does not hide the others
        rows = read_abricate(s, db)
        if rows is None:
            continue
        ab_ok[db].add(s)
        for r in rows:
            ab_rows[db].append([s, col(r, "SEQUENCE"), col(r, "START"), col(r, "END"), col(r, "STRAND"), col(r, "GENE"),
                                col(r, "%COVERAGE"), col(r, "%IDENTITY"), col(r, "ACCESSION"), col(r, "PRODUCT"), col(r, "RESISTANCE")])
            if db in ab_genes:
                ab_genes[db][s].add(col(r, "GENE"))
        genes = {col(r, "GENE") for r in rows}
        summary[s][{"vfdb": "vfdb_genes", "bacmet2": "bacmet2_genes", "plasmidfinder": "plasmidfinder_replicons"}[db]] = len(genes)
        if db == "plasmidfinder":
            summary[s]["plasmidfinder_names"] = ",".join(sorted(genes))
AHEAD = ["isolate", "contig", "start", "end", "strand", "gene", "pct_coverage", "pct_identity", "accession", "product", "resistance"]
write("vfdb_hits.tsv", AHEAD, ab_rows["vfdb"])
write("bacmet2_hits.tsv", AHEAD, ab_rows["bacmet2"])
write("plasmidfinder_hits.tsv", AHEAD, ab_rows["plasmidfinder"])
for db, label in (("vfdb", "vfdb"), ("bacmet2", "bacmet2")):
    allg = sorted({g for v in ab_genes[db].values() for g in v})
    write(f"{label}_matrix.tsv", IDHEAD + allg,
          [[s] + idcols(s) + [(1 if g in ab_genes[db][s] else 0) if s in ab_ok[db] else NA for g in allg] for s in names])

# --------------------------------------------------------------- EffectiveT3
eff_rows = []
for s in names:
    raw = FUNC / "effectiveT3" / f"{s}.raw.tsv"
    if not (done(s, "et3") and raw.exists()):
        continue
    idmap = {}
    p = FUNC / "proteins" / f"{s}.idmap.tsv"
    if p.exists():
        idmap = {r["protein_id"]: r for r in read_rows(p)}
    n_total = n_pos = n_flag = 0
    for line in raw.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or re.match(r"(?i)^id\s*;", line):
            continue
        f = [x.strip() for x in line.split(";")]
        if len(f) < 3:
            continue
        n_total += 1
        flag = f[3].lower() if len(f) > 3 and f[3] else ""
        if flag in ("true", "false", "yes", "no", "1", "0"):
            n_flag += 1
        if flag in ("true", "yes", "1"):
            n_pos += 1
            pid = f[0].split()[0]
            eff_rows.append([s, pid, f[2], idmap.get(pid, {}).get("locus_tag", ""), idmap.get(pid, {}).get("product", "")])
    if n_total and n_flag == 0:
        warnings.append(f"{s}: EffectiveT3 output has no true/false column; raw file kept, effector count left as NA")
        continue
    summary[s]["effector_candidates"] = n_pos
write("effector_candidates.tsv", ["isolate", "protein_id", "score", "locus_tag", "product"], eff_rows)

# ------------------------------------------------------------ master table
COLS = (["n_cds"] + ["cazymes_total", "cazymes_any_tool"] + [f"cazy_{c}" for c in CLASSES] +
        ["pcwde_pectin", "pcwde_cellulose", "pcwde_hemicellulose", "pcwde_total"] +
        ["secretion_systems_total", "T2SS_systems", "T3SS_systems", "T4SS_systems", "T6SS_systems"] +
        ["genomad_proviruses", "genomad_virus_contigs", "genomad_virus_bp", "genomad_plasmid_seqs", "genomad_plasmid_bp"] +
        ["mob_plasmid_contigs", "mob_plasmid_clusters", "mob_plasmid_bp", "mob_conjugative", "mob_mobilizable",
         "plasmidfinder_replicons", "plasmidfinder_names"] +
        ["vfdb_genes", "bacmet2_genes", "effector_candidates"])
rows = []
for s in names:
    summary[s]["n_cds"] = ident[s].get("n_cds", NA)
    rows.append([s] + idcols(s) + [summary[s].get(c, NA) for c in COLS])
write("functional_summary.tsv", IDHEAD + COLS, rows)

# ------------------------------------------------------------------- report
print(f"Part F summary: {len(names)} isolates -> {OUT}")
print(f"\n{'isolate':8s} {'gate':9s} " + " ".join(f"{h:>8s}" for h in ("CAZymes", "PCWDE", "T3SS", "T6SS", "provir.", "plasmid", "VFDB", "BacMet", "effect.")))
for s in names:
    d = summary[s]
    g = lambda k: str(d.get(k, NA))
    plasmids = g("mob_plasmid_clusters")
    print(f"{s:8s} {ident[s].get('gate_status', ''):9s} " + " ".join(f"{v:>8s}" for v in (
        g("cazymes_total"), g("pcwde_total"), g("T3SS_systems"), g("T6SS_systems"), g("genomad_proviruses"),
        plasmids, g("vfdb_genes"), g("bacmet2_genes"), g("effector_candidates"))))
print("\nNA = that tool did not finish for that isolate (see functional_status.tsv).")
bad = [(s, t) for s in names for t in TOOLS if status[(s, t)][0] not in ("DONE",)]
if bad:
    print(f"\n{len(bad)} isolate/tool combinations are not DONE:")
    by_tool = defaultdict(list)
    for s, t in bad:
        by_tool[t].append(f"{s} [{status[(s, t)][0]}]")
    for t, lst in by_tool.items():
        print(f"  {TOOL_LABEL[t]}: {', '.join(lst[:12])}{' ...' if len(lst) > 12 else ''}")
for w in warnings:
    print("WARNING:", w)
END_OF_SCRIPT
chmod +x "$SCRIPTS/summarize_functional.py.new" && mv -f "$SCRIPTS/summarize_functional.py.new" "$SCRIPTS/summarize_functional.py"
echo "wrote $SCRIPTS/summarize_functional.py"

use_env download_env
python "$SCRIPTS/summarize_functional.py"

### F5. What you get
| File (in `downstream/functional/tables/`) | Contents |
|---|---|
| `functional_summary.tsv` | one row per isolate: initial and genome-based identification, `gate_status`, lineage and source class (if `source_metadata_correlated.tsv` is in `metadata/`), then the counts: CAZymes (≥ 2 of 3 dbCAN methods), CAZy classes, PCWDE-family genes (pectin, cellulose, hemicellulose), secretion systems (T2SS, T3SS, T4SS, T6SS), proviruses and virus contigs, plasmid contigs and clusters, VFDB and BacMet2 genes, effector candidates |
| `functional_status.tsv` | which tool finished for which isolate, with the reason if not |
| `cazyme_families_long.tsv`, `cazyme_genes.tsv`, `pcwde_family_map.tsv` | per-family counts; every CAZyme gene with its locus tag and product; the family → PCWDE class assignment used |
| `secretion_systems_long.tsv`, `secretion_systems_matrix.tsv` | every system found (with completeness) and an isolate × system-type count matrix |
| `genomad_viruses.tsv`, `genomad_plasmids.tsv`, `mob_contigs.tsv`, `plasmidfinder_hits.tsv` | per-sequence calls |
| `vfdb_hits.tsv`, `vfdb_matrix.tsv`, `bacmet2_hits.tsv`, `bacmet2_matrix.tsv` | ABRicate hits and gene presence/absence per isolate |
| `effector_candidates.tsv` | EffectiveT3 positives with the annotation's locus tag and product |

**For the thesis:** build the figures and tables from `functional_summary.tsv` and keep the rows with `gate_status` = `PASS` or `RELABELLED` for the *Pectobacteriaceae* comparisons; show the others in a separate, labelled panel or in the supplement. Re-run E1 and E2 afterwards so the Methods draft describes Part F.

In [ ]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

ls -lh "$FUNC/tables/" 2>/dev/null || echo "F4 has not been run yet"
echo
head -n 4 "$FUNC/tables/functional_summary.tsv" 2>/dev/null | cut -f1-3,8-14 | awk -F'\t' '{for (i = 1; i <= NF; i++) printf "%-16s", substr($i, 1, 15); printf "\n"}'

# Part E — Versions, Methods draft, outputs

## E1. Record versions of tools outside containers
IQ-TREE (module) and the conda tools, including GTDB-Tk (D4c) and the Part F tools (skipped quietly if their environment does not exist yet); container versions were recorded in A0.4.

In [20]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

cat > "$SCRIPTS/record_versions.sh.new" <<'END_OF_SCRIPT'
#!/bin/bash
# Versions of the tools that run outside containers (IQ-TREE module, conda envs),
# appended to the container versions for the Methods draft.
# A tool whose environment does not exist yet is skipped (not an error).
set -uo pipefail
source "${ASAP_CONFIG:-/home/s94m696/asap_shortname_projects/config.sh}"
OUT="$SOFT/other_versions.txt"

module load "$CONDA_MODULE" 2>/dev/null
have_env() { conda env list 2>/dev/null | awk '{print $1}' | grep -qx "$1"; }
first_version() { grep -o '[0-9][0-9]*\(\.[0-9][0-9]*\)\+' | head -1; }
pyver() { python -c 'import importlib.metadata as m, sys; print(m.version(sys.argv[1]))' "$1" 2>/dev/null; }

{
    module load "$IQTREE_MODULE" 2>/dev/null
    IQ=$(command -v iqtree2 || command -v iqtree)
    echo "iqtree:      $("$IQ" --version 2>&1 | grep -m1 -o 'version [0-9.]*' | cut -d' ' -f2)"

    if have_env genomeviz_env; then
        ( use_env genomeviz_env
          echo "pygenomeviz: $(python -c 'import pygenomeviz; print(pygenomeviz.__version__)' 2>/dev/null)"
          echo "mummer:      $(nucmer --version 2>&1 | tail -1)" )
    fi
    if have_env clinker_env; then
        ( use_env clinker_env
          echo "clinker:     $(clinker --version 2>&1 | tail -1 | first_version)" )
    fi

    # ---- Part F tools
    if have_env dbcan_env; then
        ( use_env dbcan_env
          v=$(pyver dbcan); [ -n "$v" ] || v=$(run_dbcan version 2>&1 | first_version)
          echo "dbcan:       $v" )
    fi
    if have_env macsy_env; then
        ( use_env macsy_env
          echo "macsyfinder: $(macsyfinder --version 2>&1 | first_version)"
          echo "txsscan:     $(grep -m1 -E '^vers' "$FUNC_DB/macsy_models/TXSScan/metadata.yml" 2>/dev/null | first_version)" )
    fi
    if have_env genomad_env; then
        ( use_env genomad_env
          echo "genomad:     $(genomad --version 2>&1 | first_version)"
          echo "genomad_db:  $(cat "$FUNC_DB/genomad_db/version.txt" 2>/dev/null | first_version)" )
    fi
    if have_env mobsuite_env; then
        ( use_env mobsuite_env
          echo "mob_suite:   $(mob_recon --version 2>&1 | first_version)" )
    fi
    if have_env gtdbtk_env; then
        ( use_env gtdbtk_env
          echo "gtdbtk:      $(gtdbtk --version 2>&1 | first_version)" )
    fi
    if [ -s "$ET3_DIR/TTSS_GUI-1.0.1.jar" ]; then
        echo "effectivet3: 1.0.1"
    fi
} | tee "$OUT"
END_OF_SCRIPT
chmod +x "$SCRIPTS/record_versions.sh.new" && mv -f "$SCRIPTS/record_versions.sh.new" "$SCRIPTS/record_versions.sh"
echo "wrote $SCRIPTS/record_versions.sh"

bash "$SCRIPTS/record_versions.sh"

wrote /home/s94m696/asap_shortname_projects/scripts/record_versions.sh
iqtree:      2.2.2.6
pygenomeviz: 
mummer:          
dbcan:       5.2.9
macsyfinder: 2.1.6
txsscan:     
genomad:     1.11.2
genomad_db:  
gtdbtk:      2.7.1


## E2. Methods draft (generated from this run)
Builds `downstream/methods_draft.md` from the records of this run:
1. **ASA³P paragraph** — only the ASA³P steps that produced output (and for which genus).
2. **Replacement paragraph** — only if something in Part C ran, with the reason for each.
3. **Outside-ASA³P paragraph** — only the Part D analyses whose outputs exist.
4. **Functional-screen paragraph** — only the Part F tools that finished, with the notes on isolates that did not pass the species check.

Unknown versions or tables are left as `[brackets]`, and a *Notes for the author* list says what to fill in or resolve. Re-run this cell after any change.

In [21]:
%%bash
source "/home/s94m696/asap_shortname_projects/config.sh"

# the scripts this cell uses are rewritten first, so an older copy on disk can never run
mkdir -p "$SCRIPTS"
cat > "$SCRIPTS/detect_asap_version.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Find the ASA3P version that is actually installed and used, instead of guessing.

Tried in order (first hit wins):
  1. the version stamped inside asap.jar (META-INF/MANIFEST.MF)
  2. a version line printed by ASA3P in its run logs (LOGS/asap_*.out|err,
     ROOT/ASA3P_<Genus>/*.log)
  3. ASAP_VERSION from config.sh (the Zenodo download version, 1.2.2) --
     marked "unconfirmed" so the Methods notes ask you to check it

Writes PROVENANCE/asap_version.txt as "<version>\t<source>" and prints it.
"""
import os
import re
import sys
import zipfile
from pathlib import Path

VERSION = re.compile(r"\b(\d+\.\d+\.\d+)\b")


def from_jar(jar):
    try:
        with zipfile.ZipFile(jar) as z:
            text = z.read("META-INF/MANIFEST.MF").decode("utf-8", "replace")
    except (OSError, KeyError, zipfile.BadZipFile):
        return None
    for line in text.splitlines():
        if re.match(r"(Implementation|Specification|Bundle)-Version\s*:", line, re.I):
            m = VERSION.search(line)
            if m:
                return m.group(1)
    return None


def from_logs(paths):
    pat = re.compile(r"(asa.?3?p|asap)[^\n]{0,40}?\bv(?:ersion)?[\s:=]*v?(\d+\.\d+\.\d+)", re.I)
    for p in paths:
        try:
            with open(p, errors="replace") as f:
                for i, line in enumerate(f):
                    m = pat.search(line)
                    if m:
                        return m.group(2), p
                    if i > 500:          # the version is printed near the top
                        break
        except OSError:
            continue
    return None, None


def detect():
    root, logs = Path(os.environ["ROOT"]), Path(os.environ["LOGS"])
    jar = Path(os.environ["ASAPDIR"]) / "asap.jar"
    v = from_jar(jar)
    if v:
        return v, f"asap.jar manifest ({jar})"
    logfiles = sorted(logs.glob("asap_*.out")) + sorted(logs.glob("asap_*.err")) + sorted(root.glob("ASA3P_*/*.log"))
    v, src = from_logs(logfiles)
    if v:
        return v, f"ASA3P run log ({src})"
    fallback = os.environ.get("ASAP_VERSION", "").strip()
    if fallback:
        return fallback, "unconfirmed: Zenodo download version from config.sh"
    return "", "unknown"


if __name__ == "__main__":
    version, source = detect()
    prov = Path(os.environ["PROVENANCE"])
    prov.mkdir(parents=True, exist_ok=True)
    (prov / "asap_version.txt").write_text(f"{version}\t{source}\n")
    print(f"ASA3P version: {version or '[unknown]'}   (source: {source})")
    if source.startswith("unconfirmed"):
        print("Not found in asap.jar or the run logs yet. Re-run this after ASA3P has run; "
              "if it still says unconfirmed, 1.2.2 (your Zenodo download) is the value to report.")
    sys.exit(0)
END_OF_SCRIPT
chmod +x "$SCRIPTS/detect_asap_version.py.new" && mv -f "$SCRIPTS/detect_asap_version.py.new" "$SCRIPTS/detect_asap_version.py"
echo "wrote $SCRIPTS/detect_asap_version.py"

cat > "$SCRIPTS/write_methods.py.new" <<'END_OF_SCRIPT'
#!/usr/bin/env python3
"""Draft the genomics Methods paragraph from what actually ran.

Paragraph 1 describes only the ASA3P steps that produced output.
Paragraph 2 describes replacement runs outside ASA3P, each with its reason
(ASA3P step empty or reported as failed in its log, or per-isolate annotation problem).
Paragraph 3 describes the additional analyses outside ASA3P whose outputs exist (including, from v5.5, the genus-level
core-gene tree of D4b and the GTDB-Tk identification of D4c).
Paragraph 4 describes the Part F functional screens (CAZymes, secretion systems, mobile elements,
virulence / metal resistance, effectors) when functional/tables exists.
Versions come from software/container_versions.txt, software/other_versions.txt,
and detect_asap_version.py (asap.jar -> run logs -> config.sh fallback);
anything unknown is left as [bracketed] text.

  -> DOWNSTREAM/methods_draft.md
"""
import csv
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

META = Path(os.environ["META"])
DOWNSTREAM = Path(os.environ["DOWNSTREAM"])
PROV = Path(os.environ["PROVENANCE"])
SOFT = Path(os.environ["SOFT"])
sys.path.insert(0, str(Path(__file__).resolve().parent))
from detect_asap_version import detect  # noqa: E402

ASAP_VERSION, ASAP_VERSION_SOURCE = detect()


def read_tsv(path):
    if not Path(path).exists():
        return []
    with open(path, newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


def versions():
    v = {}
    for name in ("container_versions.txt", "other_versions.txt"):
        p = SOFT / name
        if p.exists():
            for line in p.read_text().splitlines():
                if ":" in line:
                    k, val = line.split(":", 1)
                    m = re.search(r"\d+(?:\.\d+)+", val)
                    if m:
                        v[k.strip().lower()] = m.group(0)
    return v


def ver(key):
    return f"v{V[key]}" if key in V else "[version]"


def join(items):
    items = list(items)
    if len(items) <= 1:
        return "".join(items)
    return ", ".join(items[:-1]) + " and " + items[-1]


def ital(names):
    return join(f"*{n}*" for n in names)


def italic_species(genus, species):
    return f"*{genus[0]}. {species}*"


V = versions()
samples = read_tsv(META / "metadata_shortnames.tsv")
refs = read_tsv(META / "reference_genomes.tsv")
audit = read_tsv(DOWNSTREAM / "asap_module_audit.tsv")
replacements = read_tsv(PROV / "replacements.tsv")
todo = read_tsv(DOWNSTREAM / "prokka_fallback" / "todo.tsv")
groups = read_tsv(DOWNSTREAM / "roary" / "roary_groups.tsv")
gate = read_tsv(DOWNSTREAM / "species_check" / "analysis_set.tsv")
split = {r["short_name"]: r for r in read_tsv(DOWNSTREAM / "contig_split" / "split_summary.tsv")}

genus_n = Counter(s["genus"] for s in samples)
genera = sorted(genus_n)
N = len(samples)
ref_species = [r for r in refs if r["role"] == "reference"]
outgroup = [r for r in refs if r["role"] == "outgroup"]
cites, notes = set(["Schwengers et al., 2020 (ASA3P)"]), []

# ---------------------------------------------------------------- paragraph 1
STEP_TEXT = [
    ("reads_qc", "read quality control and adapter/quality trimming (FastQC, Trimmomatic)"),
    ("assemblies", "*de novo* assembly (SPAdes)"),
    ("scaffolds", f"reference-guided contig ordering (MeDuSa) against {len(ref_species)} species reference genomes (Table S[X])"),
    ("annotations", "genome annotation (Prokka)"),
    ("taxonomy", "taxonomic classification (Kraken k-mer profiling, 16S rRNA gene BLAST+ against SILVA, and nucmer-based average nucleotide identity)"),
    ("mlst", "multilocus sequence typing (BLAST+ against PubMLST schemes)"),
    ("abr", "antimicrobial resistance gene detection (RGI with the CARD database)"),
    ("vf", "virulence factor detection (BLAST+ against VFDB)"),
    ("mappings", "read mapping to the reference genomes (Bowtie2)"),
    ("snps", "SNP detection (SAMtools, SnpEff)"),
    ("corepan", "core/pan-genome calculation (Roary)"),
    ("phylogeny", "an approximately maximum-likelihood phylogeny (FastTreeMP)"),
]
present = defaultdict(list)
for a in audit:
    if (a.get("usable") or ("yes" if int(a["n_files"]) > 0 else "no")) == "yes":
        present[a["module"]].append(a["genus"])

done_steps, silent_fail = [], []
for step, text in STEP_TEXT:
    gs = sorted(present.get(step, []))
    if not gs:
        silent_fail.append(step)
        continue
    if set(gs) != set(genera):
        text = text[:-1] + f"; {ital(gs)} only)" if text.endswith(")") else text + f" ({ital(gs)} only)"
    done_steps.append(text)

p1 = (f"Paired-end whole-genome reads (Element Biosciences AVITI) of {N} presumptive "
      f"{join(f'*{g}*' for g in genera)} isolates (genus and species from the initial identification; Table S[X]) were processed with ASA³P "
      f"{'v' + ASAP_VERSION if ASAP_VERSION else '[version]'} (Schwengers et al., 2020) run in its Apptainer "
      f"container, as one project per genus. ")
if done_steps:
    p1 += "Within ASA³P, the steps used were " + join(done_steps) + "."
if not ASAP_VERSION:
    notes.append("ASA³P version unknown: check the top of the ASA³P run log and set ASAP_VERSION in config.sh.")
elif ASAP_VERSION_SOURCE.startswith("unconfirmed"):
    notes.append(f"ASA³P version {ASAP_VERSION} is the Zenodo download version; it was not found in asap.jar or "
                 "the run logs. Confirm it if you can (e.g. the Zenodo record page of the file you downloaded).")
else:
    notes.append(f"ASA³P version {ASAP_VERSION} read from {ASAP_VERSION_SOURCE}.")
no_replacement = [s for s in silent_fail if s not in {"abr", "vf", "corepan", "phylogeny"}]
if no_replacement:
    notes.append("These ASA³P steps produced no usable output (empty, or reported as failed in the ASA³P log) and have no replacement here, so they are left "
                 f"out of the text: {', '.join(no_replacement)}. Check the ASA³P logs before submission.")

# ---------------------------------------------------------------- paragraph 2
p2_parts = []
fb = [t for t in todo if (DOWNSTREAM / "prokka_fallback" / t["short_name"] / f"{t['short_name']}.gff").exists()]
if fb:
    cites.add("Seemann, 2014 (Prokka)")
    if all(t["reason"].startswith("re-annotated for consistency") for t in fb):
        p2_parts.append(f"All {len(fb)} genomes were re-annotated from their assembled contigs outside ASA³P with "
                        "Prokka v1.12 (the version bundled with ASA³P, using the ASA³P protein database), so that "
                        "every genome was annotated from the same input.")
    else:
        by_reason = defaultdict(list)
        for t in fb:
            by_reason[t["reason"]].append(t["short_name"])
        PHRASE = {
            "ASA3P produced no annotation": "no annotation was produced",
            "ASA3P annotated the reference-ordered pseudo-genome": "the reference-ordered pseudo-genome had been annotated instead of the assembled contigs",
            "ASA3P annotation omitted contigs": "the annotation omitted assembled contigs",
            "ASA3P GFF lacks the sequence block Roary needs": "the annotation file lacked the sequence section required by Roary",
        }
        why = "; ".join(f"{PHRASE.get(r, r)} for {join(v)}" for r, v in by_reason.items())
        p2_parts.append(f"For {len(fb)} of {N} isolates the ASA³P annotation could not be used ({why}); their "
                        "assembled contigs were re-annotated outside ASA³P with Prokka v1.12, the version bundled "
                        "with ASA³P, using the ASA³P protein database.")
    missing_fb = [t["short_name"] for t in todo if t not in fb]
    if missing_fb:
        notes.append(f"Fallback Prokka did not produce a GFF for: {', '.join(missing_fb)}.")

rep = defaultdict(list)
for r in replacements:
    if not r["decision"].startswith("use ASA3P"):
        rep[r["step"]].append(r["genus"])
if rep.get("abr") and any((DOWNSTREAM / "replacement_abr").glob("*")):
    cites.update(["Feldgarden et al., 2021 (AMRFinderPlus)", "Seemann, ABRicate (GitHub)", "Alcock et al., 2023 (CARD)"])
    p2_parts.append(f"Because ASA³P's resistance-gene step did not produce usable output for {ital(rep['abr'])}, antimicrobial "
                    f"resistance genes in those genomes were identified with AMRFinderPlus {ver('amrfinder')} and with "
                    f"ABRicate {ver('abricate')} against CARD.")
elif rep.get("abr"):
    notes.append("ASA³P's abr step was empty or reported as failed, but the replacement has not been run yet (Part C2).")
if rep.get("vf") and any((DOWNSTREAM / "replacement_vf").glob("*")):
    cites.update(["Seemann, ABRicate (GitHub)", "Liu et al., 2022 (VFDB)"])
    p2_parts.append(f"Because ASA³P's virulence-factor step did not produce usable output for {ital(rep['vf'])}, virulence "
                    f"factors were identified with ABRicate {ver('abricate')} against VFDB.")
elif rep.get("vf"):
    notes.append("ASA³P's vf step was empty or reported as failed, but the replacement has not been run yet (Part C3).")
genus_roary = [g for g in groups if g["role"] == "replacement"
               and Path(g["out_dir"], "summary_statistics.txt").exists()]
if genus_roary:
    cites.add("Page et al., 2015 (Roary)")
    gnames = [g["group"].replace("_all", "") for g in genus_roary]
    p2_parts.append(f"Because ASA³P's core/pan-genome step did not produce usable output for {ital(gnames)}, the genus-level "
                    f"pan-genome was computed outside ASA³P with Roary {ver('roary')} (minimum BLASTp identity 80%).")
if rep.get("phylogeny"):
    p2_parts.append(f"ASA³P's phylogeny step did not produce usable output for {ital(rep['phylogeny'])}; genus-level "
                    "relationships are therefore shown with the Mashtree tree described below.")
p2 = " ".join(p2_parts)

# ---------------------------------------------------------------- paragraph 3
p3_parts = []
if gate:
    cites.add("Jain et al., 2018 (FastANI)")
    c = Counter(g["qc_status"] for g in gate)
    inc = [g for g in gate if g["include"] == "yes"]
    anis = sorted(float(g["best_ani"]) for g in gate if g["qc_status"] == "NO_MATCH" and g["best_ani"])
    txt = (f"Species identities were checked outside ASA³P by average nucleotide identity (ANI) to "
           f"the {len(ref_species)} species reference genomes with FastANI {ver('fastani')}, assigning a species at ≥95% ANI "
           f"with ≥50% of genome fragments aligned. {c['PASS']} of {len(gate)} isolates matched their initial identification")
    if c["RELABELLED"]:
        txt += f"; {c['RELABELLED']} matched a different reference species and were relabelled"
    if c["MIXED"]:
        txt += (f"; {c['MIXED']} assemblies reached ≥95% ANI to a reference species over less than half of their length "
                "and were judged to contain a second genome")
    if c["NO_MATCH"]:
        rng = f" (best ANI {anis[0]:.1f}–{anis[-1]:.1f}%)" if anis else ""
        txt += (f"; {c['NO_MATCH']} isolates matched none of the reference species{rng}, a distance consistent with "
                "genus level or beyond")
    kept = [g for g in inc if g["override"] == "use_filtered" and g["short_name"] in split]
    txt += f". Genome-level comparisons are restricted to the {len(inc) - len(kept)} isolates that passed this check"
    if kept:
        txt += (f", plus {len(kept)} mixed assembl{'y' if len(kept) == 1 else 'ies'} from which the contigs of the expected "
                "organism (≥90% ANI to the species reference) were separated before re-annotation")
        notes.append("Contig-filtered genomes are a stop-gap: " + "; ".join(
            f"{g['short_name']} kept {split[g['short_name']]['kept_mb']} of "
            f"{float(split[g['short_name']]['kept_mb']) + float(split[g['short_name']]['removed_mb']):.2f} Mb "
            f"({split[g['short_name']]['kept_over_reference']}× the reference length)" for g in kept)
            + ". Replace them with the re-sequenced isolates when available.")
    p3_parts.append(txt + ".")
    out_ = [g for g in gate if g["include"] != "yes"]
    if out_:
        notes.append("Left out of Part D by the species check: " + ", ".join(
            f"{g['short_name']} ({g['qc_status']}" + (", " + g["override"] if g["override"] else "") + ")" for g in out_)
            + ". Give the per-isolate status (analysis_set.tsv) in Table S[X].")
    forced = [g["short_name"] for g in gate if g["override"] == "include" and g["qc_status"] in {"MIXED", "NO_MATCH"}]
    if forced:
        notes.append("Kept by your override although the species check failed: " + ", ".join(forced) +
                     ". State the reason in the text (override reasons are in isolate_overrides.tsv).")
    if (DOWNSTREAM / "replacement_abr").exists() and any((DOWNSTREAM / "replacement_abr").glob("*")):
        notes.append("Resistance/virulence replacement results (Part C2-C3) cover every isolate; keep only include = yes "
                     "(analysis_set.tsv) when tabulating them for the genome set above.")
    notes.append("Species labels in metadata_shortnames.tsv are the initial identification; analysis_set.tsv holds the "
                 "ANI-based species used from Part D onward.")
tree = DOWNSTREAM / "trees" / "mashtree_isolates_refs_outgroup.nwk"
if tree.exists() and tree.stat().st_size > 0:
    cites.add("Katz et al., 2019 (Mashtree)")
    og = outgroup[0]["label"].replace("OUT_", "").replace("_", " ") if outgroup else "[outgroup]"
    og = re.sub(r"^Erwinia amylovora", "*Erwinia amylovora*", og)
    n_tree = sum(g["include"] == "yes" for g in gate) if gate else N
    p3_parts.append(f"A whole-genome distance tree of {'the ' + str(n_tree) + ' retained isolates' if gate else 'all isolates'}, "
                    f"the {len(ref_species)} species reference genomes and {og} (outgroup) was built with Mashtree {ver('mashtree')}.")
boot = DOWNSTREAM / "trees" / "mashtree_ALL_bootstrap.nwk"
if boot.exists() and boot.stat().st_size > 0:
    cites.add("Katz et al., 2019 (Mashtree)")
    binfo = {}
    info_f = DOWNSTREAM / "trees" / "mashtree_ALL_bootstrap.info"
    if info_f.exists():
        binfo = dict(l.split("\t", 1) for l in info_f.read_text().splitlines() if "\t" in l)
    reps = binfo.get("replicates", "[number of]")
    n_all = binfo.get("isolates", str(N))
    p3_parts.append(f"A second Mashtree tree contains every sequenced isolate (n = {n_all}), irrespective of the result of the species check, "
                    f"together with the reference genomes and the outgroup. Branch lengths are Mash distances. Node support is the percentage of "
                    f"{reps} bootstrap replicates (mashtree_bootstrap.pl) in which the node was found; each replicate re-sketches all genomes with a "
                    f"different random Mash seed, so the support measures how stable a node is to the sample of k-mers in the sketches, not to "
                    f"resampling of alignment sites.")
species_groups = [g for g in groups if g["role"] == "additional"
                  and Path(g["out_dir"], "summary_statistics.txt").exists()]
if species_groups:
    cites.update(["Page et al., 2015 (Roary)", "Katoh & Standley, 2013 (MAFFT)"])
    desc = [f"{italic_species(*g['group'].split('_', 1))}, n = {g['n']}" for g in species_groups]
    p3_parts.append(f"Because ASA³P computes one pan-genome per project, per-species pan-genomes were computed "
                    f"outside ASA³P for every species with at least two isolates ({'; '.join(desc)}) using Roary "
                    f"{ver('roary')} with 95% minimum BLASTp identity, with core genes aligned by MAFFT.")
trees = [g for g in species_groups
         if Path(g["out_dir"]).parent.joinpath("iqtree_core", f"{g['group']}_core.treefile").exists()]
if trees:
    cites.update(["Minh et al., 2020 (IQ-TREE 2)", "Kalyaanamoorthy et al., 2017 (ModelFinder)",
                  "Hoang et al., 2018 (UFBoot2)"])
    big = [italic_species(*g["group"].split("_", 1)) for g in trees if int(g["n"]) >= 4]
    small = [italic_species(*g["group"].split("_", 1)) for g in trees if int(g["n"]) == 3]
    txt = f"Maximum-likelihood trees were inferred from the concatenated core-gene alignments with IQ-TREE {ver('iqtree')}"
    if big:
        txt += f" ({join(big)}: ModelFinder model selection, 1,000 ultrafast bootstrap and 1,000 SH-aLRT replicates)"
    if small:
        txt += (f"; for species with three isolates ({join(small)}) trees were inferred under GTR+G without "
                "support values, and only branch lengths are interpreted")
    p3_parts.append(txt + ".")
core_info = dict(l.split("\t", 1) for l in (DOWNSTREAM / "trees" / "core_gene_tree.info").read_text().splitlines() if "\t" in l) \
    if (DOWNSTREAM / "trees" / "core_gene_tree.info").exists() else {}
if core_info and (DOWNSTREAM / "trees" / "core_gene_tree.nwk").exists():
    cites.update(["Seemann, 2014 (Prokka)", "Page et al., 2015 (Roary)", "Katoh & Standley, 2013 (MAFFT)",
                  "Minh et al., 2020 (IQ-TREE 2)", "Kalyaanamoorthy et al., 2017 (ModelFinder)",
                  "Hoang et al., 2018 (UFBoot2)", "Guindon et al., 2010 (SH-aLRT)"])
    n_core = int(core_info.get("core_genes", "0") or 0)
    aln_len = int(core_info.get("alignment_length_bp", "0") or 0)
    txt = (f"A core-gene phylogeny of the genus was inferred from {core_info.get('isolates', '[n]')} isolates that passed the species "
           f"check and {core_info.get('references', '[n]')} *Pectobacterium* reference genomes (type strains where available; Table S[X]). "
           "The reference genomes were annotated with the same Prokka v1.12 and ASA³P protein database as the isolates, "
           f"and all genomes were clustered with Roary {ver('roary')} (minimum BLASTp identity {core_info.get('roary_identity', '[x]')}%; "
           f"core genes present in ≥{core_info.get('roary_core_threshold', '99')}% of genomes). The {(format(n_core, ',') + ' core genes were') if n_core else '[number of] core genes were'} aligned with MAFFT "
           f"({aln_len:,} bp), and a maximum-likelihood tree was inferred with IQ-TREE {ver('iqtree')} under "
           f"{core_info.get('model', '[model]')} (ModelFinder restricted to the GTR family), with 1,000 ultrafast bootstrap and "
           "1,000 SH-aLRT replicates")
    if core_info.get("scf") == "yes":
        cites.add("Minh et al., 2020 (concordance factors)")
        txt += ", and site concordance factors (100 quartets per branch)"
    txt += (". Isolates that did not pass the species check (about 78% ANI to the *Pectobacterium* references, or assemblies of more "
            "than one genome) were not included: one genome of another genus is enough to remove nearly all genes from the core.")
    p3_parts.append(txt)
    notes.append("The core-gene tree is unrooted (Dickeya and Erwinia share too few genes with Pectobacterium to be in the alignment). "
                 "Root it at the midpoint or on a clade you can justify, and say which in the legend.")
    if n_core < 1000:
        notes.append(f"The core-gene alignment has only {n_core} genes: check the genome set (core_tree/core_set.tsv, D4b.5) before using the tree.")
gt = DOWNSTREAM / "gtdbtk" / "gtdbtk_classification.tsv"
if gt.exists() and gt.stat().st_size > 0:
    cites.update(["Chaumeil et al., 2022 (GTDB-Tk v2)", "Parks et al., 2022 (GTDB)"])
    gdb = DOWNSTREAM / "gtdbtk" / "gtdbtk_db_metadata.txt"
    rel = re.search(r"VERSION_DATA\s*=\s*(\S+)", gdb.read_text()) if gdb.exists() else None
    p3_parts.append(f"The genus of isolates that did not pass the species check was determined with GTDB-Tk {ver('gtdbtk')} "
                    f"(classify_wf, placement in the GTDB reference tree, release {rel.group(1) if rel else '[GTDB release]'}; ANI screening skipped).")
    notes.append("GTDB names differ from NCBI names for some genera and species: report the GTDB genus as 'GTDB genus' and keep the NCBI names for the reference genomes.")
part = read_tsv(DOWNSTREAM / "pversatile_source" / "Pversatile_gene_partition.tsv")
if part:
    n_plant = part[0]["plant_present"].split("/")[1]
    n_env = part[0]["env_present"].split("/")[1]
    p3_parts.append(f"Accessory gene content of *P. versatile* isolates from diseased plants (n = {n_plant}) and "
                    f"environmental sources (n = {n_env}) was compared descriptively from the per-species Roary "
                    "presence/absence table. Each lineage was recovered from a single source, so source-associated "
                    "genes could not be separated from lineage markers, and no test of association is reported.")
maps = DOWNSTREAM / "genome_maps" / "Pversatile_env_vs_plant" / "results"
if maps.exists() and any(maps.iterdir()):
    cites.update(["Shimoyama, pyGenomeViz (GitHub)", "Marçais et al., 2018 (MUMmer4)"])
    txt = (f"Genome synteny among selected *P. versatile* isolates was visualised with pyGenomeViz "
           f"{ver('pygenomeviz')} (MUMmer {ver('mummer')} alignments)")
    if (maps / "pversatile_clinker.html").exists():
        cites.add("Gilchrist & Chooi, 2021 (clinker)")
        txt += f" and gene-cluster comparisons with clinker {ver('clinker')}"
    p3_parts.append(txt + ".")
p3 = " ".join(p3_parts)

# ---------------------------------------------------------------- paragraph 4 (Part F)
FT = DOWNSTREAM / "functional" / "tables"
p4 = ""
fsum = read_tsv(FT / "functional_summary.tsv")
if fsum:
    fstat = read_tsv(FT / "functional_status.tsv")
    n_f = len(fsum)

    def n_done(col):
        return sum(1 for r in fstat if r.get(col) == "DONE")
    col_dbcan, col_txss, col_gen = "dbCAN", "MacSyFinder/TXSScan", "geNomad"
    col_mob, col_abr, col_et3 = "MOB-suite", "ABRicate (VFDB, BacMet2, PlasmidFinder)", "EffectiveT3"
    n_fail = sum(1 for r in fsum if r.get("gate_status") not in ("PASS", "RELABELLED"))
    p4_parts = [f"Functional gene content was screened on all {n_f} genomes outside ASA³P, from the protein translations of the "
                "annotation used above and, for nucleotide-level tools, the assembled contigs."]
    if n_done(col_dbcan):
        cites.add("Zheng et al., 2023 (dbCAN3)")
        p4_parts.append(f"Carbohydrate-active enzymes were annotated with run_dbcan {ver('dbcan')} (HMMER search against dbCAN HMMs, "
                        "dbCAN-sub and DIAMOND search against CAZy); a gene was counted as a CAZyme when at least two of the three "
                        "methods supported it. Genes in CAZy families commonly associated with plant cell wall degradation were "
                        "grouped as pectin-, cellulose- and hemicellulose-degrading candidates (family-level assignment, "
                        "functional/tables/pcwde_family_map.tsv).")
    if n_done(col_txss):
        cites.update(["Néron et al., 2023 (MacSyFinder v2)", "Abby et al., 2016 (TXSScan)"])
        p4_parts.append(f"Protein secretion systems were detected with MacSyFinder {ver('macsyfinder')} and the TXSScan models "
                        f"{ver('txsscan')}, treating each contig as a replicon (gembase mode) and reporting the best solution.")
    if n_done(col_gen) or n_done(col_mob):
        bits = []
        if n_done(col_gen):
            cites.add("Camargo et al., 2024 (geNomad)")
            bits.append(f"viral and plasmid sequences, including integrated proviruses, were identified with geNomad {ver('genomad')} "
                        f"(database {ver('genomad_db')})")
        if n_done(col_mob):
            cites.add("Robertson & Nash, 2018 (MOB-suite)")
            bits.append(f"plasmid replicons, relaxases and predicted mobility were typed with MOB-suite {ver('mob_suite')} (mob_recon)")
        p4_parts.append("Mobile elements: " + "; ".join(bits) + ".")
    if n_done(col_abr):
        cites.update(["Seemann, ABRicate (GitHub)", "Carattoli et al., 2014 (PlasmidFinder)", "Liu et al., 2022 (VFDB)", "Pal et al., 2014 (BacMet)"])
        p4_parts.append(f"Virulence factors (VFDB), metal and biocide resistance genes (BacMet2) and plasmid replicons (PlasmidFinder) "
                        f"were searched in the contigs with ABRicate {ver('abricate')} (≥{os.environ.get('ABR_MINID', '80')}% identity and "
                        f"≥{os.environ.get('ABR_MINCOV', '80')}% coverage for VFDB and BacMet2; ≥{os.environ.get('PLF_MINID', '80')}% identity and "
                        f"≥{os.environ.get('PLF_MINCOV', '60')}% coverage for PlasmidFinder).")
    if n_done(col_et3):
        cites.add("Arnold et al., 2009 (EffectiveT3)")
        p4_parts.append(f"Type III secreted effector candidates were predicted with EffectiveT3 {ver('effectivet3')} "
                        f"({os.environ.get('ET3_MODEL', 'TTSS_STD-2.0.2.jar').replace('.jar', '')} module, {os.environ.get('ET3_CUTOFF', 'selective')} cutoff); "
                        "candidates are predictions, not confirmed effectors.")
    else:
        notes.append("EffectiveT3 did not run (see functional/tables/functional_status.tsv): remove effector results from the Methods, "
                     "or fix it and re-run F2 with TOOLS=et3.")
    notes.append("EffectiveDB is a web server and was not run here; the draft cites EffectiveT3 only. Delete EffectiveDB from the "
                 "Methods, or run it separately and report it as such.")
    missing = [lab for lab, c in (("dbCAN", col_dbcan), ("MacSyFinder/TXSScan", col_txss), ("geNomad", col_gen),
                                  ("MOB-suite", col_mob), ("ABRicate", col_abr)) if n_done(c) < n_f]
    if missing:
        notes.append(f"Not every isolate finished for: {', '.join(missing)}. Their counts are NA, not 0 (functional_status.tsv).")
    if n_fail:
        notes.append(f"{n_fail} of the {n_f} isolates did not pass the species check. They are in the Part F tables so that every genome "
                     "was screened, but each row carries gate_status and the genome-based identification: report them as "
                     "'re-sequencing pending' and do not pool them with the confirmed SRP isolates.")
    notes.append("The CAZyme, PCWDE, plasmid, virus and effector columns in source_metadata_correlated.tsv came from methods that are "
                 "not recorded and cannot be reproduced here; use functional/tables/functional_summary.tsv instead.")
    p4 = " ".join(p4_parts)

# ---------------------------------------------------------------- write
if any("[version]" in p for p in (p1, p2, p3, p4)):
    notes.append("Some versions are unknown ([version]): run the version cells (A0.4 and E1) and re-run this cell.")
notes.append("ASA³P bundles SPAdes, Prokka, MeDuSa, Kraken, RGI/CARD, VFDB, Bowtie2, SAMtools, SnpEff, Roary "
             "and FastTreeMP; cite them individually if the journal asks for every tool.")
notes.append("Table S[X] = isolate list (short name, species, source) and the reference genome accessions "
             "(metadata/reference_genomes.tsv).")
out = DOWNSTREAM / "methods_draft.md"
with out.open("w") as f:
    f.write("## Genome assembly, annotation and comparative genomics\n\n")
    f.write(p1 + "\n\n")
    if p2:
        f.write(p2 + "\n\n")
    if p3:
        f.write(p3 + "\n\n")
    if p4:
        f.write(p4 + "\n\n")
    f.write("### Tools to cite\n\n" + "\n".join(f"- {c}" for c in sorted(cites)) + "\n\n")
    f.write("### Notes for the author (delete before submission)\n\n" + "\n".join(f"- {n}" for n in notes) + "\n")
print(out.read_text())
print(f"Wrote {out}")
END_OF_SCRIPT
chmod +x "$SCRIPTS/write_methods.py.new" && mv -f "$SCRIPTS/write_methods.py.new" "$SCRIPTS/write_methods.py"
echo "wrote $SCRIPTS/write_methods.py"

use_env download_env
python "$SCRIPTS/write_methods.py"

wrote /home/s94m696/asap_shortname_projects/scripts/detect_asap_version.py
wrote /home/s94m696/asap_shortname_projects/scripts/write_methods.py
## Genome assembly, annotation and comparative genomics

Paired-end whole-genome reads (Element Biosciences AVITI) of 25 presumptive *Dickeya* and *Pectobacterium* isolates (genus and species from the initial identification; Table S[X]) were processed with ASA³P v1.3.0 (Schwengers et al., 2020) run in its Apptainer container, as one project per genus. Within ASA³P, the steps used were read quality control and adapter/quality trimming (FastQC, Trimmomatic), *de novo* assembly (SPAdes), reference-guided contig ordering (MeDuSa) against 10 species reference genomes (Table S[X]), genome annotation (Prokka), taxonomic classification (Kraken k-mer profiling, 16S rRNA gene BLAST+ against SILVA, and nucmer-based average nucleotide identity), multilocus sequence typing (BLAST+ against PubMLST schemes), virulence factor detection (BLAST+ against VFDB), r

## E3. Outputs to use

| What | Source | Path (under `/home/s94m696/asap_shortname_projects/`) |
|---|---|---|
| ASA³P reports and per-step results | ASA³P | `ASA3P_<Genus>/reports/index.html`, `ASA3P_<Genus>/<step>/` |
| Genus core/pan-genome, genus phylogeny | ASA³P | `ASA3P_<Genus>/corepan/`, `ASA3P_<Genus>/phylogeny/` |
| What ran where, and why | Check | `downstream/provenance/replacements.tsv`, `downstream/prokka_fallback/todo.tsv` |
| Output audit, per-isolate files, assembly QC | Check | `downstream/asap_module_audit.tsv`, `downstream/asap_paths.tsv` |
| Standardized contigs | Check | `genomes/<short>.fna` (+ `<short>.header_map.tsv`) |
| Annotation used per isolate | ASA³P or replacement | `downstream/annotation_files.tsv` (column `source`) |
| Resistance / virulence genes (only if ASA³P's were empty or failed) | Replacement | `downstream/replacement_abr/`, `downstream/replacement_vf/` |
| Genus pan-genome (only if ASA³P's was empty or failed) | Replacement | `downstream/roary/<Genus>_all/roary_output/` |
| Species gate: status, included yes/no, species, genome used, per isolate | Outside ASA³P | `downstream/species_check/analysis_set.tsv` (Table S[X]) |
| Species calls (best and second reference hit) | Outside ASA³P | `downstream/species_check/species_calls.tsv` |
| All-vs-all fastANI groups; 16S sequences of failed isolates | Outside ASA³P | `downstream/species_check/allvsall_groups.tsv`, `16S_not_passed.fasta` |
| Your decisions | You | `metadata/isolate_overrides.tsv` |
| Contig split of `MIXED` assemblies | Outside ASA³P | `downstream/contig_split/<short>/contig_assignment.tsv`, `split_summary.tsv`, `genomes_filtered/<short>.fna` |
| Per-species pan-genomes | Outside ASA³P | `downstream/roary/<Genus_species>/roary_output/` |
| Genome-distance tree (isolates that passed the gate) | Outside ASA³P | `downstream/trees/mashtree_isolates_refs_outgroup.nwk` |
| Genome-distance tree, **every isolate** (no gate), bootstrap support 0–100 on the nodes | Outside ASA³P | `downstream/trees/mashtree_ALL_bootstrap.nwk` (+ `.info`) |
| Mash distance matrix, same genomes | Outside ASA³P | `downstream/trees/mashtree_ALL_distances.tsv` |
| Same genomes, one run without bootstrap | Outside ASA³P | `downstream/trees/mashtree_ALL_diagnostic.nwk` |
| Per-species core-gene trees | Outside ASA³P | `downstream/roary/<Genus_species>/iqtree_core/` |
| *Pectobacterium* type-strain genomes proposed for the core tree (with the reason each was or was not chosen) | Outside ASA³P (D4b.1) | `metadata/core_references_candidates.tsv`, `metadata/core_references.tsv`, `references/core/` |
| Genomes of the genus-level core tree (isolates + references) | Outside ASA³P (D4b.3) | `downstream/core_tree/core_set.tsv` |
| Roary run of the core tree; which genomes lack the common genes (new run, and ASA³P's old 22-core-gene run) | Outside ASA³P (D4b.4–5) | `downstream/core_tree/roary_output/`, `core_loo.tsv`, `core_loo_ASA3P_project.tsv` |
| **Genus-level core-gene tree** (unrooted; SH-aLRT/UFBoot on the nodes; the `_scf` file has site concordance factors) | Outside ASA³P (D4b.6) | `downstream/trees/core_gene_tree.nwk`, `core_gene_tree_scf.nwk`, `core_gene_tree.info` |
| GTDB-Tk genus and species per isolate, next to the species-check status | Outside ASA³P (D4c) | `downstream/gtdbtk/gtdbtk_classification.tsv` |
| *P. versatile* gene partition | Outside ASA³P | `downstream/pversatile_source/Pversatile_gene_partition.tsv` |
| Genome maps | Outside ASA³P | `downstream/genome_maps/Pversatile_env_vs_plant/results/` |
| Functional screens, one row per isolate (CAZymes, PCWDE-family genes, secretion systems, proviruses, plasmids, VFDB, BacMet2, effectors) with species-check status | Outside ASA³P (F4) | `downstream/functional/tables/functional_summary.tsv` |
| Which Part F tool finished for which isolate | Outside ASA³P (F3–F4) | `downstream/functional/tables/functional_status.tsv` |
| CAZyme genes and families; PCWDE family map | Outside ASA³P (F4) | `functional/tables/cazyme_genes.tsv`, `cazyme_families_long.tsv`, `pcwde_family_map.tsv` |
| Secretion systems | Outside ASA³P (F4) | `functional/tables/secretion_systems_long.tsv`, `secretion_systems_matrix.tsv` |
| Prophages, plasmids, replicons | Outside ASA³P (F4) | `functional/tables/genomad_viruses.tsv`, `genomad_plasmids.tsv`, `mob_contigs.tsv`, `plasmidfinder_hits.tsv` |
| Virulence and metal/biocide resistance genes | Outside ASA³P (F4) | `functional/tables/vfdb_hits.tsv`, `vfdb_matrix.tsv`, `bacmet2_hits.tsv`, `bacmet2_matrix.tsv` |
| Type III effector candidates | Outside ASA³P (F4) | `functional/tables/effector_candidates.tsv` |
| ASA³P taxonomy next to the gate; initial vs. genome-based identification (supplement) | Outside ASA³P (D1.3) | `downstream/species_check/asap_taxonomy.tsv`, `supplement_isolate_identity.tsv` |
| Methods draft | — | `downstream/methods_draft.md` |
| Tool versions | — | `software/container_versions.txt`, `software/other_versions.txt` |

All tables use the short names, so they join directly to `metadata_shortnames.tsv`, `source_metadata_correlated.tsv`, and the isolate codes in `ani_pathogenicity.xlsx` (after PI024 → PI024_2 and PI029 → PI029_2). Join `analysis_set.tsv` on `short_name` to keep only the isolates in the analysis set.